# Notebook 2 Generating a Transient Population

In this notebook we generate a mixed population of supernovae (SN) and kilonovae (KN) 
within the synthetic universe constructed in Notebook 1.

We assume:

- Supernova rate ∝ Star Formation Rate (SFR)
- Kilonova rate ∝ Total Stellar Mass (M_star)
- Absolute magnitude M = -16 for both populations
- Survey duration T = 50,000 years

This notebook produces an intrinsic transient population and computes 
their apparent magnitudes using the distance modulus.

This block loads every Python library the notebook relies on. numpy handles arrays and maths, pandas handles tables, and matplotlib draws the plots. scipy's UnivariateSpline fits smooth curves through scattered points, and scikit-learn's roc_curve and auc score how well a classifier separates two groups. Nothing runs yet; these imports just make the tools available for later cells.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from sklearn.metrics import roc_curve, auc


## Load Synthetic Galaxy Catalogue

We import the galaxy catalogue generated in Notebook 1.

This catalogue contains:

- Distance (Mpc)
- Stellar Mass (M_star)
- Star Formation Rate (SFR)
- Galaxy properties

This reads galaxy_catalog.csv, the mock galaxy catalogue built in the companion notebook, into a pandas table called galaxies. Each row is one galaxy with columns such as its star formation rate, stellar mass, and distance. It prints the number of galaxies and shows the first few rows so you can see the structure.

In [ ]:
galaxies = pd.read_csv("galaxy_catalog.csv")

print("Number of galaxies:", len(galaxies))
galaxies.head()


## Physical Event Rates

We assume simple host galaxy scaling relations.

The supernova rate in each galaxy is proportional to its star formation rate.

The kilonova rate in each galaxy is proportional to its stellar mass.

---

### Assumptions

- Core collapse supernovae trace ongoing star formation.
- Kilonovae trace stellar mass as a proxy for delayed mergers.
- Rates are constant over the survey duration.

Survey duration: 50,000 years.

---

### Normalizations

Supernova normalization: 1 × 10^-6 events per (M☉/yr) per year of star formation.

Kilonova normalization: 1 × 10^-18 events per year per solar mass of stellar mass.

These values reproduce realistic order-of-magnitude local event rates.

This fixes the event rates that drive the simulation. A_SN is how many supernovae a galaxy makes per unit of star formation, taken from Cappellaro et al. 1999; a supernova is an exploding star. A_KN is how many kilonovae form per unit of stellar mass, set to match measured binary neutron star merger rates; a kilonova is the glow from two neutron stars merging. T_survey is not a real survey length. It is a long pretend duration of 50000 years, chosen only so the catalogue yields enough events to study.

In [ ]:
A_SN = 1e-6       # SN per (Msun/yr) per year — Cappellaro et al. (1999)
A_KN = 1e-18      # KN per Msun per year — consistent with BNS merger rates

# T_survey is not a physical survey duration. It is a computational device
# used to generate a statistically useful number of events across the galaxy
# catalogue. The per-event injection simulation below uses its own cadence.
T_survey = 50000.0     # years


## Sampling Event Counts

For each galaxy, we compute the expected number of events over the survey duration.

The expected supernova count scales with star formation rate.

The expected kilonova count scales with stellar mass.

Actual event numbers are drawn from a Poisson distribution with these expected values as the mean.

The expected totals represent the ensemble average.

The realised totals represent one Monte Carlo survey realisation.

This turns the rates into an expected number of events for every galaxy. It multiplies each galaxy's star formation rate and stellar mass by the rates and the survey time to get lambda_SN and lambda_KN, the mean count per galaxy. It then draws an actual whole number from a Poisson distribution, which models random counts around a mean. It prints the expected totals next to the realised random totals so you can see they roughly match.

In [ ]:
lambda_SN = A_SN * galaxies["SFR"] * T_survey
lambda_KN = A_KN * galaxies["M_star"] * T_survey

N_SN = np.random.poisson(lambda_SN)
N_KN = np.random.poisson(lambda_KN)

print("Total expected SN:", lambda_SN.sum())
print("Total expected KN:", lambda_KN.sum())

print("Total realised SN:", N_SN.sum())
print("Total realised KN:", N_KN.sum())

## Construct Event Catalogue

We expand galaxy level event counts into an event-level table.

Each event stores:

- Event type (SN or KN)
- Host galaxy distance

This produces the intrinsic transient population.

This walks through every galaxy and creates one row for each event it produced. A galaxy that made three supernovae contributes three supernova rows, each carrying that galaxy's distance. The rows are gathered into a table called events, and the counts of supernovae, kilonovae, and total events are printed.

In [ ]:
events = []

for i in range(len(galaxies)):
    d = galaxies["distance"].iloc[i]

    for _ in range(N_SN[i]):
        events.append({"type": "SN", "distance": d})

    for _ in range(N_KN[i]):
        events.append({"type": "KN", "distance": d})

events = pd.DataFrame(events)

print("Total SN:", (events["type"] == "SN").sum())
print("Total KN:", (events["type"] == "KN").sum())
print("Total events:", len(events))


## Convert to Apparent Magnitude

We assume a fixed absolute magnitude:

M = -16

The distance modulus is:

μ = 5 log10(d_Mpc) + 25

The apparent magnitude is:

m = M + μ



This works out how bright each event looks from Earth. It assumes a fixed intrinsic brightness of M_abs = -16, the absolute magnitude, and adds the distance modulus mu, the amount an object dims with distance. The result is the apparent magnitude m, a brightness scale where larger numbers mean fainter. The first few rows of the updated table are shown.

In [ ]:
M_abs = -16

events["mu"] = 5 * np.log10(events["distance"]*1e6) - 5
events["m"] = M_abs + events["mu"]

events.head()

### Brightness Range of Injected Events

This cell computes the apparent magnitude range of the simulated transient population.

- The minimum value of `m` corresponds to the brightest detected event.
- The maximum value of `m` corresponds to the faintest detected event.

This provides a quick validation that the injected supernovae and kilonovae span a physically reasonable magnitude range within the survey limits.


This reports the brightness range of the simulated events. It prints the smallest and largest apparent magnitude, which are the brightest and faintest events. Remember that in the magnitude system a smaller number means brighter.

In [ ]:
print("Brightest event:", events["m"].min())
print("Faintest event:", events["m"].max())

### Apparent Magnitude as a Function of Distance

This cell compares the simulated events to the theoretical distance–magnitude relation.

- A smooth distance grid `d` is created between the minimum and maximum event distances.


  where an absolute magnitude of M = -16 is assumed.
- The simulated event magnitudes are plotted as scatter points.
- The theoretical curve is overlaid for comparison.
- The y-axis is inverted because lower magnitudes correspond to brighter objects.

This verifies that the injected events follow the expected luminosity–distance scaling.


This shows how brightness falls off with distance. It plots the apparent magnitude of every event against its distance, then overlays a smooth line from the same M_abs = -16 formula for reference. The vertical axis is flipped so brighter is up, and the figure is saved to disk as a PNG.

In [ ]:

d = np.linspace(events["distance"].min(), 
                events["distance"].max(), 500)

m_model = -16 + 5*np.log10(d*1e6) - 5

plt.figure()
plt.scatter(events["distance"], events["m"])
plt.plot(d, m_model)

plt.xlabel("Distance (Mpc)")
plt.ylabel("Apparent Magnitude")
plt.gca().invert_yaxis()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\AppMagAsFunctionOfDistance.png', dpi=300, bbox_inches='tight')
plt.show()

### Apparent Magnitude Distribution: Supernovae vs Kilonovae

This cell compares the apparent magnitude distributions of Supernovae (SN) and Kilonovae (KN).

- The events are separated by type (`SN` and `KN`).
- Histograms of their apparent magnitudes (`m`) are plotted.
- A logarithmic y-axis is used to clearly show differences in event counts.
- This highlights the relative abundance of SN compared to the much rarer KN population.

The plot allows a direct visual comparison of brightness distributions and population imbalance between the two transient types.


This compares the brightness spread of the two event types. It draws overlapping histograms of apparent magnitude, one for supernovae and one for kilonovae; a histogram counts how many events land in each brightness bin. The vertical axis is put on a log scale so small counts stay visible, and the figure is saved.

In [ ]:
plt.hist(events[events['type']=='SN']['m'], label='SN')
plt.hist(events[events['type']=='KN']['m'], label='KN')
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\AppMagDistubtion.png', dpi=300, bbox_inches='tight')
plt.semilogy()

### Inspecting Kilonova Events in the Synthetic Sample

This cell filters the event catalogue to show only Kilonova (KN) events.


This filters the events table down to just the kilonova rows. Displaying it lets you inspect the handful of kilonovae the simulation produced. The output is that smaller table.

In [ ]:
events[events['type']=='KN']


## SN 1993J r-band Light Curve

This cell plots the observed r-band light curve of the Type IIb supernova SN 1993J.

- The x-axis shows time in days since explosion.
- The y-axis shows apparent R-band magnitude.
- The y-axis is inverted because smaller magnitudes are brighter.
- Each point is a real observational measurement.

This allows direct visual comparison with AT2017gfo.

This loads a real supernova light curve from sn1993j_rband_shifted.csv. SN 1993J was a Type IIb supernova seen in the nearby galaxy M81, and a light curve is how brightness changes over time. The code keeps only the first 20 days, plots the r-band magnitude against days since explosion, and saves the figure. The r-band is a red colour filter.

In [ ]:
df_93 = pd.read_csv(os.path.expanduser("~/Downloads/sn1993j_rband_shifted.csv"))

df_93_40 = df_93[df_93["t_days"] <= 20]

plt.figure(figsize=(6,4))
plt.scatter(df_93_40["t_days"], df_93_40["R_app"], s=25)

plt.gca().invert_yaxis()
plt.yticks([10.5, 11.0, 11.5, 12.0, 12.5, 13.0])
plt.xlabel("Days since explosion")
plt.ylabel("r-band magnitude")
plt.title("Supernova r-band Light Curve")
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\1993JLC.png', dpi=300, bbox_inches='tight')
plt.show()

### Loading the AT2017gfo r-band Kilonova Data

This cell loads the real photometric data for the kilonova AT2017gfo.

- The file contains multi-band observations compiled from the literature.
- Each row corresponds to a measurement at a given time since merger (`Phase`).
- The `mag` column contains the observed apparent magnitude.
- The `Band` column specifies the filter (we will later isolate r-band only).

This dataset provides a fully data-driven kilonova light curve for direct comparison with the Type IIb supernova.


This loads the real kilonova AT2017gfo from at2017gfo_rband.csv and keeps only its r-band points. AT2017gfo is the kilonova from the 2017 neutron star merger GW170817, the only kilonova so far with a well measured light curve. Some late or faint points are relabelled as upper limits, meaning the source was too dim to pin down; this mask follows Villar et al. 2017. The data is split into detections and upper limits, and the table is previewed.

In [ ]:
df_kn = pd.read_csv(os.path.expanduser("~/Downloads/at2017gfo_rband.csv"))
df_kn = df_kn[df_kn["band"] == "r"].copy()
df_kn_r = df_kn   # alias used in later cells

print("Number of r-band points:", len(df_kn))

# Upper-limit mask applied to AT2017gfo r-band data.
# Points after 8 days with mag < 20.5 are treated as upper limits based on
# visual inspection of the light curve, consistent with Villar et al. (2017).
# Points beyond 16 days are treated as upper limits throughout.
mask_kn = ((df_kn_r["Phase"] > 8) & (df_kn_r["mag"] < 20.5)) | (df_kn_r["Phase"] > 16)
kn_det = df_kn_r[~mask_kn]
kn_lim = df_kn_r[mask_kn]

df_kn.head()


### Selecting Only r-band Kilonova Data

This cell filters the full AT2017gfo dataset to keep only r-band measurements.

- The original table contains multiple photometric bands.
- We restrict to rows where `Band == "r"`.
- A copy of the filtered dataframe is created to avoid modifying the original data.

This ensures we are comparing SN 2011dh and AT2017gfo in the same photometric band.


### AT2017gfo r-band Light Curve


We apply a physical sanity filter:

- Use r-band only  
- Respect published upper limits  
- After 8 days, any point brighter than 20.5 mag is treated as an upper limit  

This prevents artificial late-time rebrightening.


## Apparent r-band Light Curves

This cell plots the observed r-band light curves of SN 1993J and AT2017gfo.

- The x-axis shows time in days since explosion (SN 1993J) or merger (AT2017gfo).
- The y-axis shows apparent r-band magnitude.
- The y-axis is inverted because smaller magnitudes are brighter.
- Each point represents a real observational measurement.

This allows direct visual comparison between a Type IIb supernova and a kilonova in the same photometric band.

This draws the AT2017gfo r-band light curve on its own. Real detections and upper limits use different markers so you can tell them apart. The figure is saved to disk.

In [ ]:
detections   = kn_det
upper_limits = kn_lim

plt.figure(figsize=(6,4))
plt.scatter(detections["Phase"], detections["mag"], s=25, label="Detection")
plt.scatter(upper_limits["Phase"], upper_limits["mag"], marker="v", s=40, label="Upper limit")

plt.gca().invert_yaxis()
plt.xlabel("Days since merger")
plt.ylabel("r-band magnitude")
plt.title("Kilonova r-band light curve")
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\AT2017gfoRawLC.png', dpi=300, bbox_inches='tight')
plt.show()


## Combined Apparent Light Curves

The apparent r-band light curves of SN 1993J and AT2017gfo are plotted together on the same axes. This comparison illustrates the different fading rates of the two transient types before any distance correction is applied. AT2017gfo is intrinsically fainter but fades far more rapidly than SN 1993J.

This overlays the supernova and kilonova on a single brightness against time plot. It reuses the phase and magnitude arrays prepared above for SN 1993J and AT2017gfo. The comparison figure is saved.

In [ ]:
phase_sn = df_93["t_days"]
r_sn = df_93["R_app"]

phase_kn_det = kn_det["Phase"]
r_kn_det = kn_det["mag"]
phase_kn_lim = kn_lim["Phase"]
r_kn_lim = kn_lim["mag"]

plt.figure(figsize=(8,5))
plt.scatter(phase_sn, r_sn, s=30, label="SN 1993J (Type IIb)")
plt.scatter(phase_kn_det, r_kn_det, s=30, label="AT2017gfo (Detection)")
plt.scatter(phase_kn_lim, r_kn_lim, marker="v", s=40, label="AT2017gfo (Upper limit)")

plt.xlabel("Days since explosion / merger")
plt.ylabel("Apparent r-band magnitude")
plt.title("Apparent r-band Light Curves")
plt.gca().invert_yaxis()
plt.xlim(0, 45)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\ApparentRBandLC.png', dpi=300, bbox_inches='tight')
plt.show()


## Absolute r-band Light Curves

This cell converts both light curves to absolute magnitude.

- The distance modulus is computed using:
  μ = 5 log10(d_Mpc × 10^6) − 5
- SN 1993J is assumed to be at 3.6 Mpc.
- AT2017gfo is assumed to be at 40 Mpc.
- Absolute magnitude is computed as:
  M = m − μ

This allows direct physical comparison of intrinsic brightness.

This removes the effect of distance so the two objects can be compared fairly. Using each object's known distance, 3.6 Mpc for SN 1993J and 40 Mpc for AT2017gfo, it subtracts the distance modulus to get absolute magnitude, the brightness each would show at a standard distance. It then plots and saves the absolute light curves. Mpc means megaparsec, a large unit of astronomical distance.

In [ ]:
d_93 = 3.6    # Mpc
d_kn = 40.0   # Mpc

mu_93 = 5 * np.log10(d_93 * 1e6) - 5
mu_kn = 5 * np.log10(d_kn * 1e6) - 5

M_93      = r_sn - mu_93
M_kn_det  = r_kn_det - mu_kn
M_kn_lim  = r_kn_lim - mu_kn

plt.figure(figsize=(8,5))
plt.scatter(phase_sn, M_93, s=30, label="SN 1993J (Type IIb)")
plt.scatter(phase_kn_det, M_kn_det, s=30, label="AT2017gfo (Detection)")
plt.scatter(phase_kn_lim, M_kn_lim, marker="v", s=60, edgecolor="black", linewidth=0.5, label="AT2017gfo (Upper limit)")

plt.xlabel("Days since explosion / merger")
plt.ylabel("Absolute r-band magnitude")
plt.title("Absolute r-band Light Curves")
plt.gca().invert_yaxis()
plt.xlim(0, 80)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\AbsoulteRBandLC.png', dpi=300, bbox_inches='tight')
plt.show()


## r-band Spline Fitting

The r-band data for both templates are sorted, cleaned of NaNs and duplicate time values, and used to fit smoothing splines. The smoothing parameter controls the balance between interpolation accuracy and curve smoothness. The fitted splines allow the template light curves to be evaluated at arbitrary times during the simulation.

This turns the scattered points into smooth curves that can be read at any time. First it sorts the data, drops missing values, and removes duplicate times, all of which the fitting routine requires. Then it fits a UnivariateSpline to each object; a spline is a smooth curve threaded through the data, and its s setting controls how tightly it follows the points. Finally it prints sanity checks confirming the smoothed curves contain no gaps and lie in a sensible range.

In [ ]:
# --- Sort data ---
idx_sn = np.argsort(phase_sn)
phase_sn_sorted = phase_sn.values[idx_sn]
M_sn_sorted     = M_93.values[idx_sn]

idx_kn = np.argsort(phase_kn_det)
phase_kn_sorted = phase_kn_det.values[idx_kn]
M_kn_sorted     = M_kn_det.values[idx_kn]


# --- CLEAN DATA (critical) ---
# remove NaNs
mask_sn = np.isfinite(phase_sn_sorted) & np.isfinite(M_sn_sorted)
phase_sn_sorted = phase_sn_sorted[mask_sn]
M_sn_sorted     = M_sn_sorted[mask_sn]

mask_kn = np.isfinite(phase_kn_sorted) & np.isfinite(M_kn_sorted)
phase_kn_sorted = phase_kn_sorted[mask_kn]
M_kn_sorted     = M_kn_sorted[mask_kn]

# remove duplicate times (required for spline)
phase_sn_sorted, idx_unique_sn = np.unique(phase_sn_sorted, return_index=True)
M_sn_sorted = M_sn_sorted[idx_unique_sn]

phase_kn_sorted, idx_unique_kn = np.unique(phase_kn_sorted, return_index=True)
M_kn_sorted = M_kn_sorted[idx_unique_kn]


# --- Fit splines ---
spline_sn = UnivariateSpline(phase_sn_sorted, M_sn_sorted, s=0.5)
spline_kn = UnivariateSpline(phase_kn_sorted, M_kn_sorted, s=3.9)

# Save raw splines
_spline_sn_raw = spline_sn
_spline_kn_raw = spline_kn


# --- Evaluation grids ---
t_sn = np.linspace(phase_sn_sorted.min(), phase_sn_sorted.max(), 500)
t_kn = np.linspace(phase_kn_sorted.min(), phase_kn_sorted.max(), 500)

M_sn_smooth = _spline_sn_raw(t_sn)
M_kn_smooth = _spline_kn_raw(t_kn)


# --- Sanity checks (must pass) ---
print("SN NaNs:", np.isnan(M_sn_smooth).any())
print("KN NaNs:", np.isnan(M_kn_smooth).any())
print("SN range:", M_sn_smooth.min(), M_sn_smooth.max())
print("KN range:", M_kn_smooth.min(), M_kn_smooth.max())

## r-band Template Normalisation

Both r-band splines are shifted so their peaks align at an absolute magnitude of -16. This normalisation places both templates on a common scale for injection into the simulation. Residual scatter and intrinsic diversity are combined into uncertainty bands that characterise the expected variation around each template.

This puts both templates on an equal footing. It shifts each smooth curve vertically so its peak sits at the same reference brightness of M_peak = -16, without altering the underlying fits. It also measures how far the real points scatter around each curve and combines that with an assumed intrinsic spread of 0.2 magnitudes to form uncertainty bands. The shifts and band widths are printed.

In [ ]:
# --- Normalise both templates to the same peak absolute magnitude ---

M_peak = -16.0

sn_offset = M_peak - M_sn_smooth.min()
kn_offset = M_peak - M_kn_smooth.min()

# Apply offsets without modifying raw splines
spline_sn = lambda t: _spline_sn_raw(t) + sn_offset
spline_kn = lambda t: _spline_kn_raw(t) + kn_offset

print(f"SN template shifted by {sn_offset:+.2f} mag (raw peak was {M_sn_smooth.min():.2f})")
print(f"KN template shifted by {kn_offset:+.2f} mag (raw peak was {M_kn_smooth.min():.2f})")


# --- Residual-scatter uncertainty bands ---

sn_resid_std = np.std(M_sn_sorted - _spline_sn_raw(phase_sn_sorted))
kn_resid_std = np.std(M_kn_sorted - _spline_kn_raw(phase_kn_sorted))

SN_INTRINSIC_SCATTER = 0.2   # mag, Type IIb diversity
KN_INTRINSIC_SCATTER = 0.2  # mag, kilonova diversity

sn_band = np.sqrt(sn_resid_std**2 + SN_INTRINSIC_SCATTER**2)
kn_band = np.sqrt(kn_resid_std**2 + KN_INTRINSIC_SCATTER**2)

print(f"SN residual scatter: {sn_resid_std:.3f} mag → band (incl. intrinsic): {sn_band:.3f} mag")
print(f"KN residual scatter: {kn_resid_std:.3f} mag → band (incl. intrinsic): {kn_band:.3f} mag")

## r-band Template Plot

The normalised r-band splines are plotted with 1-sigma uncertainty bands derived from residual scatter and assumed intrinsic diversity. The kilonova template fades steeply while the supernova template evolves more slowly, illustrating the photometric basis for the fade rate classifier.

This plots the two shifted templates together with their data points and uncertainty bands. The shaded bands show roughly one standard deviation, a measure of spread. The figure is saved to disk.

In [ ]:
# Plot normalised splines with 1σ uncertainty bands
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(phase_sn_sorted, M_sn_sorted + sn_offset, s=20, alpha=0.6, color="C0")
ax.scatter(phase_kn_sorted, M_kn_sorted + kn_offset, s=20, alpha=0.6, color="C1")
ax.scatter(phase_kn_lim, M_kn_lim + kn_offset, marker="v", s=50, edgecolor="black", color="C1", label="AT2017gfo (Upper limit)")

ax.plot(t_sn, M_sn_smooth + sn_offset, linewidth=2, color="C0", label="SN 1993J")
ax.plot(t_kn, M_kn_smooth + kn_offset, linewidth=2, color="C1", label="AT2017gfo")

ax.fill_between(t_sn,
                M_sn_smooth + sn_offset - sn_band,
                M_sn_smooth + sn_offset + sn_band,
                alpha=0.25, color="C0", label=f"SN 1σ band (±{sn_band:.2f} mag)")
ax.fill_between(t_kn,
                M_kn_smooth + kn_offset - kn_band,
                M_kn_smooth + kn_offset + kn_band,
                alpha=0.25, color="C1", label=f"KN 1σ band (±{kn_band:.2f} mag)")

ax.set_xlim(0, 20)
ax.invert_yaxis()
ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("Absolute r-band magnitude")
ax.set_title("Kilonova vs supernova r-band evolution")
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\RBandSplineFit.png', dpi=300, bbox_inches='tight')
plt.show()

This quantifies how sharply each object fades early on. It reads the smoothed brightness at day zero and day one and takes the difference for each template. It prints the one day fade for the kilonova and supernova and the ratio, showing the kilonova drops much faster.

In [ ]:
kn_day0 = M_kn_smooth[t_kn <= 0.01].mean()
kn_day1 = M_kn_smooth[np.abs(t_kn - 1.0) < 0.2].mean()

sn_day0 = M_sn_smooth[t_sn <= 0.1].mean()
sn_day1 = M_sn_smooth[np.abs(t_sn - 1.0) < 0.2].mean()

kn_fade = kn_day1 - kn_day0
sn_fade = sn_day1 - sn_day0

print(f"KN fade in first 24 hrs: {kn_fade:.3f} mag")
print(f"SN fade in first 24 hrs: {sn_fade:.3f} mag")
print(f"KN fades {kn_fade/sn_fade:.1f}x faster than SN")


#### Two-Epoch Survey Configuration

We simulate a two-epoch, 24-hour baseline survey designed to represent a
rapid-response LSST-style observing strategy. Observations are made at
t = 0 and t = 1 day after the first detection. This minimal cadence is
sufficient to measure a fade rate and a g-r colour rate, which are the
two features used in the classifier. The LSST single-visit r-band limiting
magnitude of 24.5 is applied. Events fainter than this limit at any epoch
are recorded as non-detections.


This sets up the event injection experiment used from here on. np.random.seed fixes the random numbers so runs repeat, t_obs sets two observing epochs at day 0 and day 1, and m_lim = 24.5 is the faintest an object can be and still be seen, matching a single visit of the Rubin Observatory LSST survey in the r-band. It also defines sample_distance, which draws a random distance between 50 and 300 Mpc spread evenly through volume, so nearby and far events appear in realistic proportions.

In [ ]:
np.random.seed(37)


t_obs = np.array([0, 1,])  # days

# LSST single-visit r-band limiting magnitude
m_lim = 24.5


def sample_distance(d_min=50, d_max=300):
    """Draw a distance uniformly in volume (proportional to d^2 dd)."""
    u = np.random.uniform()
    return (d_min**3 + u * (d_max**3 - d_min**3)) ** (1 / 3)


### Observed Light Curve Model

For each injected event, the following steps are applied:

1. Evaluate the absolute magnitude template spline at rest-frame time.
2. Remove pre-explosion epochs (t < 0).
3. Convert to apparent magnitude using the distance modulus:
   μ = 5 log10(d_Mpc × 10^6) − 5
4. Apply magnitude-dependent photometric uncertainty:
   σ = 0.1 mag at 20 mag,
   σ = 0.2 mag at 21 mag,
   linear interpolation between.

Gaussian noise is added to produce observed magnitudes.

This defines generate_observed_lightcurve, which fakes what a telescope would record for one event. Given the event type, an explosion time t0, and a distance, it reads the matching r-band template, blanks out times before the explosion, and converts absolute to apparent magnitude using the distance. It then adds Gaussian measurement noise that grows for fainter sources. It returns the noisy magnitudes and their error sizes.

In [ ]:
def generate_observed_lightcurve(event_type, t0, d_mpc):

    # Rest-frame time
    t_rest = t_obs - t0

    # Evaluate absolute magnitude template
    if event_type == "SN":
        M = spline_sn(t_rest)
    else:
        M = spline_kn(t_rest)

    # Remove pre-explosion epochs
    M = np.array(M)
    M[t_rest < 0] = np.nan

    # Convert to apparent magnitude
    mu = 5 * np.log10(d_mpc * 1e6) - 5
    m_true = M + mu

    # Magnitude-dependent photometric uncertainty (linear 0.1→0.2 over 20–21 mag)
    sigma = np.where(m_true <= 20, 0.1, np.where(m_true >= 21, 0.2, 0.1 + 0.1 * (m_true - 20)))

    # Add Gaussian noise
    m_obs = m_true + np.random.normal(0, sigma)

    return m_obs, sigma


### Injection of 1 Kilonova and 50 Supernovae

We inject:
- 1 kilonova
- 50 Type IIb supernovae

Distances are drawn uniformly between 50 and 300 Mpc.

Explosion times are drawn uniformly between −1 and 0 days,
allowing events to be partially evolved at first detection.

This configuration approximates the expected contamination level in a 1000 deg² survey over 96 hours.

This runs one example survey. It injects a single kilonova at a random distance and time, then fifty supernovae the same way, marking anything fainter than m_lim as not seen. The kilonova and supernova light curves are stored for the plots and classifier that follow.

In [ ]:
# --- Kilonova ---
kn_distance = sample_distance()
kn_t0 = np.random.uniform(-1, 0)
kn_lc, kn_sigma = generate_observed_lightcurve("KN", kn_t0, kn_distance)
kn_lc[kn_lc > m_lim] = np.nan   # epochs below limiting magnitude → non-detection

print(f"KN: d = {kn_distance:.1f} Mpc,  detected epochs = {np.sum(~np.isnan(kn_lc))} / {len(t_obs)}")

# --- Supernovae ---
sn_lcs = []
sn_sigmas = []

for _ in range(50):
    d = sample_distance()
    t0 = np.random.uniform(-1, 0)
    m_obs, sig = generate_observed_lightcurve("SN", t0, d)
    m_obs[m_obs > m_lim] = np.nan   # apply limiting magnitude
    sn_lcs.append(m_obs)
    sn_sigmas.append(sig)


### 24-Hour Light Curve Comparison

Observed light curves are plotted over the four survey epochs.

Supernovae are shown in grey.
The kilonova is shown in red.

Over a 3 day baseline:
- Supernovae exhibit minimal magnitude evolution.
- Kilonovae exhibit rapid fading.

Separation between populations is therefore driven primarily by the measured light curve slope.

This plots that example survey. Each supernova and the single kilonova are drawn with error bars showing measurement uncertainty. The figure is saved to disk.

In [ ]:
plt.figure(figsize=(8,6))

# Supernovae
for lc, sig in zip(sn_lcs, sn_sigmas):
    plt.errorbar(t_obs, lc, yerr=sig, fmt='o-', alpha=0.4)

# Kilonova
plt.errorbar(t_obs, kn_lc, yerr=kn_sigma, fmt='o-', linewidth=2, label="Kilonova")

plt.gca().invert_yaxis()
plt.xlabel("Time (days)")
plt.ylabel("Apparent magnitude")
plt.title("Survey")
plt.legend()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\KNSurvey2epoch.png', dpi=300, bbox_inches='tight')
plt.show()

### Fade-Rate Classifier

To separate kilonovae from supernovae, a simple classifier based on how fast the light curve fades was used. Kilonovae fade much faster than supernovae during the first few days after peak brightness.

For each simulated light curve, the fade rate was estimated by comparing the brightness at the first detected observation and the last detected observation. Only valid detections were used. If fewer than two detections were available, the fade rate could not be calculated.

A threshold of **0.35 magnitudes per day** was used for classification.  
- If an event faded faster than this threshold, it was classified as a **kilonova candidate**.  
- If it faded more slowly, it was classified as a **supernova**.

The histogram shows the distribution of fade rates measured for the simulated supernova population. The red vertical line marks the fade rate of the injected kilonova, and the dashed black line shows the classification threshold.

This builds the first, simplest classifier. It defines fade_rate, how many magnitudes an object dims per day between the two epochs, and flags anything fading faster than KN_FADE_THRESHOLD = 0.35 magnitudes per day as a kilonova candidate. It applies this to the example kilonova and the fifty supernovae, prints how many supernovae are wrongly flagged, and plots the fade rate distribution with the threshold marked.

In [ ]:
# --- Fade-rate classifier ---
# Kilonovae fade > ~ 
KN_FADE_THRESHOLD = 0.35 # mag/day


def fade_rate(m_obs):
    """Linear fade rate (mag/day) over detected epochs. Returns NaN if < 2 detections."""
    valid = ~np.isnan(m_obs)
    if valid.sum() < 2:
        return np.nan
    t_valid = t_obs[valid]
    m_valid = m_obs[valid]
    # Positive = fading (getting fainter)
    return (m_valid[-1] - m_valid[0]) / (t_valid[-1] - t_valid[0])


kn_rate = fade_rate(kn_lc)
sn_rates = np.array([fade_rate(lc) for lc in sn_lcs])

# Classify
kn_label = "KN ✓" if (not np.isnan(kn_rate) and kn_rate > KN_FADE_THRESHOLD) else "SN ✗ (missed)"
sn_fp = np.sum(sn_rates[~np.isnan(sn_rates)] > KN_FADE_THRESHOLD)
sn_valid = np.sum(~np.isnan(sn_rates))

print(f"KN fade rate : {kn_rate:.3f} mag/day  →  {kn_label}")
print(f"SN fade rates: {[f'{r:.3f}' for r in sn_rates if not np.isnan(r)]}")
print(f"SN false positives (misclassified as KN): {sn_fp} / {sn_valid}")

# Plot fade-rate distributions
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(sn_rates[~np.isnan(sn_rates)], bins=10, label="SN fade rates", alpha=0.7)
if not np.isnan(kn_rate):
    ax.axvline(kn_rate, color="red", linewidth=2, label=f"KN fade rate ({kn_rate:.2f} mag/day)")
ax.axvline(KN_FADE_THRESHOLD, color="black", linestyle="--", label=f"Threshold ({KN_FADE_THRESHOLD} mag/day)")
ax.set_xlabel("Fade rate (mag/day)")
ax.set_ylabel("Count")
ax.set_title("Fade-rate classifier")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\FadeRateClassifer.png', dpi=300, bbox_inches='tight')
plt.show()


### Multi-Seed Survey Simulation

The survey was simulated **100 times with different random seeds**.

In each simulation:

- **1 kilonova** and **50 supernovae** were injected at random distances and explosion times.
- Light curves were generated using the survey cadence and limiting magnitude.
- Fade rates were calculated and the same **0.35 mag/day threshold** was used for classification.

This scales the single example up to a proper experiment. It repeats the survey for one hundred random seeds, each injecting one kilonova and fifty supernovae, and records for every seed whether the kilonova was caught and how many supernovae leaked through. The per seed results are gathered into results_df, and the overall detection rate and mean number of false positives are printed.

In [ ]:
# --- Multi-seed survey simulation (seeds 1–100) ---
# Re-uses all existing functions and parameters unchanged.

results = []

for seed in range(1, 101):
    np.random.seed(seed)

    # Inject 1 KN
    kn_d = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)
    kn_lc, _ = generate_observed_lightcurve("KN", kn_t0, kn_d)
    kn_lc[kn_lc > m_lim] = np.nan

    # Inject 50 SNe
    sn_rates_seed = []
    for _ in range(50):
        d = sample_distance()
        t0 = np.random.uniform(-1, 0)
        m_obs, _ = generate_observed_lightcurve("SN", t0, d)
        m_obs[m_obs > m_lim] = np.nan
        sn_rates_seed.append(fade_rate(m_obs))

    kn_r = fade_rate(kn_lc)
    sn_r = np.array(sn_rates_seed)

    results.append({
        "seed":               seed,
        "kn_distance":        kn_d,
        "kn_fade_rate":       kn_r,
        "kn_detected":        (not np.isnan(kn_r)) and (kn_r > KN_FADE_THRESHOLD),
        "sn_false_positives": int(np.sum(sn_r[~np.isnan(sn_r)] > KN_FADE_THRESHOLD)),
        "sn_valid":           int(np.sum(~np.isnan(sn_r))),
    })

results_df = pd.DataFrame(results)
print(f"KN detection rate:       {results_df['kn_detected'].mean()*100:.1f}%  "
      f"({results_df['kn_detected'].sum()} / 100 seeds)")
print(f"Mean SN false positives: {results_df['sn_false_positives'].mean():.2f} per realisation")


## Bootstrap Confidence Intervals

Bootstrap resampling over the 100 seeds is used to compute 95% confidence
intervals on the KN detection rate and mean SN false positive count. Note
that only 100 resamples are used here for speed. Standard practice recommends
at least 1000 resamples for stable confidence interval estimates. The intervals
reported here should be interpreted as approximate.


This attaches error bars to those headline numbers using bootstrapping, which repeatedly resamples the one hundred seeds with replacement to see how much the answer wobbles. From the spread of one hundred resamples it forms 95 percent confidence intervals, the range the true value most likely falls in. It prints the detection rate and false positive count each with its interval.

In [ ]:
# --- Bootstrap confidence intervals on detection rate (100 resamples) ---
n_seeds = len(results_df)
kn_detected_arr = results_df["kn_detected"].values
sn_fp_arr = results_df["sn_false_positives"].values

rng = np.random.default_rng(42)
boot_kn_rate = []
boot_sn_fp   = []
for _ in range(100):
    idx = rng.integers(0, n_seeds, size=n_seeds)
    boot_kn_rate.append(kn_detected_arr[idx].mean())
    boot_sn_fp.append(sn_fp_arr[idx].mean())

boot_kn_rate = np.array(boot_kn_rate)
boot_sn_fp   = np.array(boot_sn_fp)

kn_lo, kn_hi = np.percentile(boot_kn_rate, [2.5, 97.5])
fp_lo, fp_hi = np.percentile(boot_sn_fp,   [2.5, 97.5])

print(f"KN detection rate: {results_df['kn_detected'].mean()*100:.1f}%  "
      f"95% CI: [{kn_lo*100:.1f}%, {kn_hi*100:.1f}%]")
print(f"Mean SN false positives: {results_df['sn_false_positives'].mean():.2f}  "
      f"95% CI: [{fp_lo:.2f}, {fp_hi:.2f}]")

### Classifier Performance Across 100 Simulations

The performance of the fade-rate classifier was examined using 100 independent survey simulations.

The left panel shows the distribution of measured kilonova fade rates. The dashed line marks the classification threshold of **0.35 mag/day**. 

The middle panel shows the number of supernovae that are incorrectly classified as kilonovae in each simulation. On average, several fast-fading supernovae contaminate the kilonova sample.

The right panel shows kilonova fade rate as a function of distance. Orange points indicate correctly detected kilonovae, while grey points show missed events. Missed detections occur when the measured fade rate falls below the classification threshold.

This summarises the one hundred seed run in three panels. The panels show the distribution of kilonova fade rates, the count of supernova false positives per run, and kilonova detection against distance. The combined figure is saved to disk.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
# Panel 1 — KN fade-rate distribution
axes[0].hist(results_df["kn_fade_rate"].dropna(), bins=20, color="C1", alpha=0.8)
axes[0].axvline(KN_FADE_THRESHOLD, color="black", linestyle="--",
                label=f"Threshold ({KN_FADE_THRESHOLD} mag/day)")
axes[0].set_xlabel("KN fade rate (mag/day)")
axes[0].set_ylabel("Count")
axes[0].set_title("KN fade rate — 100 seeds")
axes[0].legend(frameon=False)

# Panel 2 — SN false-positive count per realisation
fp_max = max(results_df["sn_false_positives"].max(), 1)
axes[1].hist(results_df["sn_false_positives"], bins=range(0, fp_max + 2),
             color="C0", alpha=0.8, align="left")
axes[1].set_xlabel("SN false positives per realisation")
axes[1].set_ylabel("Count")
axes[1].set_title("SN false positives (50 SNe/seed)")

# Panel 3 — KN detection vs distance
colors = results_df["kn_detected"].map({True: "C1", False: "grey"})
axes[2].scatter(results_df["kn_distance"], results_df["kn_fade_rate"],
                c=colors, s=30, alpha=0.8)
axes[2].axhline(KN_FADE_THRESHOLD, color="black", linestyle="--")
axes[2].set_xlabel("KN distance (Mpc)")
axes[2].set_ylabel("KN fade rate (mag/day)")
axes[2].set_title("KN detection vs distance\n(orange = detected, grey = missed)")

plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\3Graph.png', dpi=300, bbox_inches='tight')
plt.show()

### ROC Curve Calculation

The classifier threshold is scanned across the full range of measured fade rates. For each threshold, events with a fade rate above the threshold are classified as kilonova candidates.

At every threshold the fraction of real kilonovae correctly identified and the fraction of supernovae incorrectly classified as kilonovae are calculated.

These values are used to construct the Receiver Operating Characteristic (ROC) curve. The ROC curve shows how the detection efficiency changes as the classification threshold is varied.

The area under the ROC curve (AUC) is then computed to summarise the overall performance of the classifier. A value close to one indicates strong separation between kilonovae and supernovae.

Finally, the operating point corresponding to the adopted threshold of **0.35 mag/day** is identified on the ROC curve.

This rebuilds the one hundred seed simulation but keeps every individual fade rate rather than just pass or fail counts. Kilonovae are labelled 1 and supernovae 0, and the two are stacked into score and label arrays for a threshold study. It prints how many events of each type produced a usable fade rate.

In [ ]:
# --- Build score arrays for ROC analysis ---
# Collect all per-event fade rates across all 100 seeds.

kn_scores_list = []
sn_scores_list = []

for seed in range(1, 101):
    np.random.seed(seed)

    # KN
    kn_d = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)
    kn_lc_r, _ = generate_observed_lightcurve("KN", kn_t0, kn_d)
    kn_lc_r[kn_lc_r > m_lim] = np.nan
    r = fade_rate(kn_lc_r)
    if not np.isnan(r):
        kn_scores_list.append(r)

    # 50 SNe
    for _ in range(50):
        d = sample_distance()
        t0 = np.random.uniform(-1, 0)
        m_obs, _ = generate_observed_lightcurve("SN", t0, d)
        m_obs[m_obs > m_lim] = np.nan
        r = fade_rate(m_obs)
        if not np.isnan(r):
            sn_scores_list.append(r)

all_scores = np.concatenate([kn_scores_list, sn_scores_list])
all_labels = np.concatenate([np.ones(len(kn_scores_list)), np.zeros(len(sn_scores_list))])

n_kn = len(kn_scores_list)
n_sn = len(sn_scores_list)

print(f"KN events with valid fade rate: {n_kn}")
print(f"SN events with valid fade rate: {n_sn}")


## ROC Threshold Sweep

The classifier threshold is swept from the minimum to maximum observed fade rate. At each threshold, the true positive rate and false positive rate are computed. The AUC is calculated using the trapezoidal rule after sorting by false positive rate, and the operating point at the chosen threshold is marked.

This measures the classifier across every possible cutoff instead of one fixed threshold. Sweeping the fade rate threshold, it records at each step the true positive rate, the fraction of kilonovae caught, and the false positive rate, the fraction of supernovae wrongly flagged, tracing out a ROC curve. The area under that curve, the AUC, condenses overall performance into one number, where 1.0 is perfect and 0.5 is random guessing. It also locates the operating point at the chosen 0.35 threshold.

In [ ]:
# Sweep thresholds from min to max fade rate
thresholds = np.linspace(all_scores.min() - 0.01, all_scores.max() + 0.01, 500)

tpr_vals, fpr_vals, prec_vals = [], [], []
for t in thresholds:
    pred_pos = all_scores >= t
    tp = ((pred_pos) & (all_labels == 1)).sum()
    fp = ((pred_pos) & (all_labels == 0)).sum()
    tpr_vals.append(tp / n_kn if n_kn > 0 else 0)
    fpr_vals.append(fp / n_sn if n_sn > 0 else 0)
    prec_vals.append(tp / (tp + fp) if (tp + fp) > 0 else 1.0)

tpr_vals  = np.array(tpr_vals)
fpr_vals  = np.array(fpr_vals)
prec_vals = np.array(prec_vals)

# AUC via trapezoidal rule (sort by FPR for proper integration)
sort_idx = np.argsort(fpr_vals)
auc = np.trapezoid(tpr_vals[sort_idx], fpr_vals[sort_idx])

# Mark operating point at KN_FADE_THRESHOLD
op_idx = np.argmin(np.abs(thresholds - KN_FADE_THRESHOLD))
op_fpr  = fpr_vals[op_idx]
op_tpr  = tpr_vals[op_idx]
op_prec = prec_vals[op_idx]
op_rec  = tpr_vals[op_idx]   # recall = TPR

## AUC Bootstrap Confidence Interval

Bootstrap resampling over the per-event score arrays is used to compute a 95% confidence interval on the AUC. KN and SN score arrays are resampled independently to preserve the class sizes. This quantifies the uncertainty in the AUC arising from the finite number of simulated events.

This puts an uncertainty range on the AUC score. It resamples the kilonova and supernova fade rates with replacement one hundred times, rebuilding the ROC curve and its area each time. From the spread it prints the AUC with a 95 percent confidence interval.

In [ ]:
# --- Bootstrap confidence interval on AUC (100 resamples) ---
kn_scores = all_scores[all_labels == 1]
sn_scores = all_scores[all_labels == 0]

boot_aucs = []
rng2 = np.random.default_rng(42)
for _ in range(100):
    kn_boot = rng2.choice(kn_scores, size=len(kn_scores), replace=True)
    sn_boot = rng2.choice(sn_scores, size=len(sn_scores), replace=True)
    boot_all_scores = np.concatenate([kn_boot, sn_boot])
    boot_all_labels = np.concatenate([np.ones(len(kn_boot)), np.zeros(len(sn_boot))])

    boot_tpr, boot_fpr = [], []
    for t in thresholds:
        pred = boot_all_scores >= t
        tp = ((pred) & (boot_all_labels == 1)).sum()
        fp = ((pred) & (boot_all_labels == 0)).sum()
        boot_tpr.append(tp / len(kn_boot) if len(kn_boot) > 0 else 0)
        boot_fpr.append(fp / len(sn_boot) if len(sn_boot) > 0 else 0)

    s = np.argsort(boot_fpr)
    boot_aucs.append(np.trapezoid(np.array(boot_tpr)[s], np.array(boot_fpr)[s]))

auc_lo, auc_hi = np.percentile(boot_aucs, [2.5, 97.5])
print(f"AUC = {auc:.3f}  95% CI: [{auc_lo:.3f}, {auc_hi:.3f}]")

## 1D ROC Curve

The ROC curve for the fade rate alone classifier is plotted with its AUC and 95% confidence interval. The operating point at the 0.35 mag/day threshold is marked. This establishes the baseline classifier performance against which the two-feature classifier will be compared.

This draws the ROC curve for the fade rate classifier, with a diagonal line marking random guessing for comparison. The chosen 0.35 magnitudes per day operating point is marked as a dot. The figure is saved to disk.

In [ ]:
fig, ax = plt.subplots(figsize=(8,7))

ax.plot(fpr_vals, tpr_vals, linewidth=2, label=f"ROC (AUC = {auc:.3f}, 95% CI [{auc_lo:.3f}–{auc_hi:.3f}])")
ax.plot([0,1], [0,1], "k--", linewidth=1, label="Random classifier")

ax.scatter([op_fpr], [op_tpr], s=80, zorder=5,
           label=f"Threshold = {KN_FADE_THRESHOLD} mag/day\n"
                 f"(TPR={op_tpr:.2f}, FPR={op_fpr:.2f})")

ax.set_xlabel("False Positive Rate (SN contamination)")
ax.set_ylabel("True Positive Rate (KN completeness)")
ax.set_title("ROC Curve – Fade-rate classifier")
ax.legend(frameon=False, fontsize=9)
ax.set_xlim(0,1)
ax.set_ylim(0,1)
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\1DROCCUrve.png', dpi=300, bbox_inches='tight')
plt.show()


## Loading g-band Light Curve Templates

We load the g-band observations for SN 1993J and AT2017gfo from the standardised CSV files (columns: `t_days`, `mag`, `mag_err`, `band`, `upperlimit`). For AT2017gfo we keep only the 52 real detections — the 11 upper limits mark epochs where the rapidly fading kilonova fell below the g-band detection threshold.

This loads the second colour for both objects from sn1993j_gband.csv and at2017gfo_gband.csv. The g-band is a bluer filter than the r-band, and comparing the two gives colour information. It prints a short summary of the point count, time span, and magnitude range for each dataset.

In [ ]:
df_sn_g = pd.read_csv(os.path.expanduser('~/Downloads/sn1993j_gband.csv'))
df_kn_g = pd.read_csv(os.path.expanduser('~/Downloads/at2017gfo_gband.csv'))

print(f"SN 1993J g-band : {len(df_sn_g)} points, "
      f"t = {df_sn_g['t_days'].min():.1f}-{df_sn_g['t_days'].max():.1f} d, "
      f"mag = {df_sn_g['mag'].min():.2f}-{df_sn_g['mag'].max():.2f}")

print(f"AT2017gfo g-band : {len(df_kn_g)} points, "
      f"t = {df_kn_g['t_days'].min():.2f}-{df_kn_g['t_days'].max():.2f} d, "
      f"mag = {df_kn_g['mag'].min():.2f}-{df_kn_g['mag'].max():.2f}")

## Raw g-band Light Curves

The raw apparent g-band light curves for both templates are plotted. This provides a visual check of the data quality and time coverage before any conversion to absolute magnitude. The rapidly declining AT2017gfo g-band brightness is immediately apparent compared to the slower evolution of SN 1993J.

This plots the raw g-band light curves for SN 1993J and AT2017gfo side by side, one object per panel. It is a quick look at the new data before any processing. The figure is saved to disk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df_sn_g["t_days"], df_sn_g["mag"], s=25, label="SN 1993J")
axes[0].invert_yaxis()
axes[0].set_xlabel("Days since explosion")
axes[0].set_ylabel("g-band magnitude")
axes[0].set_title("SN 1993J g-band light curve")
axes[0].legend()

axes[1].scatter(df_kn_g["t_days"], df_kn_g["mag"], s=25, label="AT2017gfo")
axes[1].invert_yaxis()
axes[1].set_xlabel("Days since merger")
axes[1].set_ylabel("g-band magnitude")
axes[1].set_title("AT2017gfo g-band light curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\RawGBand.png', dpi=300, bbox_inches='tight')
plt.show()

## AT2017gfo g-band Absolute Magnitudes

The AT2017gfo g-band apparent magnitudes are converted to absolute magnitudes using its known distance of 40 Mpc. A manual upper limit mask is applied using hardcoded time windows corresponding to epochs identified as non-detections in the literature. The resulting detections and upper limits are stored separately.

This prepares the kilonova g-band data. Points in several time windows, identified by eye from the published photometry, are marked as upper limits where the source was not clearly detected. The remaining points are converted to absolute magnitude using the 40 Mpc distance, then plotted and saved.

In [ ]:
d_kn_g = 40.0  # Mpc
mu_kn_g = 5 * np.log10(d_kn_g * 1e6) - 5

# Manual upper limit mask for AT2017gfo g-band data.
# Time windows were identified by visual inspection of the published
# AT2017gfo photometry. Points in these windows correspond to epochs
# where the source was not significantly detected and published flux
# values represent upper bounds only.
mask_kn_g = (
    ((df_kn_g["t_days"] > 4.323) & (df_kn_g["t_days"] < 4.523)) |
    ((df_kn_g["t_days"] > 6.976) & (df_kn_g["t_days"] < 6.983)) |
    ((df_kn_g["t_days"] > 7.923) & (df_kn_g["t_days"] < 8.123)) |
    (df_kn_g["t_days"] > 9.523)
)

kn_g_det = df_kn_g[~mask_kn_g].copy()
kn_g_lim = df_kn_g[mask_kn_g].copy()

kn_g_det["M"] = kn_g_det["mag"] - mu_kn_g
kn_g_lim["M"] = kn_g_lim["mag"] - mu_kn_g

plt.figure(figsize=(8, 5))
plt.scatter(kn_g_det["t_days"], kn_g_det["M"], s=25, label="AT2017gfo g (Detection)")
plt.scatter(kn_g_lim["t_days"], kn_g_lim["M"], marker="v", s=60,
            edgecolor="black", linewidth=0.5, label="AT2017gfo g (Upper limit)")
plt.gca().invert_yaxis()
plt.xlabel("Days since merger")
plt.ylabel("Absolute g-band magnitude")
plt.title("AT2017gfo Absolute g-band Light Curve")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\AT2017gfoGbandABS_LC.png', dpi=300, bbox_inches='tight')
plt.show()


## SN 1993J g-band Absolute Magnitudes

The SN 1993J g-band data are converted to absolute magnitudes using its distance of 3.6 Mpc. The phase and absolute magnitude arrays are extracted ready for spline fitting. This step mirrors the r-band treatment applied earlier and ensures both templates are on a consistent absolute magnitude scale.

This assembles the g-band brightness of both objects in absolute magnitude, ready for fitting. SN 1993J is converted with its 3.6 Mpc distance modulus, while the kilonova values come from the detections and upper limits split in the previous cell. Nothing is printed; the arrays are stored for the spline step.

In [ ]:
mu_93_g = 5 * np.log10(d_93 * 1e6) - 5   # d_93 = 3.6 Mpc, defined in cell 30

phase_sn_g = df_sn_g["t_days"]
M_sn_g     = df_sn_g["mag"] - mu_93_g

phase_kn_g_det = kn_g_det["t_days"]
M_kn_g_det     = kn_g_det["M"]

phase_kn_g_lim = kn_g_lim["t_days"]
M_kn_g_lim     = kn_g_lim["M"]

## g-band Spline Fitting

The g-band data for both templates are sorted, cleaned, and used to fit smoothing splines. The same cleaning procedure applied to the r-band data is repeated here. The fitted splines are saved as raw spline objects so that the normalisation offset can be applied separately without modifying the underlying fits.

This fits smooth g-band templates the same way the r-band ones were built. It sorts the points, removes missing values and duplicate times, then fits a UnivariateSpline to each object, with a smoothing setting chosen by eye. A spline is a smooth curve drawn through data. It prints checks confirming the curves have no gaps and lie in a sensible range.

In [ ]:
# Smoothing parameters (s) were chosen by visual inspection of spline residuals.
# s=0.5 for SN 1993J gives a close fit to the well-sampled light curve.
# s=3.0 for AT2017gfo allows more flexibility given the sparser g-band coverage.

# Sort data
idx_sn_g = np.argsort(phase_sn_g)
phase_sn_g_sorted = phase_sn_g.values[idx_sn_g]
M_sn_g_sorted = M_sn_g.values[idx_sn_g]

idx_kn_g = np.argsort(phase_kn_g_det)
phase_kn_g_sorted = phase_kn_g_det.values[idx_kn_g]
M_kn_g_sorted = M_kn_g_det.values[idx_kn_g]


# --- CLEAN DATA (critical) ---
# Remove NaNs
mask_sn_g = np.isfinite(phase_sn_g_sorted) & np.isfinite(M_sn_g_sorted)
phase_sn_g_sorted = phase_sn_g_sorted[mask_sn_g]
M_sn_g_sorted     = M_sn_g_sorted[mask_sn_g]

mask_kn_g = np.isfinite(phase_kn_g_sorted) & np.isfinite(M_kn_g_sorted)
phase_kn_g_sorted = phase_kn_g_sorted[mask_kn_g]
M_kn_g_sorted     = M_kn_g_sorted[mask_kn_g]

# Remove duplicate times (required for spline)
phase_sn_g_sorted, idx_unique_sn_g = np.unique(phase_sn_g_sorted, return_index=True)
M_sn_g_sorted = M_sn_g_sorted[idx_unique_sn_g]

phase_kn_g_sorted, idx_unique_kn_g = np.unique(phase_kn_g_sorted, return_index=True)
M_kn_g_sorted = M_kn_g_sorted[idx_unique_kn_g]

# --- Fit splines ---
spline_sn_g = UnivariateSpline(phase_sn_g_sorted, M_sn_g_sorted, s=0.5)
spline_kn_g = UnivariateSpline(phase_kn_g_sorted, M_kn_g_sorted, s=3.0)

# Save raw splines
_spline_sn_g_raw = spline_sn_g
_spline_kn_g_raw = spline_kn_g

# --- Evaluation grids ---
t_sn_g = np.linspace(phase_sn_g_sorted.min(), phase_sn_g_sorted.max(), 500)
t_kn_g = np.linspace(phase_kn_g_sorted.min(), phase_kn_g_sorted.max(), 500)

M_sn_g_smooth = _spline_sn_g_raw(t_sn_g)
M_kn_g_smooth = _spline_kn_g_raw(t_kn_g)

# --- Sanity checks ---
print("SN NaN:", np.isnan(M_sn_g_smooth).any())
print("KN NaN:", np.isnan(M_kn_g_smooth).any())
print("SN range:", M_sn_g_smooth.min(), M_sn_g_smooth.max())
print("KN range:", M_kn_g_smooth.min(), M_kn_g_smooth.max())

## g-band Normalisation

The g-band splines are shifted using the same offset computed for the r-band templates. This anchors the g-band normalisation to the r-band and preserves the physical g-r colour difference between the two objects. Independent g-band normalisation would artificially eliminate the colour separation that makes this feature informative.

This normalises the g-band templates using the very same vertical shifts found for the r-band, which preserves the real colour difference between the objects. It rebuilds the uncertainty bands and, as a check, prints the g-r colour of each at day one; a kilonova should look redder than a supernova. It then plots the g-band templates with their data and bands.

In [ ]:
# Normalise g-band using the SAME offset as r-band
# This preserves the real g-r colour difference between the two objects

spline_sn_g = lambda t: _spline_sn_g_raw(t) + sn_offset   # sn_offset from r-band cell
spline_kn_g = lambda t: _spline_kn_g_raw(t) + kn_offset   # kn_offset from r-band cell

# Residual scatter + uncertainty bands (unchanged)
sn_resid_std_g = np.std(M_sn_g_sorted - _spline_sn_g_raw(phase_sn_g_sorted))
kn_resid_std_g = np.std(M_kn_g_sorted - _spline_kn_g_raw(phase_kn_g_sorted))

sn_band_g = np.sqrt(sn_resid_std_g**2 + SN_INTRINSIC_SCATTER**2)
kn_band_g = np.sqrt(kn_resid_std_g**2 + KN_INTRINSIC_SCATTER**2)

# Sanity check — print peak g-r colour for both objects
t_check = np.array([1.0])
gr_sn_peak = spline_sn_g(t_check) - spline_sn(t_check)
gr_kn_peak = spline_kn_g(t_check) - spline_kn(t_check)
print(f"SN 1993J  g-r at day 1: {gr_sn_peak[0]:+.2f} mag")
print(f"AT2017gfo g-r at day 1: {gr_kn_peak[0]:+.2f} mag")
print("KN should be redder (larger g-r) than SN")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(phase_sn_g_sorted, M_sn_g_sorted + sn_offset, s=20, alpha=0.8, color="C0")
ax.scatter(phase_kn_g_sorted, M_kn_g_sorted + kn_offset, s=20, alpha=0.8, color="C1")
ax.scatter(phase_kn_g_lim, M_kn_g_lim + kn_offset, marker="v", s=50,
           edgecolor="black", color="C1", label="AT2017gfo (Upper limit)")

ax.plot(t_sn_g, spline_sn_g(t_sn_g), linewidth=2, color="C0", label="SN 1993J spline")
ax.plot(t_kn_g, spline_kn_g(t_kn_g), linewidth=2, color="C1", label="AT2017gfo spline")

ax.fill_between(t_kn_g,
                spline_kn_g(t_kn_g) - kn_band_g,
                spline_kn_g(t_kn_g) + kn_band_g,
                alpha=0.25, color="C1", label=f"KN 1σ band ({kn_band_g:.2f} mag)")

ax.set_xlim(0, 20)
ax.invert_yaxis()
ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("Absolute g-band magnitude")
ax.set_title("g-band spline templates")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Side-by-Side Template Plots

The normalised r-band and g-band spline templates are plotted side by side with 1-sigma uncertainty bands. This allows a direct visual comparison of the two bands for both templates. The different rates of decline in g and r are the physical basis for the colour rate feature introduced in the next section.

This places the r-band and g-band templates in one two panel figure. Each panel shows the smooth curves, the data points, and the uncertainty bands. The combined figure is saved to disk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- r-band ---
ax = axes[0]
ax.scatter(phase_sn_sorted, M_sn_sorted + sn_offset, s=20, alpha=0.6, color="C0")
ax.scatter(phase_kn_sorted, M_kn_sorted + kn_offset, s=20, alpha=0.6, color="C1")
ax.scatter(phase_kn_lim, M_kn_lim + kn_offset, marker="v", s=50,
           edgecolor="black", color="C1", label="AT2017gfo (Upper limit)")
ax.plot(t_sn, M_sn_smooth + sn_offset, linewidth=2, color="C0", label="SN 1993J spline")
ax.plot(t_kn, M_kn_smooth + kn_offset, linewidth=2, color="C1", label="AT2017gfo spline")
ax.fill_between(t_sn, M_sn_smooth + sn_offset - sn_band,
                M_sn_smooth + sn_offset + sn_band, alpha=0.25, color="C0")
ax.fill_between(t_kn, M_kn_smooth + kn_offset - kn_band,
                M_kn_smooth + kn_offset + kn_band, alpha=0.25, color="C1")
ax.set_xlim(0, 20)
ax.invert_yaxis()
ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("Absolute r-band magnitude")
ax.set_title("r-band spline templates")
ax.legend(frameon=False)

# --- g-band ---
# Uses same sn_offset / kn_offset as r-band (physically correct normalisation)
ax = axes[1]
ax.scatter(phase_sn_g_sorted, M_sn_g_sorted + sn_offset, s=20, alpha=0.6, color="C0")
ax.scatter(phase_kn_g_sorted, M_kn_g_sorted + kn_offset, s=20, alpha=0.6, color="C1")
ax.scatter(phase_kn_g_lim, M_kn_g_lim + kn_offset, marker="v", s=50,
           edgecolor="black", color="C1", label="AT2017gfo (Upper limit)")
ax.plot(t_sn_g, spline_sn_g(t_sn_g), linewidth=2, color="C0", label="SN 1993J spline")
ax.plot(t_kn_g, spline_kn_g(t_kn_g), linewidth=2, color="C1", label="AT2017gfo spline")
ax.fill_between(t_sn_g, spline_sn_g(t_sn_g) - sn_band_g,
                spline_sn_g(t_sn_g) + sn_band_g, alpha=0.25, color="C0")
ax.fill_between(t_kn_g, spline_kn_g(t_kn_g) - kn_band_g,
                spline_kn_g(t_kn_g) + kn_band_g, alpha=0.25, color="C1")
ax.set_xlim(0, 20)
ax.invert_yaxis()
ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("Absolute g-band magnitude")
ax.set_title("g-band spline templates")
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\RandGBandSplines.png', dpi=300, bbox_inches='tight')
plt.show()


## g-r Colour Evolution

The g-r colour is computed from the corrected splines on a common time grid and plotted for both templates. AT2017gfo reddens rapidly as the blue ejecta component fades within hours and the red lanthanide component dominates. SN 1993J evolves much more slowly in colour. The colour rate threshold is set at the midpoint between the two template values at the observed epochs.

This tracks how colour changes over the first week for both objects. The g-r colour is the g magnitude minus the r magnitude, and a larger value means redder. It computes the change from day zero to day one, prints it for each object, and plots the colour against time. Kilonovae redden quickly, which is the signal the later classifier exploits.

In [ ]:
# g-r colour evolution from corrected splines
t_check = np.linspace(0.0, 7.5, 200)
colour_sn = spline_sn_g(t_check) - spline_sn(t_check)
colour_kn = spline_kn_g(t_check) - spline_kn(t_check)

gr_sn_day0 = (spline_sn_g(np.array([0.0])) - spline_sn(np.array([0.0])))[0]
gr_sn_day1 = (spline_sn_g(np.array([1.0])) - spline_sn(np.array([1.0])))[0]
delta_gr_sn = gr_sn_day1 - gr_sn_day0

gr_kn_day0 = (spline_kn_g(np.array([0.0])) - spline_kn(np.array([0.0])))[0]
gr_kn_day1 = (spline_kn_g(np.array([1.0])) - spline_kn(np.array([1.0])))[0]
delta_gr_kn = gr_kn_day1 - gr_kn_day0

print(f"Supernova delta(g-r) over day 1: {delta_gr_sn:+.2f} mag")
print(f"Kilonova delta(g-r) over day 1: {delta_gr_kn:+.2f} mag")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_check, colour_sn, color="C0", linewidth=2,
        label=f"Supernova ((g-r) day 0 → 1: {delta_gr_sn:+.2f} mag)")
ax.plot(t_check, colour_kn, color="C1", linewidth=2,\
        label=f"Kilonova ((g-r) day 0 → 1: {delta_gr_kn:+.2f} mag)")
ax.axhline(0, color="grey", linewidth=0.6, linestyle=":")
ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("g - r colour (mag)")
ax.set_title("g - r colour evolution")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\GminusREvo.png', dpi=300, bbox_inches='tight')
plt.show()

## Colour Rate Function

The colour rate function is defined here. It computes the rate of change of g-r colour between the first two epochs where both bands are detected, expressed in magnitudes per day. A positive colour rate indicates that the source is getting redder over time, which is the physical signature of a kilonova transitioning from its blue to red ejecta component.

This defines colour_rate, the second classifier feature. Given the r-band and g-band magnitudes at the observing epochs, it measures how fast the g-r colour changes per day between the first two epochs seen in both filters. It returns that rate, or a missing value if fewer than two shared detections exist. It is defined here early so later cells do not fail if run out of order.

In [ ]:
# colour_rate is defined in the 2D classifier simulation cell below.
# It is placed there to keep the function definition co-located with its use.
# Defined here as a placeholder to avoid NameErrors if cells are run out of order.

def colour_rate(m_obs_r, m_obs_g):
    """Rate of change of g-r colour between first two detected epochs (mag/day).
    Positive = getting redder. Kilonova signature.
    Returns NaN if fewer than 2 epochs detected in both bands.
    """
    valid = (~np.isnan(m_obs_r)) & (~np.isnan(m_obs_g))
    if valid.sum() < 2:
        return np.nan
    idx = np.where(valid)[0]
    t0, t1 = t_obs[idx[0]], t_obs[idx[1]]
    gr0 = m_obs_g[idx[0]] - m_obs_r[idx[0]]
    gr1 = m_obs_g[idx[1]] - m_obs_r[idx[1]]
    return (gr1 - gr0) / (t1 - t0)


## Template Colour Rate Verification

The g-r colour at both observed epochs is printed for each template, along with the resulting colour rate. This confirms that the two templates have substantially different colour rates and that the physical separation expected from the light curve analysis is present in the numerical values used by the classifier.

This is a quick numerical check on the templates. It prints the g-r colour of the supernova and kilonova at day zero and day one and the day to day change. It confirms the kilonova reddens far faster than the supernova.

In [ ]:
t_eval = np.array([0.0, 1.0])
gr0_sn = spline_sn_g(t_eval)[0] - spline_sn(t_eval)[0]
gr1_sn = spline_sn_g(t_eval)[1] - spline_sn(t_eval)[1]
gr0_kn = spline_kn_g(t_eval)[0] - spline_kn(t_eval)[0]
gr1_kn = spline_kn_g(t_eval)[1] - spline_kn(t_eval)[1]

print(f"SN g-r at t=0: {gr0_sn:+.3f}  at t=1: {gr1_sn:+.3f}  rate: {gr1_sn - gr0_sn:+.3f} mag/day")
print(f"KN g-r at t=0: {gr0_kn:+.3f}  at t=1: {gr1_kn:+.3f}  rate: {gr1_kn - gr0_kn:+.3f} mag/day")

## g-band Light Curve Generator

This function mirrors the r-band generator but evaluates the g-band splines instead. It applies the same distance modulus, magnitude-dependent noise model, and limiting magnitude mask. Having separate generators for each band allows independent noise realisations to be drawn, reflecting the fact that g and r observations are not perfectly correlated.

This defines generate_observed_lightcurve_g, the g-band twin of the earlier r-band generator. Given an event type, explosion time, and distance, it returns a noisy g-band light curve with the same distance and noise treatment. Having both bands lets the simulation produce a colour for every event.

In [ ]:
#  g-band light curve generator 
def generate_observed_lightcurve_g(event_type, t0, d_mpc):
    t_rest = t_obs - t0
    if event_type == "SN":
        M = spline_sn_g(t_rest)
    else:
        M = spline_kn_g(t_rest)
    M = np.array(M)
    M[t_rest < 0] = np.nan
    mu = 5 * np.log10(d_mpc * 1e6) - 5
    m_true = M + mu
    sigma = np.where(m_true <= 20, 0.1, np.where(m_true >= 21, 0.2, 0.1 + 0.1 * (m_true - 20)))
    m_obs = m_true + np.random.normal(0, sigma)
    return m_obs, sigma

## 2D Classifier Simulation

The two-feature classifier simulation is run across 100 seeds. For each event, both r-band and g-band light curves are generated and the fade rate and colour rate are computed. Events must pass both a fade rate threshold of 0.35 mag/day and a colour rate threshold of 0.10 mag/day to be classified as kilonova candidates.

This reruns the one hundred seed simulation while recording two features per event, the fade rate and the colour rate. An event is flagged as a kilonova only if both features clear their thresholds, KN_FADE_THRESHOLD = 0.35 and KN_COLOUR_RATE_THRESHOLD = 0.10 magnitudes per day. The colour threshold comes from a sensitivity study aimed at keeping most real kilonovae while cutting supernova false positives. It prints the detection numbers and a summary of the feature distributions, and stores everything in events_2d.

In [ ]:
# --- 2D classifier: fade rate + g-r colour rate ---

def colour_rate(m_obs_r, m_obs_g):
    """Rate of change of g-r colour between first two detected epochs (mag/day).
    Positive = getting redder. Kilonova signature.
    Returns NaN if fewer than 2 epochs detected in both bands.
    """
    valid = (~np.isnan(m_obs_r)) & (~np.isnan(m_obs_g))
    if valid.sum() < 2:
        return np.nan
    idx = np.where(valid)[0]
    t0, t1 = t_obs[idx[0]], t_obs[idx[1]]
    gr0 = m_obs_g[idx[0]] - m_obs_r[idx[0]]
    gr1 = m_obs_g[idx[1]] - m_obs_r[idx[1]]
    return (gr1 - gr0) / (t1 - t0)


records = []

for seed in range(1, 101):
    np.random.seed(seed)

    # --- KN ---
    kn_d = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)
    kn_r, _ = generate_observed_lightcurve("KN", kn_t0, kn_d)
    kn_g, _ = generate_observed_lightcurve_g("KN", kn_t0, kn_d)
    kn_r[kn_r > m_lim] = np.nan
    kn_g[kn_g > m_lim] = np.nan

    records.append({
        "label":       1,
        "fade_rate":   fade_rate(kn_r),
        "colour_rate": colour_rate(kn_r, kn_g),
        "distance":    kn_d,
    })

    # --- 50 SNe ---
    for _ in range(50):
        d  = sample_distance()
        t0 = np.random.uniform(-1, 0)
        sn_r, _ = generate_observed_lightcurve("SN", t0, d)
        sn_g, _ = generate_observed_lightcurve_g("SN", t0, d)
        sn_r[sn_r > m_lim] = np.nan
        sn_g[sn_g > m_lim] = np.nan

        records.append({
            "label":       0,
            "fade_rate":   fade_rate(sn_r),
            "colour_rate": colour_rate(sn_r, sn_g),
            "distance":    d,
        })

events_2d = pd.DataFrame(records)

# Thresholds — consistent with 1D classifier throughout
# KN_FADE_THRESHOLD: same 0.35 mag/day used in the 1D classifier
# KN_COLOUR_RATE_THRESHOLD: set at 0.10 mag/day based on threshold sensitivity
# analysis (see threshold table cell). This value was chosen to maintain
# completeness above 86% while reducing SN false positives by ~70%.
KN_FADE_THRESHOLD        = 0.35    # mag/day — consistent throughout notebook
KN_COLOUR_RATE_THRESHOLD = 0.10   # mag/day — from threshold sensitivity analysis

events_2d["pred_2d"] = (
    (events_2d["fade_rate"]   >= KN_FADE_THRESHOLD) &
    (events_2d["colour_rate"] >= KN_COLOUR_RATE_THRESHOLD)
)

kn_mask = events_2d["label"] == 1
sn_mask = events_2d["label"] == 0

tp = (events_2d["pred_2d"] & kn_mask).sum()
fp = (events_2d["pred_2d"] & sn_mask).sum()
fn = (~events_2d["pred_2d"] & kn_mask).sum()

print(f"Colour rate threshold : {KN_COLOUR_RATE_THRESHOLD:.3f} mag/day")
print(f"TPR : {tp / kn_mask.sum():.3f}   FPR : {fp / sn_mask.sum():.4f}")
print(f"TP={tp}  FP={fp}  FN={fn}")

# Feature distribution summary
print(events_2d.groupby("label")[["fade_rate", "colour_rate"]].describe().round(3))


## Template Evolution Plot

The fade rate and g-r colour evolution are plotted as functions of time for both templates on a single axes. Solid lines show fade rate and dashed lines show colour. The vertical dotted lines mark the two observed epochs at t=0 and t=1 day. This illustrates how both features evolve over the observing window.

This plots how the fade rate and g-r colour evolve for both templates over the first week, with the two observing epochs marked. It shows why measuring at day 0 and day 1 separates the two classes. The figure is saved to disk.

In [ ]:
# --- Fade rate and g-r colour evolution for both templates ---

t_grid = np.linspace(0, 7.5, 200)
dt = t_grid[1] - t_grid[0]

# Fade rate: instantaneous gradient of r-band spline
fr_sn = np.gradient(spline_sn(t_grid), dt)
fr_kn = np.gradient(spline_kn(t_grid), dt)

# g-r colour
gr_sn = spline_sn_g(t_grid) - spline_sn(t_grid)
gr_kn = spline_kn_g(t_grid) - spline_kn(t_grid)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(t_grid, fr_sn, color="C0", linewidth=2, linestyle="-",  label="SN 1993J — fade rate (mag/day)")
ax.plot(t_grid, gr_sn, color="C0", linewidth=2, linestyle="--", label="SN 1993J — g-r colour (mag)")
ax.plot(t_grid, fr_kn, color="C1", linewidth=2, linestyle="-",  label="AT2017gfo — fade rate (mag/day)")
ax.plot(t_grid, gr_kn, color="C1", linewidth=2, linestyle="--", label="AT2017gfo — g-r colour (mag)")

# Mark the two observed epochs
for t_mark in [0.0, 1.0]:
    ax.axvline(t_mark, color="grey", linewidth=0.8, linestyle=":")

ax.axvline(0.0, color="grey", linewidth=0.8, linestyle=":", label="Observed epochs (t=0, t=1)")
ax.axhline(0, color="black", linewidth=0.5)

ax.set_xlabel("Days since explosion / merger")
ax.set_ylabel("mag/day  or  mag")
ax.set_title("Fade rate and g-r colour evolution — SN 1993J vs AT2017gfo")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\FDandGminusR.png', dpi=300, bbox_inches='tight')
plt.show()

## ROC Curve and Classification Summary: Fade Rate Only vs Hard AND Classifier

This cell compares the 1D fade rate classifier against the hard AND two-feature classifier using a ROC curve and a side-by-side bar chart.

### What the ROC Curve Is

The ROC curve shows classifier performance across every possible threshold simultaneously. For each threshold value it computes two numbers — the true positive rate (fraction of real KN correctly flagged) and the false positive rate (fraction of SN incorrectly flagged). Every possible threshold produces one point on the plot. Connecting all those points gives the full curve. A perfect classifier hugs the top left corner with AUC = 1.0. A random classifier produces a diagonal line with AUC = 0.5. The further the curve sits above the diagonal, the better the classifier.

### How the AND Score is Computed

The AND score is constructed as:
```python
and_score = np.minimum(
    fade_rate    / KN_FADE_THRESHOLD,
    colour_rate  / KN_COLOUR_RATE_THRESHOLD
)
```

Dividing each feature by its threshold normalises it so that 1.0 means exactly at the threshold, above 1.0 means passed, and below 1.0 means failed. Taking the minimum of the two means the combined score is entirely controlled by whichever feature is weakest. A fade rate of 3.0 times threshold cannot rescue a colour rate of 0.9 times threshold — the minimum is 0.9 and the event fails. Both features must independently exceed their thresholds for the event to score above 1.0. This is the AND logic encoded as a continuous number so it can be fed into the ROC curve function.

### What the Bar Chart Shows

The bar chart compares the raw classification counts for both classifiers. The fade rate only classifier flags everything above 0.35 mag/day regardless of colour. The hard AND classifier additionally requires the colour rate to exceed 0.10 mag/day. The result is fewer SN false positives because supernovae rarely pass both conditions simultaneously, but also fewer KN detected because some kilonovae caught late in their evolution fail the colour rate condition despite having a strong fade rate. This rigidity — where one weak feature kills an otherwise strong event — is the fundamental limitation of the hard AND approach and directly motivates the soft score classifier introduced in the following cell.

This compares the single feature classifier against a stricter one that requires both features. Using scikit-learn's roc_curve, it builds a ROC curve for fade rate alone and for a combined AND score, finds each best operating point by Youden's J, the point sitting furthest above the random line, and reports the areas. It then draws the ROC curves and a bar chart of outcomes, and prints precision, recall, and F1, three standard accuracy measures.

In [ ]:
from sklearn.metrics import roc_curve, auc

# ── Data preparation ──────────────────────────────────────────────────────────
valid  = events_2d.dropna(subset=["fade_rate", "colour_rate"]).copy()
labels = valid["label"].values
kn     = valid[valid["label"] == 1]
sn     = valid[valid["label"] == 0]

# ── ROC curves ────────────────────────────────────────────────────────────────
# 1D: fade rate alone
fpr_1d, tpr_1d, thresh_1d = roc_curve(labels, valid["fade_rate"].values)
auc_1d = auc(fpr_1d, tpr_1d)

# AND score: minimum of both threshold-normalised features
# An event only scores highly if both features pass independently######################################################
and_score = np.minimum(
    valid["fade_rate"]    / KN_FADE_THRESHOLD,
    valid["colour_rate"] / KN_COLOUR_RATE_THRESHOLD
)
#############################################################################################################################
fpr_and, tpr_and, thresh_and = roc_curve(labels, and_score.values)
auc_and = auc(fpr_and, tpr_and)

# Optimal points via Youden's J
opt_1d  = np.argmax(tpr_1d  - fpr_1d)
opt_and = np.argmax(tpr_and - fpr_and)

print(f"1D   AUC={auc_1d:.3f}  optimal TPR={tpr_1d[opt_1d]:.3f}  FPR={fpr_1d[opt_1d]:.3f}")
print(f"AND  AUC={auc_and:.3f}  optimal TPR={tpr_and[opt_and]:.3f}  FPR={fpr_and[opt_and]:.3f}")

# ── ROC plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_1d,  tpr_1d,  linewidth=2, color="C0",
        label=f"Fade rate only  (AUC = {auc_1d:.3f})")
ax.plot(fpr_and, tpr_and, linewidth=2, color="C1",
        label=f"Hard AND        (AUC = {auc_and:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
ax.scatter([fpr_1d[opt_1d]],  [tpr_1d[opt_1d]],  s=100, color="C0", zorder=5,
           label=f"1D optimal  (TPR={tpr_1d[opt_1d]:.2f}, FPR={fpr_1d[opt_1d]:.2f})")
ax.scatter([fpr_and[opt_and]], [tpr_and[opt_and]], s=100, color="C1", zorder=5,
           label=f"AND optimal (TPR={tpr_and[opt_and]:.2f}, FPR={fpr_and[opt_and]:.2f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC: fade rate only vs hard AND classifier")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\ROCFDvsHardAND.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Bar chart ─────────────────────────────────────────────────────────────────
pred_1d_kn  = kn["fade_rate"] >= KN_FADE_THRESHOLD
pred_1d_sn  = sn["fade_rate"] >= KN_FADE_THRESHOLD
pred_and_kn = (kn["fade_rate"] >= KN_FADE_THRESHOLD) & (kn["colour_rate"] >= KN_COLOUR_RATE_THRESHOLD)
pred_and_sn = (sn["fade_rate"] >= KN_FADE_THRESHOLD) & (sn["colour_rate"] >= KN_COLOUR_RATE_THRESHOLD)

categories  = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]
counts_1d   = [pred_1d_kn.sum(),  (~pred_1d_kn).sum(),  pred_1d_sn.sum(),  (~pred_1d_sn).sum()]
counts_and  = [pred_and_kn.sum(), (~pred_and_kn).sum(), pred_and_sn.sum(), (~pred_and_sn).sum()]

x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
b1 = ax.bar(x - width/2, counts_1d,  width, label="Fade rate only",     color="C0", alpha=0.8)
b2 = ax.bar(x + width/2, counts_and, width, label="Hard AND thresholds", color="C1", alpha=0.8)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Classifier comparison: fade rate only vs hard AND\n(100 seeds, 1 KN + 50 SN per seed)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarFDvsHardAND.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Metrics ───────────────────────────────────────────────────────────────────
for name, tp, fn, fp in [
    ("Fade rate only ", counts_1d[0],  counts_1d[1],  counts_1d[2]),
    ("Hard AND       ", counts_and[0], counts_and[1], counts_and[2]),
]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{name}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

## ROC Curves, Soft Score Classifier, and Three-Way Performance Summary

This cell constructs the ROC curves, applies the soft score classifier, and produces the three-way summary bar chart comparing all three classifiers.

### What is the Soft Score and How is it Built

The soft score is a single number that combines the fade rate and the colour rate into one overall measure of how kilonova-like an event looks. Rather than asking two separate yes/no questions as the hard AND classifier does, it asks one question — is the combined signal strong enough?

The score is built using z-scoring. For any feature $x$, the z-score is:

$$z = \frac{x - \bar{x}}{\sigma_x}$$

where $\bar{x}$ is the mean of that feature across all events and $\sigma_x$ is its standard deviation. In plain English: subtract the average and divide by the spread.



This introduces the soft score classifier. Rather than hard cutoffs, it converts each feature to a z-score, meaning how many standard deviations above average it is, then adds the two; the z-score puts the differently scaled features on one common footing. It finds the best threshold on this score, applies it, and compares against fade rate alone with ROC curves and a bar chart. Precision, recall, and F1 are printed for each method.

In [ ]:
from sklearn.metrics import roc_curve, auc

valid = events_2d.dropna(subset=["fade_rate", "colour_rate"])
labels = valid["label"].values

# 1D score
fpr_1d, tpr_1d, thresholds_1d = roc_curve(labels, valid["fade_rate"].values)
auc_1d = auc(fpr_1d, tpr_1d)

# 2D soft score: z-score both features and sum##########################################################
fr_norm = (valid["fade_rate"]    - valid["fade_rate"].mean())    / valid["fade_rate"].std()
cs_norm = (valid["colour_rate"] - valid["colour_rate"].mean()) / valid["colour_rate"].std()
score_2d = fr_norm + cs_norm
###############################################################################################################
fpr_2d, tpr_2d, thresholds_2d = roc_curve(labels, score_2d.values)
auc_2d = auc(fpr_2d, tpr_2d)

# --- Optimal points via Youden's J (max TPR - FPR) ---
opt_idx_1d = np.argmax(tpr_1d - fpr_1d)
opt_fpr_1d  = fpr_1d[opt_idx_1d]
opt_tpr_1d  = tpr_1d[opt_idx_1d]
opt_thresh_1d = thresholds_1d[opt_idx_1d]

opt_idx_2d = np.argmax(tpr_2d - fpr_2d)
opt_fpr_2d  = fpr_2d[opt_idx_2d]
opt_tpr_2d  = tpr_2d[opt_idx_2d]
opt_thresh_2d = thresholds_2d[opt_idx_2d]

print(f"1D optimal threshold : {opt_thresh_1d:.3f} mag/day "
      f"(TPR={opt_tpr_1d:.3f}, FPR={opt_fpr_1d:.3f})")
print(f"2D optimal soft score: {opt_thresh_2d:.3f} "
      f"(TPR={opt_tpr_2d:.3f}, FPR={opt_fpr_2d:.3f})")

# --- Apply soft score classifier ---
events_2d.loc[valid.index, "soft_score"] = score_2d.values
events_2d["pred_soft"] = events_2d["soft_score"] >= opt_thresh_2d

kn_mask = events_2d["label"] == 1
sn_mask = events_2d["label"] == 0

tp_soft = (events_2d["pred_soft"] & kn_mask).sum()
fp_soft = (events_2d["pred_soft"] & sn_mask).sum()
fn_soft = (~events_2d["pred_soft"] & kn_mask).sum()

precision_soft = tp_soft / (tp_soft + fp_soft) if (tp_soft + fp_soft) > 0 else 0
recall_soft    = tp_soft / (tp_soft + fn_soft)  if (tp_soft + fn_soft) > 0 else 0
f1_soft        = (2 * precision_soft * recall_soft / (precision_soft + recall_soft)
                  if (precision_soft + recall_soft) > 0 else 0)

print(f"\nSoft score classifier:")
print(f"  TP={tp_soft}  FP={fp_soft}  FN={fn_soft}")
print(f"  Precision={precision_soft:.3f}  Recall={recall_soft:.3f}  F1={f1_soft:.3f}")

# --- ROC plot with optimal points marked ---
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_1d, tpr_1d, linewidth=2, color="C0",
        label=f"Fade rate only (AUC = {auc_1d:.3f})")
ax.plot(fpr_2d, tpr_2d, linewidth=2, color="C1",
        label=f"Fade rate + colour (AUC = {auc_2d:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")

ax.scatter([opt_fpr_1d], [opt_tpr_1d], s=100, color="C0", zorder=5,
           label=f"1D optimal (TPR={opt_tpr_1d:.2f}, FPR={opt_fpr_1d:.2f})")
ax.scatter([opt_fpr_2d], [opt_tpr_2d], s=100, color="C1", zorder=5,
           label=f"2D optimal (TPR={opt_tpr_2d:.2f}, FPR={opt_fpr_2d:.2f})")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC: fade vs fade + colour classifier")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\ROCFDvFDandCoulour.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Three-way summary bar chart ---
pred_1d_kn = kn_mask & (events_2d["fade_rate"] >= KN_FADE_THRESHOLD)
pred_1d_sn = sn_mask & (events_2d["fade_rate"] >= KN_FADE_THRESHOLD)

pred_hard_kn = kn_mask & events_2d["pred_2d"]
pred_hard_sn = sn_mask & events_2d["pred_2d"]

pred_soft_kn = kn_mask & events_2d["pred_soft"]
pred_soft_sn = sn_mask & events_2d["pred_soft"]

categories = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]

counts_1d   = [pred_1d_kn.sum(),   (~pred_1d_kn & kn_mask).sum(),   pred_1d_sn.sum(),   (~pred_1d_sn & sn_mask).sum()]
counts_hard = [pred_hard_kn.sum(), (~pred_hard_kn & kn_mask).sum(), pred_hard_sn.sum(), (~pred_hard_sn & sn_mask).sum()]
counts_soft = [pred_soft_kn.sum(), (~pred_soft_kn & kn_mask).sum(), pred_soft_sn.sum(), (~pred_soft_sn & sn_mask).sum()]

x     = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, counts_1d,   width, label="Fade rate only",         color="C0", alpha=0.8)
b2 = ax.bar(x,         counts_hard, width, label="Hard AND thresholds",     color="C1", alpha=0.8)
b3 = ax.bar(x + width, counts_soft, width, label="Soft score (z-sum)",      color="C2", alpha=0.8)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Classifier comparison: fade rate / hard AND / soft score\n(100 seeds — note: soft score is in-sample)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarFDvFDandCoulour.png', dpi=300, bbox_inches='tight')
plt.show()

for name, tp, fn, fp in [
    ("Fade rate only ", counts_1d[0],   counts_1d[1],   counts_1d[2]),
    ("Hard AND       ", counts_hard[0], counts_hard[1], counts_hard[2]),
    ("Soft score     ", counts_soft[0], counts_soft[1], counts_soft[2]),
]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{name}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

## 200-Seed Train/Test Split Simulation

This cell runs the full simulation across 200 independent seeds and splits
them into a training set (seeds 1-100) and a test set (seeds 101-200).
The soft score normalisation parameters and optimal threshold are computed
exclusively from the training set. They are then applied unchanged to the
test set, which the classifier has never seen. The test set performance
numbers are the honest, unbiased estimates reported in the thesis.


This is the first honest evaluation. It simulates two hundred seeds, using seeds 1 to 100 as a training set to learn the score's averages, spreads, and best threshold, and holding seeds 101 to 200 as an untouched test set. Nothing from the test set influences the setup. It prints the training AUC and threshold, then the precision, recall, F1, and AUC measured on the unseen test data.

In [ ]:
# --- 200-seed simulation with train/test split ---
# Seeds 1-100: training set (find soft score normalisation and threshold)
# Seeds 101-200: test set (honest evaluation on unseen data)
# Nothing from the test set influences the classifier setup.

records_tt = []

for seed in range(1, 201):
    np.random.seed(seed)

    kn_d  = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)
    kn_r, _ = generate_observed_lightcurve("KN", kn_t0, kn_d)
    kn_g, _ = generate_observed_lightcurve_g("KN", kn_t0, kn_d)
    kn_r[kn_r > m_lim] = np.nan
    kn_g[kn_g > m_lim] = np.nan

    records_tt.append({
        "seed":        seed,
        "label":       1,
        "fade_rate":   fade_rate(kn_r),
        "colour_rate": colour_rate(kn_r, kn_g),
        "distance":    kn_d,
    })

    for _ in range(50):
        d  = sample_distance()
        t0 = np.random.uniform(-1, 0)
        sn_r, _ = generate_observed_lightcurve("SN", t0, d)
        sn_g, _ = generate_observed_lightcurve_g("SN", t0, d)
        sn_r[sn_r > m_lim] = np.nan
        sn_g[sn_g > m_lim] = np.nan

        records_tt.append({
            "seed":        seed,
            "label":       0,
            "fade_rate":   fade_rate(sn_r),
            "colour_rate": colour_rate(sn_r, sn_g),
            "distance":    d,
        })

events_tt = pd.DataFrame(records_tt)

# --- Train/test split ---
train = events_tt[events_tt["seed"] <= 100].copy()
test  = events_tt[events_tt["seed"] >  100].copy()

print(f"Training set: {len(train)} events ({(train['label']==1).sum()} KN, {(train['label']==0).sum()} SN)")
print(f"Test set:     {len(test)} events ({(test['label']==1).sum()} KN, {(test['label']==0).sum()} SN)")

# --- Compute normalisation from training set only ---
fr_mean = train["fade_rate"].mean()
fr_std  = train["fade_rate"].std()
cs_mean = train["colour_rate"].mean()
cs_std  = train["colour_rate"].std()

train_valid = train.dropna(subset=["fade_rate", "colour_rate"]).copy()
test_valid  = test.dropna(subset=["fade_rate", "colour_rate"]).copy()

train_valid["soft_score"] = (
    (train_valid["fade_rate"]    - fr_mean) / fr_std +
    (train_valid["colour_rate"] - cs_mean) / cs_std
)

# --- Find optimal threshold on training set via Youden's J ---
train_labels = train_valid["label"].values
fpr_tr, tpr_tr, thresholds_tr = roc_curve(train_labels, train_valid["soft_score"].values)
auc_train = auc(fpr_tr, tpr_tr)

opt_idx    = np.argmax(tpr_tr - fpr_tr)
opt_thresh = thresholds_tr[opt_idx]

print(f"\nTraining AUC: {auc_train:.3f}")
print(f"Optimal threshold (Youden's J, training set): {opt_thresh:.4f}")
print(f"Training TPR={tpr_tr[opt_idx]:.3f}, FPR={fpr_tr[opt_idx]:.3f}")

# --- Apply to test set using training parameters only ---
test_valid["soft_score"] = (
    (test_valid["fade_rate"]    - fr_mean) / fr_std +
    (test_valid["colour_rate"] - cs_mean) / cs_std
)

test_valid["pred_soft"] = test_valid["soft_score"] >= opt_thresh

tp_te = test_valid["pred_soft"][test_valid["label"] == 1].sum()
fp_te = test_valid["pred_soft"][test_valid["label"] == 0].sum()
fn_te = (~test_valid["pred_soft"])[test_valid["label"] == 1].sum()

precision_te = tp_te / (tp_te + fp_te) if (tp_te + fp_te) > 0 else 0
recall_te    = tp_te / (tp_te + fn_te) if (tp_te + fn_te) > 0 else 0
f1_te        = (2 * precision_te * recall_te / (precision_te + recall_te)
                if (precision_te + recall_te) > 0 else 0)

print(f"\nTest set performance (honest — unseen seeds 101-200):")
print(f"  TP={tp_te}  FP={fp_te}  FN={fn_te}")
print(f"  Precision={precision_te:.3f}  Recall={recall_te:.3f}  F1={f1_te:.3f}")

fpr_te_roc, tpr_te_roc, _ = roc_curve(test_valid["label"].values, test_valid["soft_score"].values)
auc_test = auc(fpr_te_roc, tpr_te_roc)
print(f"  Test AUC: {auc_test:.3f}")


This prints the five numbers that fully define the trained soft score classifier: the mean and standard deviation of each feature and the chosen threshold. It also spells out the scoring formula in words. These values are all you need to score any new event.

In [ ]:
# --- Soft score normalisation parameters from training set ---

print("Soft score normalisation parameters (computed from training seeds 1-100)")
print("=" * 60)
print(f"Fade rate   mean : {fr_mean:.4f} mag/day")
print(f"Fade rate   std  : {fr_std:.4f} mag/day")
print(f"Colour rate mean : {cs_mean:.4f} mag/day")
print(f"Colour rate std  : {cs_std:.4f} mag/day")
print()
print(f"Optimal soft score threshold : {opt_thresh:.4f}")
print()
print("These five values are all that is needed to deploy the classifier.")
print("For any new event, the soft score is computed as:")
print()
print("  soft_score = (fade_rate - fr_mean) / fr_std")
print("             + (colour_rate - cs_mean) / cs_std")
print()
print("  If soft_score >= threshold → kilonova candidate")

## Train/Test Split Evaluation of the Soft Score Classifier

This is the most important validation cell in the notebook. Everything before this point computed the soft score parameters and threshold using the same data the classifier was then tested on, which produces optimistically biased results. This cell fixes that by using a proper train/test split — the classifier is built entirely on one set of data and evaluated on a completely separate set it has never seen.

### Why This Matters

When you find the optimal threshold by looking at all your data and then test performance on that same data, the classifier has effectively seen the exam paper before sitting the exam. It knows exactly where to draw the line because it was tuned on those specific noise realisations. A new survey with different events at different distances will not reproduce those exact noise patterns, so the in-sample performance numbers are too optimistic. The train/test split gives you the honest answer — how well does the classifier actually generalise to new data?

### How the Split Works

The simulation is extended from 100 to 200 seeds. Seeds 1 to 100 form the training set. Seeds 101 to 200 form the test set. Each seed is a completely independent survey realisation with a different kilonova and 50 different supernovae at different random distances and explosion times. The two sets are the same size — 5100 events each, 100 KN and 5000 SN — so the comparison is fair.

The key rule is that **nothing from the test set is allowed to influence the classifier setup**. The normalisation parameters — the mean and standard deviation of the fade rate and colour rate — are computed exclusively from the training set. The optimal threshold is found exclusively on the training set using Youden's J statistic. These values are then frozen and applied unchanged to the test set.

### The Z-Scoring Step

The soft score for any event is computed as:

$$S = \frac{f - \bar{f}_{\text{train}}}{\sigma_{f,\text{train}}} + \frac{c - \bar{c}_{\text{train}}}{\sigma_{c,\text{train}}}$$

where $\bar{f}_{\text{train}}$ and $\sigma_{f,\text{train}}$ are the mean and standard deviation of the fade rate computed from the **training set only**, and similarly for the colour rate. When this formula is applied to a test set event, it uses the training set statistics — not the test set statistics. This is critical. If you recomputed the means and standard deviations on the test set you would be leaking test information into the normalisation step. In code:
```python
# Compute from training set only
fr_mean = train["fade_rate"].mean()
fr_std  = train["fade_rate"].std()
cs_mean = train["colour_rate"].mean()
cs_std  = train["colour_rate"].std()

# Apply the SAME parameters to the test set
test_valid["soft_score"] = (
    (test_valid["fade_rate"]    - fr_mean) / fr_std +
    (test_valid["colour_rate"] - cs_mean) / cs_std
)
```

### What the Results Show

The training ROC and test ROC are plotted together. If the two curves sit close to each other the classifier has generalised well — the performance on unseen data matches what was observed during training. A large gap between the curves would indicate overfitting, meaning the classifier had tuned itself too closely to the specific noise patterns in the training seeds.

The bar chart compares the classification outcomes side by side for the training and test sets using the same threshold throughout. True positives, false negatives, false positives, and true negatives are shown for both. Precision, recall, and F1 are printed for both sets. The test set numbers are the ones to report in the thesis — they are the honest, unbiased estimate of how the classifier would perform on a real LSST survey it has never seen before.

This checks whether the classifier generalises by plotting the training and test ROC curves together. It also applies the training threshold to both sets, draws a bar chart of outcomes, and prints precision, recall, and F1 for each. Similar training and test performance means the classifier is not overfitting.

In [ ]:
# --- ROC and bar chart: training set vs test set ---

from sklearn.metrics import roc_curve, auc

# --- ROC curves ---
fpr_tr, tpr_tr, _ = roc_curve(train_valid["label"].values, train_valid["soft_score"].values)
fpr_te, tpr_te, _ = roc_curve(test_valid["label"].values,  test_valid["soft_score"].values)
auc_tr = auc(fpr_tr, tpr_tr)
auc_te = auc(fpr_te, tpr_te)

# Optimal points
opt_tr = np.argmax(tpr_tr - fpr_tr)
opt_te = np.argmax(tpr_te - fpr_te)

print(f"Training AUC : {auc_tr:.3f}  optimal TPR={tpr_tr[opt_tr]:.3f}  FPR={fpr_tr[opt_tr]:.3f}")
print(f"Test AUC     : {auc_te:.3f}  optimal TPR={tpr_te[opt_te]:.3f}  FPR={fpr_te[opt_te]:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_tr, tpr_tr, linewidth=2, color="C0",
        label=f"Training set — seeds 1–100   (AUC = {auc_tr:.3f})")
ax.plot(fpr_te, tpr_te, linewidth=2, color="C1", linestyle="--",
        label=f"Test set — seeds 101–200  (AUC = {auc_te:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
ax.scatter([fpr_tr[opt_tr]], [tpr_tr[opt_tr]], s=100, color="C0", zorder=5,
           label=f"Training optimal (TPR={tpr_tr[opt_tr]:.2f}, FPR={fpr_tr[opt_tr]:.2f})")
ax.scatter([fpr_te[opt_te]], [tpr_te[opt_te]], s=100, color="C1", zorder=5,
           label=f"Test optimal     (TPR={tpr_te[opt_te]:.2f}, FPR={fpr_te[opt_te]:.2f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Soft score ROC — training set vs test set")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\ROCTraining.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Bar chart: training vs test ---
# Apply threshold (found on training set) to both sets
train_valid["pred_soft"] = train_valid["soft_score"] >= opt_thresh
test_valid["pred_soft"]  = test_valid["soft_score"]  >= opt_thresh

def get_counts(df):
    kn = df[df["label"] == 1]
    sn = df[df["label"] == 0]
    tp = kn["pred_soft"].sum()
    fn = (~kn["pred_soft"]).sum()
    fp = sn["pred_soft"].sum()
    tn = (~sn["pred_soft"]).sum()
    return [tp, fn, fp, tn]

counts_train = get_counts(train_valid)
counts_test  = get_counts(test_valid)

categories = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]
x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
b1 = ax.bar(x - width/2, counts_train, width, label="Training (seeds 1–100)",  color="C0", alpha=0.8)
b2 = ax.bar(x + width/2, counts_test,  width, label="Test (seeds 101–200)",     color="C1", alpha=0.8)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Soft score classifier — training vs test set\n(100 seeds each, 1 KN + 50 SN per seed)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarTraining.png', dpi=300, bbox_inches='tight')
plt.show()

# Print metrics for both
for name, counts in [("Training", counts_train), ("Test    ", counts_test)]:
    tp, fn, fp, tn = counts
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"{name}  TP={tp}  FP={fp}  FN={fn}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

## Deployment: Pre-Trained Soft Score on Unseen Seeds

This cell is the final validation of the classifier. It applies the soft score with completely fixed pre-trained parameters to a third batch of survey realisations that were never involved in any part of the classifier development. No learning takes place here — the four normalisation parameters and the threshold are all imported directly from the training set and applied without modification.

The purpose is to simulate real-world deployment. In practice, a classifier trained on simulated data would be applied to new LSST alerts as they arrive, with no opportunity to retune parameters. This cell recreates that scenario exactly. The deployment parameters are:
```python
DEPLOY_FR_MEAN   = fr_mean      # mean fade rate from training seeds 1-100
DEPLOY_FR_STD    = fr_std       # std fade rate from training seeds 1-100
DEPLOY_CS_MEAN   = cs_mean      # mean colour rate from training seeds 1-100
DEPLOY_CS_STD    = cs_std       # std colour rate from training seeds 1-100
DEPLOY_THRESHOLD = 1.4285       # optimal threshold from Youden's J on training set
```

For each new event the soft score is computed as:

$$S = \frac{f - \bar{f}_{\text{train}}}{\sigma_{f,\text{train}}} + \frac{c - \bar{c}_{\text{train}}}{\sigma_{c,\text{train}}}$$

If $S \geq 1.4285$ the event is flagged as a kilonova candidate. This is a single arithmetic operation applied independently to each new alert with no reference to any other event in the deployment set.

The ROC curve plots the deployment set alongside the training and test ROC curves from the previous cell. If all three curves sit close together the classifier has genuinely learned a stable physical signal rather than memorising noise patterns from specific seeds. The bar chart shows the raw classification counts for the deployment set alone. Precision, recall, and F1 are printed for direct comparison against the training and test set results reported earlier.

This is a pure deployment test with no learning at all. The five soft score constants are hard coded from the training set, fade rate mean 0.3535 and standard deviation 0.2701, colour rate mean 0.1416 and standard deviation 0.3831, and threshold 1.4285, then applied to a fresh batch of seeds 301 to 400. It generates those events, scores them, and prints precision, recall, and F1. Finally it plots ROC curves for training, test, and deployment together and a bar chart of the deployment outcomes.

In [ ]:
# --- Pure deployment cell: apply pre-trained soft score to new seeds 201-300 ---
# No learning. No threshold tuning. Parameters fixed from training set (seeds 1-100).

DEPLOY_FR_MEAN   = 0.3535   # fade rate mean (mag/day) — training seeds 1-100
DEPLOY_FR_STD    = 0.2701   # fade rate std  (mag/day) — training seeds 1-100
DEPLOY_CS_MEAN   = 0.1416   # colour rate mean (mag/day) — training seeds 1-100
DEPLOY_CS_STD    = 0.3831   # colour rate std  (mag/day) — training seeds 1-100
DEPLOY_THRESHOLD = 1.4285   # optimal threshold from Youden's J — training seeds 1-100

print("Deployment parameters (fixed from training set):")
print(f"  Fade rate   mean : {DEPLOY_FR_MEAN:.4f} mag/day")
print(f"  Fade rate   std  : {DEPLOY_FR_STD:.4f} mag/day")
print(f"  Colour rate mean : {DEPLOY_CS_MEAN:.4f} mag/day")
print(f"  Colour rate std  : {DEPLOY_CS_STD:.4f} mag/day")
print(f"  Threshold        : {DEPLOY_THRESHOLD:.4f}")
print()

# --- Generate new events: seeds 201-300 ---
deploy_records = []

for seed in range(301, 401):
    np.random.seed(seed)

    kn_d  = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)
    kn_r, _ = generate_observed_lightcurve("KN", kn_t0, kn_d)
    kn_g, _ = generate_observed_lightcurve_g("KN", kn_t0, kn_d)
    kn_r[kn_r > m_lim] = np.nan
    kn_g[kn_g > m_lim] = np.nan

    deploy_records.append({
        "seed":        seed,
        "label":       1,
        "fade_rate":   fade_rate(kn_r),
        "colour_rate": colour_rate(kn_r, kn_g),
        "distance":    kn_d,
    })

    for _ in range(50):
        d  = sample_distance()
        t0 = np.random.uniform(-1, 0)
        sn_r, _ = generate_observed_lightcurve("SN", t0, d)
        sn_g, _ = generate_observed_lightcurve_g("SN", t0, d)
        sn_r[sn_r > m_lim] = np.nan
        sn_g[sn_g > m_lim] = np.nan

        deploy_records.append({
            "seed":        seed,
            "label":       0,
            "fade_rate":   fade_rate(sn_r),
            "colour_rate": colour_rate(sn_r, sn_g),
            "distance":    d,
        })

deploy_df    = pd.DataFrame(deploy_records)
deploy_valid = deploy_df.dropna(subset=["fade_rate", "colour_rate"]).copy()

# --- Apply pre-trained soft score ---
deploy_valid["soft_score"] = (
    (deploy_valid["fade_rate"]    - DEPLOY_FR_MEAN) / DEPLOY_FR_STD +
    (deploy_valid["colour_rate"] - DEPLOY_CS_MEAN) / DEPLOY_CS_STD
)
deploy_valid["pred"] = deploy_valid["soft_score"] >= DEPLOY_THRESHOLD

# --- Metrics ---
kn_dep = deploy_valid[deploy_valid["label"] == 1]
sn_dep = deploy_valid[deploy_valid["label"] == 0]

tp = kn_dep["pred"].sum()
fn = (~kn_dep["pred"]).sum()
fp = sn_dep["pred"].sum()
tn = (~sn_dep["pred"]).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Deployment results (seeds 201-300, never seen before):")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

# --- ROC curve ---
from sklearn.metrics import roc_curve, auc

fpr_dep, tpr_dep, _ = roc_curve(deploy_valid["label"].values,
                                 deploy_valid["soft_score"].values)
auc_dep = auc(fpr_dep, tpr_dep)

# Mark deployment threshold on ROC
dep_tpr = recall
dep_fpr = fp / len(sn_dep)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_dep, tpr_dep, linewidth=2, color="C2",
        label=f"Deployment — seeds 201–300 (AUC = {auc_dep:.3f})")
ax.plot(fpr_tr, tpr_tr, linewidth=2, color="C0", linestyle="--",
        label=f"Training  — seeds   1–100 (AUC = {auc_train:.3f})")
ax.plot(fpr_te, tpr_te, linewidth=2, color="C1", linestyle="--",
        label=f"Test      — seeds 101–200 (AUC = {auc_test:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
ax.scatter([dep_fpr], [dep_tpr], s=100, color="C2", zorder=5,
           label=f"Deployment threshold (TPR={dep_tpr:.2f}, FPR={dep_fpr:.2f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Soft score ROC — training / test / deployment")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\ROCTrainvTestvDeployment.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Bar chart ---
categories = ["KN detected\n(TP)", "KN missed\n(FN)",
              "SN false pos\n(FP)", "SN true neg\n(TN)"]
counts_dep = [tp, fn, fp, tn]

x     = np.arange(len(categories))
width = 0.5

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(x, counts_dep, width, color="C2", alpha=0.8,
              label="Deployment — seeds 201–300")

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Deployment performance — pre-trained soft score\n"
             f"(seeds 201–300, threshold = {DEPLOY_THRESHOLD})")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarTrainvTestvDeployment.png', dpi=300, bbox_inches='tight')
plt.show()

This maps every simulated event into the two dimensional feature space of fade rate against colour rate. Supernovae and kilonovae are drawn in different colours, the two thresholds are shown as lines, and the top right kilonova region is shaded. The template positions are marked with stars for reference. The figure is saved, and the counts of each type inside the kilonova region are printed.

In [ ]:
# --- Scatter plot: fade rate vs g-r colour rate for every simulated event ---

valid = events_2d.dropna(subset=["fade_rate", "colour_rate"])
kn_ev = valid[valid["label"] == 1]
sn_ev = valid[valid["label"] == 0]

fig, ax = plt.subplots(figsize=(9, 7))

# Plot every SN point
ax.scatter(sn_ev["fade_rate"], sn_ev["colour_rate"],
           s=8, alpha=0.2, color="C0", label=f"SN ({len(sn_ev)} events)")

# Plot every KN point
ax.scatter(kn_ev["fade_rate"], kn_ev["colour_rate"],
           s=40, alpha=0.8, color="C1", label=f"KN ({len(kn_ev)} events)")

# Threshold lines
ax.axvline(KN_FADE_THRESHOLD, color="black", linestyle="--", linewidth=1,
           label=f"Fade threshold ({KN_FADE_THRESHOLD} mag/day)")
ax.axhline(KN_COLOUR_RATE_THRESHOLD, color="black", linestyle=":", linewidth=1,
           label=f"Colour rate threshold ({KN_COLOUR_RATE_THRESHOLD} mag/day)")

# Shade the classification region — top right is KN candidate zone
ax.fill_between(
    [KN_FADE_THRESHOLD, valid["fade_rate"].max() + 0.1],
    KN_COLOUR_RATE_THRESHOLD,
    valid["colour_rate"].max() + 0.1,
    alpha=0.05, color="C1", label="KN candidate region"
)

# Template positions at t=0 and t=1
t_eval = np.array([0.0, 1.0])
fr_sn_template = float(np.gradient(spline_sn(t_eval), 1.0)[1])
fr_kn_template = float(np.gradient(spline_kn(t_eval), 1.0)[1])
gr0_sn = spline_sn_g(t_eval)[0] - spline_sn(t_eval)[0]
gr1_sn = spline_sn_g(t_eval)[1] - spline_sn(t_eval)[1]
gr0_kn = spline_kn_g(t_eval)[0] - spline_kn(t_eval)[0]
gr1_kn = spline_kn_g(t_eval)[1] - spline_kn(t_eval)[1]
cr_sn_template = gr1_sn - gr0_sn
cr_kn_template = gr1_kn - gr0_kn

ax.scatter([fr_sn_template], [cr_sn_template], s=200, color="C0",
           marker="*", zorder=6, label="SN 1993J template")
ax.scatter([fr_kn_template], [cr_kn_template], s=200, color="C1",
           marker="*", zorder=6, label="AT2017gfo template")

ax.set_xlabel("Fade rate (mag/day)")
ax.set_ylabel("g-r colour rate (mag/day)")
ax.set_title("Feature space: every simulated event\nSN vs KN with classifier boundaries")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\Featurespace.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Events in KN candidate region (top right):")
print(f"  KN: {((kn_ev['fade_rate'] >= KN_FADE_THRESHOLD) & (kn_ev['colour_rate'] >= KN_COLOUR_RATE_THRESHOLD)).sum()} / {len(kn_ev)}")
print(f"  SN: {((sn_ev['fade_rate'] >= KN_FADE_THRESHOLD) & (sn_ev['colour_rate'] >= KN_COLOUR_RATE_THRESHOLD)).sum()} / {len(sn_ev)}")

This tries a k nearest neighbours classifier, which labels a new event by the majority vote of its five closest training events. It fits on the training features and predicts on the test set, printing precision, recall, and F1. It also plots the decision boundary, the dividing line the method draws between the two classes.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# --- Prepare training data (seeds 1-100) ---

X_train = train_valid[["fade_rate", "colour_rate"]].values
y_train = train_valid["label"].values

X_test = test_valid[["fade_rate", "colour_rate"]].values
y_test = test_valid["label"].values

# --- Fit kNN: k=5, distance weighting ---
knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
knn.fit(X_train, y_train)

# --- Predict on test set ---
y_pred = knn.predict(X_test)

# --- Metrics ---
tp = ((y_pred == 1) & (y_test == 1)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()
tn = ((y_pred == 0) & (y_test == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"kNN classifier (k=5, distance weighting):")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

# --- Decision boundary plot ---
# Create a grid covering the feature space
fr_min, fr_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
cr_min, cr_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2

xx, yy = np.meshgrid(np.linspace(fr_min, fr_max, 300),
                     np.linspace(cr_min, cr_max, 300))

Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 7))

# Decision boundary
ax.contourf(xx, yy, Z, alpha=0.15, cmap="RdYlBu", levels=[-0.5, 0.5, 1.5])
ax.contour(xx, yy, Z, colors="black", linewidths=0.8, levels=[0.5])

# All test set events
kn_test = test_valid[test_valid["label"] == 1]
sn_test = test_valid[test_valid["label"] == 0]

ax.scatter(sn_test["fade_rate"], sn_test["colour_rate"],
           s=8, alpha=0.2, color="C0", label=f"SN ({len(sn_test)} events)")
ax.scatter(kn_test["fade_rate"], kn_test["colour_rate"],
           s=40, alpha=0.8, color="C1", label=f"KN ({len(kn_test)} events)")

ax.set_xlabel("Fade rate (mag/day)")
ax.set_ylabel("g-r colour rate (mag/day)")
ax.set_title(f"kNN decision boundary (k=5, distance weighted)\n"
             f"Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\KNNFeaturespace.png', dpi=300, bbox_inches='tight')
plt.show()

This repeats the k nearest neighbours classifier but first standardises both features with a StandardScaler fitted on the training data only. Standardising rescales the features so neither dominates the distance calculation just by having larger numbers. It prints the accuracy scores and plots the resulting decision boundary.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# --- Feature matrices from existing train/test split ---
X_train = train_valid[["fade_rate", "colour_rate"]].values
y_train = train_valid["label"].values

X_test = test_valid[["fade_rate", "colour_rate"]].values
y_test = test_valid["label"].values

# --- Z-score using training set statistics only ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)    # use training mean/std, not test

# --- Fit kNN: k=5, distance weighting, on z-scored features --- ###################
knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
knn.fit(X_train_scaled, y_train)
####################################################################################
# --- Predict on test set ---
y_pred = knn.predict(X_test_scaled)

# --- Metrics ---
tp = ((y_pred == 1) & (y_test == 1)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()
tn = ((y_pred == 0) & (y_test == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"kNN classifier (k=5, distance weighted, z-scored):")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

# --- Decision boundary plot ---
fr_min, fr_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
cr_min, cr_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2

xx, yy = np.meshgrid(np.linspace(fr_min, fr_max, 300),
                     np.linspace(cr_min, cr_max, 300))

grid_scaled = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = knn.predict(grid_scaled).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 7))

ax.contourf(xx, yy, Z, alpha=0.15, cmap="RdYlBu", levels=[-0.5, 0.5, 1.5])
ax.contour(xx, yy, Z, colors="black", linewidths=0.8, levels=[0.5])

kn_test = test_valid[test_valid["label"] == 1]
sn_test = test_valid[test_valid["label"] == 0]

ax.scatter(sn_test["fade_rate"], sn_test["colour_rate"],
           s=8, alpha=0.2, color="C0", label=f"SN ({len(sn_test)} events)")
ax.scatter(kn_test["fade_rate"], kn_test["colour_rate"],
           s=40, alpha=0.8, color="C1", label=f"KN ({len(kn_test)} events)")

ax.set_xlabel("Fade rate (mag/day)")
ax.set_ylabel("g-r colour rate (mag/day)")
ax.set_title(f"kNN decision boundary (k=5, distance weighted, z-scored)\n"
             f"Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

This lays the soft score and k nearest neighbours classifiers side by side on the test set. It prints a table of true and false counts and the precision, recall, and F1 for each, marking the winner per metric. A grouped bar chart shows the same outcomes.

In [ ]:
# --- Comparison table: soft score vs kNN ---

# Soft score test set results
soft_tp = tp_te
soft_fp = fp_te
soft_fn = fn_te
soft_tn = len(test_valid) - tp_te - fp_te - fn_te

soft_precision = soft_tp / (soft_tp + soft_fp) if (soft_tp + soft_fp) > 0 else 0
soft_recall    = soft_tp / (soft_tp + soft_fn) if (soft_tp + soft_fn) > 0 else 0
soft_f1        = 2 * soft_precision * soft_recall / (soft_precision + soft_recall) if (soft_precision + soft_recall) > 0 else 0

# kNN test set results
knn_precision = precision
knn_recall    = recall
knn_f1        = f1

# --- Print table ---
print(f"{'Metric':<20} {'Soft Score':>12} {'kNN (k=5)':>12} {'Winner':>10}")
print("-" * 58)
print(f"{'TP':<20} {soft_tp:>12} {tp:>12}")
print(f"{'FP':<20} {soft_fp:>12} {fp:>12}")
print(f"{'FN':<20} {soft_fn:>12} {fn:>12}")
print(f"{'TN':<20} {soft_tn:>12} {tn:>12}")
print("-" * 58)
print(f"{'Precision':<20} {soft_precision:>12.3f} {knn_precision:>12.3f} {'kNN' if knn_precision > soft_precision else 'Soft':>10}")
print(f"{'Recall':<20} {soft_recall:>12.3f} {knn_recall:>12.3f} {'kNN' if knn_recall > soft_recall else 'Soft':>10}")
print(f"{'F1':<20} {soft_f1:>12.3f} {knn_f1:>12.3f} {'kNN' if knn_f1 > soft_f1 else 'Soft':>10}")
print("-" * 58)

# --- Bar chart: TP FP FN TN side by side ---
categories = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]

counts_soft = [soft_tp, soft_fn, soft_fp, soft_tn]
counts_knn  = [tp,      fn,      fp,      tn]

x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, counts_soft, width, label="Soft score (z-sum)", color="C1", alpha=0.8)
b2 = ax.bar(x + width/2, counts_knn,  width, label="kNN (k=5, distance weighted)", color="C2", alpha=0.8)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Classifier comparison: soft score vs kNN\n(test set — seeds 101-200)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\SoftScorevsKNNBar.png', dpi=300, bbox_inches='tight')
plt.show()

This tries Linear Discriminant Analysis, which separates the two classes with a single straight line in the feature plane. It fits on the training data, predicts on the test set, and prints precision, recall, and F1 along with the equation of the boundary line. It then plots that boundary over the test events.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# --- Feature matrices ---
X_train = train_valid[["fade_rate", "colour_rate"]].values
y_train = train_valid["label"].values
X_test  = test_valid[["fade_rate", "colour_rate"]].values
y_test  = test_valid["label"].values

# --- Fit LDA ---
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)

# --- Metrics ---
tp = ((y_pred == 1) & (y_test == 1)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()
tn = ((y_pred == 0) & (y_test == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Linear Discriminant Analysis:")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

# Print the decision boundary equation
coef = lda.coef_[0]
intercept = lda.intercept_[0]
print(f"\nDecision boundary:")
print(f"  {coef[0]:.3f} × fade_rate + {coef[1]:.3f} × colour_rate + {intercept:.3f} = 0")

# --- Plot ---
fr_min, fr_max = X_train[:, 0].min() - 0.2, X_train[:, 0].max() + 0.2
cr_min, cr_max = X_train[:, 1].min() - 0.2, X_train[:, 1].max() + 0.2

xx, yy = np.meshgrid(np.linspace(fr_min, fr_max, 300),
                     np.linspace(cr_min, cr_max, 300))

Z = lda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 7))

ax.contourf(xx, yy, Z, alpha=0.15, cmap="RdYlBu", levels=[-0.5, 0.5, 1.5])
ax.contour(xx, yy, Z, colors="black", linewidths=1.5, levels=[0.5])

kn_test = test_valid[test_valid["label"] == 1]
sn_test = test_valid[test_valid["label"] == 0]

ax.scatter(sn_test["fade_rate"], sn_test["colour_rate"],
           s=8, alpha=0.2, color="C0", label=f"SN ({len(sn_test)} events)")
ax.scatter(kn_test["fade_rate"], kn_test["colour_rate"],
           s=40, alpha=0.8, color="C1", label=f"KN ({len(kn_test)} events)")

ax.set_xlabel("Fade rate (mag/day)")
ax.set_ylabel("g-r colour rate (mag/day)")
ax.set_title(f"Linear Discriminant Analysis decision boundary")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\FeaturespaceLDA.png', dpi=300, bbox_inches='tight')
plt.show()

This compares the soft score and the linear classifier on the test set. It prints a table of counts and the precision, recall, and F1 for each. A bar chart shows the outcomes side by side.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# --- Feature matrices ---
X_train = train_valid[["fade_rate", "colour_rate"]].values
y_train = train_valid["label"].values
X_test  = test_valid[["fade_rate", "colour_rate"]].values
y_test  = test_valid["label"].values

# --- Fit classifiers ---
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)

# Soft score predictions
test_valid["soft_score"] = (
    (test_valid["fade_rate"]   - fr_mean) / fr_std +
    (test_valid["colour_rate"] - cs_mean) / cs_std
)
y_pred_soft = (test_valid["soft_score"] >= opt_thresh).values

# --- Compute metrics ---
def get_metrics(y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return tp, fp, fn, tn, precision, recall, f1

soft_tp, soft_fp, soft_fn, soft_tn, soft_prec, soft_rec, soft_f1 = get_metrics(y_test, y_pred_soft)
lda_tp,  lda_fp,  lda_fn,  lda_tn,  lda_prec,  lda_rec,  lda_f1  = get_metrics(y_test, y_pred_lda)

# --- Print table ---
print(f"{'Metric':<20} {'Soft Score':>12} {'LDA':>12}")
print("-" * 46)
print(f"{'KN detected (TP)':<20} {soft_tp:>12} {lda_tp:>12}")
print(f"{'KN missed (FN)':<20} {soft_fn:>12} {lda_fn:>12}")
print(f"{'SN false pos (FP)':<20} {soft_fp:>12} {lda_fp:>12}")
print(f"{'SN true neg (TN)':<20} {soft_tn:>12} {lda_tn:>12}")
print("-" * 46)
print(f"{'Precision':<20} {soft_prec:>12.3f} {lda_prec:>12.3f}")
print(f"{'Recall':<20} {soft_rec:>12.3f} {lda_rec:>12.3f}")
print(f"{'F1':<20} {soft_f1:>12.3f} {lda_f1:>12.3f}")
print("-" * 46)

# --- Bar chart ---
categories  = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]
counts_soft = [soft_tp, soft_fn, soft_fp, soft_tn]
counts_lda  = [lda_tp,  lda_fn,  lda_fp,  lda_tn]

x     = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, counts_soft, width, label="GW Trigger", color="C1", alpha=0.8)
b2 = ax.bar(x + width/2, counts_lda,  width, label="Full Sky Survey",        color="C3", alpha=0.8)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Classifier comparison: GW Trigger vs Full Sky Survey")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarSSvLDA.png', dpi=300, bbox_inches='tight')
plt.show()

This brings all three classifiers together, the soft score, k nearest neighbours, and the linear method, on the same test set. It prints one table listing their counts and accuracy scores. A grouped bar chart compares them at a glance.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# --- Feature matrices ---
X_train = train_valid[["fade_rate", "colour_rate"]].values
y_train = train_valid["label"].values
X_test  = test_valid[["fade_rate", "colour_rate"]].values
y_test  = test_valid["label"].values

# Z-score for kNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --- Fit all three classifiers ---
knn = KNeighborsClassifier(n_neighbors=5, weights="distance")
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
y_pred_lda = lda.predict(X_test)

# Soft score predictions
test_valid["soft_score"] = (
    (test_valid["fade_rate"]    - fr_mean) / fr_std +
    (test_valid["colour_rate"] - cs_mean) / cs_std
)
y_pred_soft = (test_valid["soft_score"] >= opt_thresh).values

# --- Compute metrics for all three ---
def get_metrics(y_true, y_pred):
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return tp, fp, fn, tn, precision, recall, f1

soft_tp, soft_fp, soft_fn, soft_tn, soft_prec, soft_rec, soft_f1 = get_metrics(y_test, y_pred_soft)
knn_tp,  knn_fp,  knn_fn,  knn_tn,  knn_prec,  knn_rec,  knn_f1  = get_metrics(y_test, y_pred_knn)
lda_tp,  lda_fp,  lda_fn,  lda_tn,  lda_prec,  lda_rec,  lda_f1  = get_metrics(y_test, y_pred_lda)

# --- Print table ---
print(f"{'Metric':<20} {'Soft Score':>12} {'kNN (k=5)':>12} {'LDA':>12}")
print("-" * 60)
print(f"{'KN detected (TP)':<20} {soft_tp:>12} {knn_tp:>12} {lda_tp:>12}")
print(f"{'KN missed (FN)':<20} {soft_fn:>12} {knn_fn:>12} {lda_fn:>12}")
print(f"{'SN false pos (FP)':<20} {soft_fp:>12} {knn_fp:>12} {lda_fp:>12}")
print(f"{'SN true neg (TN)':<20} {soft_tn:>12} {knn_tn:>12} {lda_tn:>12}")
print("-" * 60)
print(f"{'Precision':<20} {soft_prec:>12.3f} {knn_prec:>12.3f} {lda_prec:>12.3f}")
print(f"{'Recall':<20} {soft_rec:>12.3f} {knn_rec:>12.3f} {lda_rec:>12.3f}")
print(f"{'F1':<20} {soft_f1:>12.3f} {knn_f1:>12.3f} {lda_f1:>12.3f}")
print("-" * 60)

# --- Bar chart ---
categories  = ["KN detected\n(TP)", "KN missed\n(FN)", "SN false pos\n(FP)", "SN true neg\n(TN)"]
counts_soft = [soft_tp, soft_fn, soft_fp, soft_tn]
counts_knn  = [knn_tp,  knn_fn,  knn_fp,  knn_tn]
counts_lda  = [lda_tp,  lda_fn,  lda_fp,  lda_tn]

x     = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, counts_soft, width, label="Soft score (z-sum)", color="C1", alpha=0.8)
b2 = ax.bar(x,         counts_knn,  width, label="kNN (k=5)",          color="C2", alpha=0.8)
b3 = ax.bar(x + width, counts_lda,  width, label="LDA (linear)",       color="C3", alpha=0.8)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Event count")
ax.set_title("Classifier comparison: soft score vs kNN vs LDA\n(test set — seeds 101-200)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(r'C:\Users\edcon\Downloads\UCD exam papers\Masters Thesis\Master plots\BarSSvKNNvLDA.png', dpi=300, bbox_inches='tight')
plt.show()

This inspects the colour rates the classifier really encounters. Across one hundred kilonova injections it records the first detection epoch and the measured g-r colour rate for each. It prints summary statistics of both and lists the template colour rates for comparison, showing how noise and timing spread the real values around the ideal.

In [ ]:
colour_rates_actual = []
first_epochs = []

for seed in range(1, 101):
    np.random.seed(seed)
    kn_d  = sample_distance()
    kn_t0 = np.random.uniform(-1, 0)

    kn_r, _ = generate_observed_lightcurve('KN', kn_t0, kn_d)
    kn_g, _ = generate_observed_lightcurve_g('KN', kn_t0, kn_d)
    kn_r[kn_r > m_lim] = np.nan
    kn_g[kn_g > m_lim] = np.nan

    valid = (~np.isnan(kn_r)) & (~np.isnan(kn_g))
    if valid.sum() < 2:
        colour_rates_actual.append(np.nan)
        first_epochs.append(np.nan)
        continue

    idx = np.where(valid)[0]
    t0_obs, t1_obs = t_obs[idx[0]], t_obs[idx[1]]
    gr0 = kn_g[idx[0]] - kn_r[idx[0]]
    gr1 = kn_g[idx[1]] - kn_r[idx[1]]
    cr  = (gr1 - gr0) / (t1_obs - t0_obs)

    colour_rates_actual.append(cr)
    first_epochs.append(t0_obs)

colour_rates_actual = np.array(colour_rates_actual)
first_epochs        = np.array(first_epochs)

valid_mask = np.isfinite(colour_rates_actual)
print(f"KN events with >=2 detections: {valid_mask.sum()} / 100")
print(f"Mean first detection epoch:     {first_epochs[valid_mask].mean():.2f} days post merger")
print(f"Median first detection epoch:   {np.median(first_epochs[valid_mask]):.2f} days")
print()
print(f"Actual g-r colour rate seen by classifier:")
print(f"  Mean:   {colour_rates_actual[valid_mask].mean():.3f} mag/day")
print(f"  Median: {np.median(colour_rates_actual[valid_mask]):.3f} mag/day")
print(f"  Min:    {colour_rates_actual[valid_mask].min():.3f} mag/day")
print(f"  Max:    {colour_rates_actual[valid_mask].max():.3f} mag/day")
print()
print("Template reference values:")
print(f"  t=0 to t=1: 0.837 mag/day")
print(f"  t=1 to t=2: 0.301 mag/day")
print(f"  t=2 to t=3: 0.177 mag/day")

This runs the trained classifier on genuine data. It loads real DECam photometry from decam_photometry.csv, where DECam is the Dark Energy Camera and photometry means brightness measurements, and averages the g and r magnitudes per night. From the first nights it computes a fade rate and colour rate, feeds them through the deployed soft score, and prints the measurements and the final kilonova or non kilonova verdict.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\edcon\Downloads\kn_data\decam_photometry.csv')

# Get g and r, bin by night
gr = df[df['band'].isin(['g','r'])].copy()
gr['night'] = gr['mjd'].apply(np.floor)

g_nightly = gr[gr['band']=='g'].groupby('night')['mag'].mean().reset_index()
r_nightly = gr[gr['band']=='r'].groupby('night')['mag'].mean().reset_index()

print("r-band nightly:")
print(r_nightly.to_string())
print("\ng-band nightly:")
print(g_nightly.to_string())

# Fade rate: first two r nights
dt_r = r_nightly['night'].iloc[1] - r_nightly['night'].iloc[0]
dm_r = r_nightly['mag'].iloc[1] - r_nightly['mag'].iloc[0]
fade_rate = dm_r / dt_r

# g-r colour rate: first two paired nights
pairs = []
for _, rrow in r_nightly.iterrows():
    diff = (g_nightly['night'] - rrow['night']).abs()
    if diff.min() <= 1.0:
        g_mag = g_nightly.iloc[diff.argsort().iloc[0]]['mag']
        pairs.append({'night': rrow['night'], 'gr': g_mag - rrow['mag']})

dt_c = pairs[1]['night'] - pairs[0]['night']
colour_rate = (pairs[1]['gr'] - pairs[0]['gr']) / dt_c

print(f"\nFade rate: {fade_rate:.4f} mag/day")
print(f"Colour rate: {colour_rate:.4f} mag/day")

# Soft score
DEPLOY_FR_MEAN   = 0.3535
DEPLOY_FR_STD    = 0.2701
DEPLOY_CS_MEAN   = 0.1416
DEPLOY_CS_STD    = 0.3831
DEPLOY_THRESHOLD = 1.4285

z_fr = (fade_rate - DEPLOY_FR_MEAN) / DEPLOY_FR_STD
z_cs = (colour_rate - DEPLOY_CS_MEAN) / DEPLOY_CS_STD
score = z_fr + z_cs

print(f"\nSoft score: {score:.4f}")
print(f"Threshold: {DEPLOY_THRESHOLD}")
print(f"Classified as: {'KN' if score >= DEPLOY_THRESHOLD else 'non-KN'}")

This plots that real object. The left panel shows its g and r light curves over time, and the right panel shows the g-r colour evolving. The soft score verdict is written in the title.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Nightly data
r_mags = r_nightly['mag'].values
g_mags = g_nightly['mag'].values
r_nights = r_nightly['night'].values - r_nightly['night'].values[0]
g_nights = g_nightly['night'].values - g_nightly['night'].values[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: light curves
axes[0].plot(r_nights, r_mags, 'o-', color='C1', label='r band')
axes[0].plot(g_nights, g_mags, 'o-', color='C0', label='g band')
axes[0].invert_yaxis()
axes[0].set_xlabel('Days since merger')
axes[0].set_ylabel('Apparent magnitude')
axes[0].set_title('AT2017gfo light curve')
axes[0].legend(frameon=False)

# Right panel: g-r colour evolution
common_nights = []
gr_colours = []
for i, rrow in r_nightly.iterrows():
    diff = (g_nightly['night'] - rrow['night']).abs()
    if diff.min() <= 1.0:
        g_mag = g_nightly.iloc[diff.argsort().iloc[0]]['mag']
        common_nights.append(rrow['night'] - r_nightly['night'].iloc[0])
        gr_colours.append(g_mag - rrow['mag'])

axes[1].plot(common_nights, gr_colours, 'o-', color='purple')
axes[1].set_xlabel('Days since merger')
axes[1].set_ylabel('g - r colour (mag)')
axes[1].set_title('AT2017gfo colour evolution')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)

plt.suptitle(f'AT2017gfo | Soft score: 3.44 | Classified as: KN', fontsize=11)
plt.tight_layout()
plt.show()

This prints the fitted linear classifier's coefficients and intercept. Together these numbers define its straight line boundary. They are copied into later cells as the fixed constants LDA_COEF and LDA_INT.

In [ ]:
print('Coefficients:', lda.coef_)
print('Intercept:', lda.intercept_)

This loads a precomputed grid of Kasen kilonova models from kasen_grid.npz. Kasen models are detailed radiative transfer simulations of kilonovae from Daniel Kasen's 2017 work, each with different physical settings. It prints the names of the arrays stored inside the file.

In [ ]:
import numpy as np
grid = np.load('kasen_grid.npz')
print(list(grid.keys()))

This makes each stored model easy to evaluate at any moment. For every model it builds an interpolator, a function that reads a value between the tabulated time points, for the g and r bands, and records the time each model peaks in r. Missing entries are filled with a faint placeholder so the interpolators never break. It prints how many were built.

In [ ]:
from scipy.interpolate import interp1d

kasen_times  = grid['times']
kasen_M_g    = grid['M_g']
kasen_M_r    = grid['M_r']
kasen_params = grid['params']
N_KASEN = len(kasen_params)

_interp_r, _interp_g = [], []
_peak_time_r = np.zeros(N_KASEN)

for i in range(N_KASEN):
    mr = np.where(np.isfinite(kasen_M_r[i]), kasen_M_r[i], 99.0)
    mg = np.where(np.isfinite(kasen_M_g[i]), kasen_M_g[i], 99.0)
    _interp_r.append(interp1d(kasen_times, mr, bounds_error=False, fill_value=99.0))
    _interp_g.append(interp1d(kasen_times, mg, bounds_error=False, fill_value=99.0))
    valid = kasen_M_r[i] < 90
    if valid.any():
        _peak_time_r[i] = kasen_times[valid][np.argmin(kasen_M_r[i][valid])]

print(f"Built {len(_interp_r)} interpolators")

This defines generate_observed_lightcurve_kasen, which fakes an observation of one Kasen model. Given a model index, timing, distance, and observing times, it reads the model brightness, applies the distance, adds noise, and hides anything fainter than the limit. A companion function, sample_kasen_model, picks a model at random from the grid.

In [ ]:
def generate_observed_lightcurve_kasen(model_idx, t0, d_mpc, t_obs, m_lim=24.5, band='r'):
    """
    Generate an observed lightcurve for a Kasen model kilonova.

    model_idx : index into the Kasen grid (0 to 328)
    t0        : time of first detection in the observer frame (days)
    d_mpc     : distance in Mpc
    t_obs     : observation times (days)
    m_lim     : limiting apparent magnitude
    band      : 'r' or 'g'
    """
    t_rest = t_obs - t0
    t_model = t_rest + _peak_time_r[model_idx]

    interp = _interp_r[model_idx] if band == 'r' else _interp_g[model_idx]
    M = interp(t_model)

    M = np.where(t_model < 0, np.nan, M)
    M = np.where(M > 90, np.nan, M)

    mu = 5 * np.log10(d_mpc * 1e6) - 5
    m_true = M + mu

    sigma = np.where(m_true <= 20, 0.1,
            np.where(m_true >= 21, 0.2,
                     0.1 + 0.1 * (m_true - 20)))

    m_obs = m_true + np.random.normal(0, sigma)
    m_obs = np.where(m_obs > m_lim, np.nan, m_obs)

    return m_obs, sigma


def sample_kasen_model():
    """Draw a random model index from the Kasen grid (uniform)."""
    return np.random.randint(0, N_KASEN)

This is a quick test of the Kasen machinery. It draws one random model, prints its physical parameters, and generates its r and g magnitudes at days 0 and 1 for a source at 40 Mpc. Seeing finite numbers confirms the generator works.

In [ ]:
np.random.seed(2)
idx = sample_kasen_model()
print(kasen_params[idx])

m_r, sig_r = generate_observed_lightcurve_kasen(idx, t0=0, d_mpc=40, t_obs=np.array([0,1]), m_lim=24.5, band='r')
m_g, sig_g = generate_observed_lightcurve_kasen(idx, t0=0, d_mpc=40, t_obs=np.array([0,1]), m_lim=24.5, band='g')
print(m_r, m_g)

This checks how detectable the Kasen models are at 40 Mpc. It runs fifty random models and counts, at both epochs, whether each was seen in both bands, in r only, or missed. It prints those fractions, giving a feel for how often a colour can actually be measured.

In [ ]:
np.random.seed(0)
t_obs = np.array([0, 1])
d_mpc = 40
m_lim = 24.5

both_bands_ok = 0
r_only_ok = 0
neither_ok = 0

for i in range(50):
    idx = sample_kasen_model()
    m_r, _ = generate_observed_lightcurve_kasen(idx, 0, d_mpc, t_obs, m_lim, 'r')
    m_g, _ = generate_observed_lightcurve_kasen(idx, 0, d_mpc, t_obs, m_lim, 'g')

    r_ok = np.isfinite(m_r).all()
    g_ok = np.isfinite(m_g).all()

    if r_ok and g_ok:
        both_bands_ok += 1
    elif r_ok:
        r_only_ok += 1
    else:
        neither_ok += 1

print(f"Distance: {d_mpc} Mpc, limiting mag: {m_lim}")
print(f"Both bands detected at both epochs: {both_bands_ok}/50 ({100*both_bands_ok/50:.0f}%)")
print(f"r detected, g missing at some epoch: {r_only_ok}/50 ({100*r_only_ok/50:.0f}%)")
print(f"r missing at some epoch:             {neither_ok}/50 ({100*neither_ok/50:.0f}%)")

This is a self contained study of the whole Kasen grid. It draws kilonovae across distances of 50 to 300 Mpc, measures each event's fade rate and colour rate, and applies the two thresholds. It prints how many events are detected, pass the fade cut, and pass both cuts, plus summary statistics of the features.

In [ ]:
# ============================================================
# Standalone Kasen grid simulation
# Does not modify any existing variables, functions, or cells.
# Compares KN classifier performance: Kasen grid vs single template
# ============================================================

import numpy as np

# --- Simulation settings ---
N_SEEDS   = 100
N_KN      = 50          # KN events per seed
D_MIN, D_MAX = 50, 300  # Mpc
T_OBS     = np.array([0, 1])
M_LIM     = 24.5
FR_THRESH = 0.35
CR_THRESH = 0.10


def kasen_sample_distance(d_min=D_MIN, d_max=D_MAX):
    u = np.random.uniform()
    return (d_min**3 + u * (d_max**3 - d_min**3)) ** (1 / 3)


def kasen_fade_and_colour_rate(model_idx, d_mpc):
    """Generate one KN event from the Kasen grid and return (fade_rate, colour_rate)."""
    m_r, _ = generate_observed_lightcurve_kasen(model_idx, 0, d_mpc, T_OBS, M_LIM, band='r')
    m_g, _ = generate_observed_lightcurve_kasen(model_idx, 0, d_mpc, T_OBS, M_LIM, band='g')

    if not (np.isfinite(m_r).all()):
        return np.nan, np.nan

    fade_rate = (m_r[1] - m_r[0]) / (T_OBS[1] - T_OBS[0])

    if np.isfinite(m_g).all():
        gr0 = m_g[0] - m_r[0]
        gr1 = m_g[1] - m_r[1]
        colour_rate = (gr1 - gr0) / (T_OBS[1] - T_OBS[0])
    else:
        colour_rate = np.nan

    return fade_rate, colour_rate


# --- Run N_SEEDS realisations ---
all_fade   = []
all_colour = []
all_dist   = []
all_xlan   = []

for seed in range(N_SEEDS):
    np.random.seed(seed)
    for _ in range(N_KN):
        idx   = sample_kasen_model()
        d_mpc = kasen_sample_distance()
        fr, cr = kasen_fade_and_colour_rate(idx, d_mpc)
        all_fade.append(fr)
        all_colour.append(cr)
        all_dist.append(d_mpc)
        all_xlan.append(kasen_params[idx, 2])

all_fade   = np.array(all_fade)
all_colour = np.array(all_colour)
all_dist   = np.array(all_dist)
all_xlan   = np.array(all_xlan)

n_total      = len(all_fade)
n_detected   = np.isfinite(all_fade).sum()
n_pass_fr    = (np.nan_to_num(all_fade, nan=-1) > FR_THRESH).sum()
n_pass_both  = ((np.nan_to_num(all_fade, nan=-1) > FR_THRESH) &
                (np.nan_to_num(all_colour, nan=-1) > CR_THRESH)).sum()

print(f"Total simulated KN events:        {n_total}")
print(f"Detected (r-band, both epochs):   {n_detected} ({100*n_detected/n_total:.0f}%)")
print(f"Pass fade rate threshold only:    {n_pass_fr} ({100*n_pass_fr/n_total:.0f}%)")
print(f"Pass fade rate AND colour rate:   {n_pass_both} ({100*n_pass_both/n_total:.0f}%)")
print()
print(f"Fade rate (detected only):   mean={np.nanmean(all_fade):.3f}, std={np.nanstd(all_fade):.3f}")
print(f"Colour rate (detected only): mean={np.nanmean(all_colour):.3f}, std={np.nanstd(all_colour):.3f}")

This inspects a second Kasen data file, kasen_grid_synphot.npz, to learn what it holds. By examining the peak brightness of the first model, it decides whether magnitudes are stored as absolute or apparent, which matters before any comparison to real data. It prints the file's keys, array shapes, and its conclusion.

In [ ]:
# Cell 1: confirm whether kasen_grid_synphot.npz stores apparent or absolute mag
import numpy as np

NPZ = r"C:\Users\edcon\Downloads\kasen_grid_synphot.npz"
data = np.load(NPZ, allow_pickle=True)

print("Keys:", list(data.keys()), "\n")
for k in data.keys():
    arr = np.asarray(data[k])
    print(f"  {k:12s} shape={arr.shape} dtype={arr.dtype}")

# Adjust these two names if the printout above shows different keys.
g_key = next((k for k in data.keys() if k.lower() in ("g", "g_mag", "gmag", "mag_g")), None)
r_key = next((k for k in data.keys() if k.lower() in ("r", "r_mag", "rmag", "mag_r")), None)
print("\nDetected g key:", g_key, " r key:", r_key)

if g_key and r_key:
    g0 = np.asarray(data[g_key]); r0 = np.asarray(data[r_key])
    g_first = g0[0] if g0.ndim > 1 else g0
    r_first = r0[0] if r0.ndim > 1 else r0
    g_peak, r_peak = np.nanmin(g_first), np.nanmin(r_first)
    print(f"\nFirst model peak (min) g = {g_peak:.3f}")
    print(f"First model peak (min) r = {r_peak:.3f}")
    if -20 < g_peak < -10:
        print("\n=> ABSOLUTE magnitude. Apply distance modulus before any comparison to real photometry.")
    elif 10 < g_peak < 25:
        print("\n=> APPARENT magnitude at a fixed distance (around 40 Mpc if peaks are 17 to 18).")
    else:
        print("\n=> Out of expected range. Inspect g_first and r_first by hand.")

This fetches the full Kasen model archive so the raw physics files are available locally. It clones the dnkasen/Kasen_Kilonova_Models_2017 repository from GitHub, falling back to a direct zip download if git is unavailable. It prints how many model files ending in .h5 were found.

In [ ]:
# Clone (or download) the Kasen grid into your working directory
import subprocess, sys, zipfile, io, urllib.request
from pathlib import Path

WORKDIR = Path(r"C:\Users\edcon\Downloads")
KASEN_REPO = WORKDIR / "Kasen_Kilonova_Models_2017"
URL_GIT = "https://github.com/dnkasen/Kasen_Kilonova_Models_2017.git"
URL_ZIP = "https://codeload.github.com/dnkasen/Kasen_Kilonova_Models_2017/zip/refs/heads/master"

if KASEN_REPO.exists() and any(KASEN_REPO.rglob("knova_*.h5")):
    print("Already present at", KASEN_REPO)
else:
    ok = False
    try:
        subprocess.run(["git", "clone", "--depth", "1", URL_GIT, str(KASEN_REPO)],
                       check=True, cwd=str(WORKDIR))
        ok = True
    except Exception as e:
        print("git clone failed, falling back to zip download. Reason:", e)

    if not ok:
        print("Downloading zip ...")
        raw = urllib.request.urlopen(URL_ZIP).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            z.extractall(WORKDIR)
        # the zip extracts to a folder suffixed with the branch name
        extracted = next(WORKDIR.glob("Kasen_Kilonova_Models_2017-*"))
        extracted.rename(KASEN_REPO)

n = len(list(KASEN_REPO.rglob("knova_*.h5")))
print(f"Done. {n} .h5 files now under {KASEN_REPO}")

This turns the archive's filenames into a searchable table. Each Kasen filename encodes physical parameters such as ejecta mass, velocity, and lanthanide fraction, the amount of heavy rare earth elements, and short regular expressions pull these values out. It builds a grid table of every model, prints how many parsed cleanly, and demonstrates looking up specific models.

In [ ]:
# Cell 2: parse the Kasen grid filenames into a dataframe
import re
import numpy as np
import pandas as pd
from pathlib import Path

# Set this to wherever you cloned dnkasen/Kasen_Kilonova_Models_2017
KASEN_REPO = Path(r"C:\Users\edcon\Downloads\Kasen_Kilonova_Models_2017")
SUBDIRS = ["kilonova_models", "systematic_kilonova_model_grid"]

def parse_xlan(name):
    # Xlan1e-N.M encodes 10^(-N.M). The optional decimal group stops before .h5,
    # so the trailing period in e.g. Xlan1e-2.0.h5 is never captured.
    m = re.search(r"Xlan1e-(\d+(?:\.\d+)?)", name)
    return 10.0 ** (-float(m.group(1))) if m else None

def parse_tag(name, tag):
    # Underscore anchor stops _d matching inside _fd and _n inside _ns.
    m = re.search(rf"_{tag}(\d+(?:\.\d+)?)", name)
    return float(m.group(1)) if m else None

rows = []
for sub in SUBDIRS:
    for p in sorted((KASEN_REPO / sub).glob("*.h5")):
        name = p.name
        rows.append({
            "directory": sub, "filename": name, "path": str(p),
            "d":  parse_tag(name, "d"),  "n":  parse_tag(name, "n"),
            "m":  parse_tag(name, "m"),  "vk": parse_tag(name, "vk"),
            "Xlan": parse_xlan(name),
            "fd": parse_tag(name, "fd"), "vs": parse_tag(name, "vs"),
            "ns": parse_tag(name, "ns"),
        })

grid = pd.DataFrame(rows)

core = ["d", "n", "m", "vk", "Xlan"]
print("Total files parsed:", len(grid))
print("Missing any core param:", int(grid[core].isna().any(axis=1).sum()))
print(grid["directory"].value_counts().to_string())

def find_model(grid, m, vk, Xlan, directory=None, rtol=1e-3):
    sel = (np.isclose(grid["m"], m, rtol=rtol)
           & np.isclose(grid["vk"], vk, rtol=rtol)
           & np.isclose(grid["Xlan"], Xlan, rtol=rtol))
    if directory:
        sel &= grid["directory"] == directory
    return grid[sel]

print("\nBlue component (m=0.025, vk=0.30, Xlan=1e-4):")
print(find_model(grid, 0.025, 0.30, 1e-4)[["directory", "filename"]].to_string(index=False))
print("\nRed component (m=0.040, vk=0.15, Xlan=1e-1.5):")
print(find_model(grid, 0.040, 0.15, 10**-1.5)[["directory", "filename"]].to_string(index=False))

This defines the routines that convert a raw Kasen physics file into g and r magnitudes. Each file stores luminosity against frequency and time; the code turns that into apparent flux at a chosen distance, then folds it through the SDSS g and r filter response curves loaded from sdss_g.dat and sdss_r.dat to get band magnitudes. Folding through a filter weights the light by how much that filter actually transmits. It tests the routine on one model and prints its peak brightness.

In [ ]:
# Cell 3: convert a Kasen HDF5 model to apparent g and r magnitudes at a given distance
import h5py
import numpy as np

trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
C_AA = 2.99792458e18          # speed of light in Angstrom/s
MPC_CM = 3.0856775814913673e24

# Load your SDSS throughput curves once (two columns: wavelength A, transmission)
G_WL, G_TR = np.loadtxt(r"C:\Users\edcon\Downloads\sdss_g.dat", unpack=True)
R_WL, R_TR = np.loadtxt(r"C:\Users\edcon\Downloads\sdss_r.dat", unpack=True)

def band_flux(h5path, D_mpc=40.0):
    """Return time in days and band averaged apparent Fnu for g and r.
    Lnu is intrinsic, so divide by 4 pi D^2 to get apparent flux first."""
    D = D_mpc * MPC_CM
    with h5py.File(h5path, "r") as h:
        nu  = np.asarray(h["nu"], float)               # Hz
        t   = np.asarray(h["time"], float) / 86400.0   # days
        Lnu = np.asarray(h["Lnu"], float)              # erg/s/Hz [time, nu]
    lam = C_AA / nu
    order = np.argsort(lam)                             # ascending wavelength
    lam, Lnu = lam[order], Lnu[:, order]
    Fnu = Lnu / (4 * np.pi * D**2)                      # apparent flux density
    nub = C_AA / lam
    x = nub[::-1]
    out = {}
    for band, (wl, tr) in [("g", (G_WL, G_TR)), ("r", (R_WL, R_TR))]:
        T = np.interp(lam, wl, tr, left=0.0, right=0.0)
        den = trapz((T / nub)[::-1], x)
        fnu = np.array([trapz((Fnu[i] * T / nub)[::-1], x) / den
                        for i in range(Lnu.shape[0])])
        out[band] = fnu
    return t, out

def fnu_to_mag(fnu):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(fnu > 0, -2.5 * np.log10(fnu) - 48.60, np.nan)

def kasen_mags(h5path, D_mpc=40.0, tmin=0.1):
    """Convenience wrapper: time and AB magnitudes for one model, times >= tmin days."""
    t, f = band_flux(h5path, D_mpc)
    keep = t >= tmin
    return t[keep], fnu_to_mag(f["g"][keep]), fnu_to_mag(f["r"][keep])

# quick test on the blue component
blue_path = grid.query("directory=='systematic_kilonova_model_grid' "
                       "& abs(m-0.025)<1e-6 & abs(vk-0.30)<1e-6 & abs(Xlan-1e-4)<1e-7"
                       ).iloc[0]["path"]
t, g, r = kasen_mags(blue_path)
print(f"Blue g peak {np.nanmin(g):.2f}, r peak {np.nanmin(r):.2f}  (expect ~17 at 40 Mpc)")

This builds a two component kilonova by combining a blue and a red model. It picks a specific blue model with low lanthanides and faster ejecta and a red model with higher lanthanides and slower ejecta, adds their light in flux, the raw energy units, then converts back to magnitude. Real kilonovae like AT2017gfo show both components, so this is more realistic than one model alone. It plots the composite light curve and prints a couple of check values.

In [ ]:
# Cell 4: build the two component composite (systematic blue + original red)
import matplotlib.pyplot as plt

blue_path = grid.query("directory=='systematic_kilonova_model_grid' "
                       "& abs(m-0.025)<1e-6 & abs(vk-0.30)<1e-6 & abs(Xlan-1e-4)<1e-7"
                       ).iloc[0]["path"]
red_path  = grid.query("directory=='kilonova_models' "
                       "& abs(m-0.040)<1e-6 & abs(vk-0.15)<1e-6 & abs(Xlan-10**-1.5)<1e-7"
                       ).iloc[0]["path"]
print("blue:", blue_path.split('\\')[-1])
print("red :", red_path.split('\\')[-1])

tb, fb = band_flux(blue_path, 40.0)
tr, fr = band_flux(red_path, 40.0)

comp = {}
for b in ("g", "r"):
    fr_on_tb = np.interp(tb, tr, fr[b])      # red onto blue time grid, flux space
    comp[b] = fnu_to_mag(fb[b] + fr_on_tb)   # sum in flux, then convert to mag

# store on the blue grid for later reuse
comp_t = tb
keep = comp_t >= 0.1

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
for i, b in enumerate(("g", "r")):
    ax[i].plot(comp_t[keep], comp[b][keep], "k-", lw=2.5, label="blue + red composite")
    ax[i].set_title(f"{b} band"); ax[i].set_xlim(0, 15); ax[i].invert_yaxis()
    ax[i].grid(alpha=0.3); ax[i].set_xlabel("days since merger")
ax[0].set_ylabel("apparent AB magnitude"); ax[0].legend()
plt.tight_layout(); plt.show()

print(f"Composite r: peak {np.nanmin(comp['r'][keep]):.2f}, "
      f"r at 5d {np.interp(5, comp_t[keep], comp['r'][keep]):.2f}")

This finds which single Kasen model best matches the real AT2017gfo data. It loads the real r-band and g-band detections, then for every model computes chi square, a goodness of fit number where smaller means closer, summed over both bands. Points outside a model's time coverage are skipped. It sorts the models and prints the five best fitting ones.

In [ ]:
# Cell 5: rank all 385 models by chi square against real AT2017gfo photometry
import pandas as pd

R_XLSX = r"C:\Users\edcon\Downloads\at2017gfo_rband.csv.xlsx"
G_CSV  = r"C:\Users\edcon\Downloads\at2017gfo_gband.csv"

# fail loudly and early if a path is wrong, instead of deep in pandas
from pathlib import Path
for p in (R_XLSX, G_CSV):
    assert Path(p).exists(), f"Not found: {p}"

def detection_mask(col):
    # treat a row as an upper limit only if it clearly says so; everything else is a detection
    s = col.astype(str).str.strip().str.lower()
    return ~s.isin(["true", "t", "1", "1.0", "yes", "y"])

# r band: no error column, assign a flat 0.1 mag
rdf = pd.read_excel(R_XLSX)
rdf = rdf[detection_mask(rdf["upperlimit"])]
r_t, r_m = rdf["t_days"].to_numpy(float), rdf["mag"].to_numpy(float)
r_e = np.full_like(r_m, 0.1)

# g band: real mag_err, but drop the 2 detection rows where mag_err == 0 (missing placeholder)
gdf = pd.read_csv(G_CSV)
gdf = gdf[detection_mask(gdf["upperlimit"])]
gdf = gdf[gdf["mag_err"] != 0]
g_t, g_m, g_e = (gdf["t_days"].to_numpy(float),
                 gdf["mag"].to_numpy(float),
                 gdf["mag_err"].to_numpy(float))

print(f"r detections used: {len(r_t)}  |  g detections used: {len(g_t)}")

def chi2_band(dt, dm, de, mt, mm):
    ok = np.isfinite(mm)
    mt, mm = mt[ok], mm[ok]
    if mt.size < 2:
        return np.nan, 0
    inside = (dt >= mt.min()) & (dt <= mt.max())     # skip data outside model coverage
    if inside.sum() == 0:
        return np.nan, 0
    model = np.interp(dt[inside], mt, mm)
    return float(np.sum(((dm[inside] - model) / de[inside])**2)), int(inside.sum())

rows = []
for _, row in grid.iterrows():
    try:
        t, gmag, rmag = kasen_mags(row["path"])
    except Exception:
        continue
    cg, ng = chi2_band(g_t, g_m, g_e, t, gmag)
    cr, nr = chi2_band(r_t, r_m, r_e, t, rmag)
    if np.isnan(cg) or np.isnan(cr):
        continue
    rows.append({**row[["directory", "filename", "path", "m", "vk", "Xlan"]].to_dict(),
                 "chi2_g": cg, "chi2_r": cr, "chi2_tot": cg + cr, "n_pts": ng + nr})

fit = pd.DataFrame(rows).sort_values("chi2_tot").reset_index(drop=True)
print("\nTop 5 best fit single models:")
print(fit.head(5)[["filename", "m", "vk", "Xlan", "chi2_g", "chi2_r", "chi2_tot"]].to_string(index=False))

This shows the winners visually. It plots the blue plus red composite and the five best fitting Kasen models against the real AT2017gfo detections, one panel per band. The figure is saved as a PNG.

In [ ]:
# Cell 6: two panel plot, save PNG
import matplotlib.pyplot as plt

top5 = fit.head(5)
colors = plt.cm.viridis(np.linspace(0, 0.85, 5))
k = comp_t >= 0.1

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
for i, b in enumerate(("g", "r")):
    ax[i].plot(comp_t[k], comp[b][k], "k-", lw=3, zorder=5, label="blue + red composite")
    for c, (_, row) in zip(colors, top5.iterrows()):
        t, gmag, rmag = kasen_mags(row["path"])
        ax[i].plot(t, gmag if b == "g" else rmag, "-", color=c, lw=1.3,
                   label=row["filename"].replace("knova_", "").replace(".h5", ""))
    ax[i].set_title(f"{b} band"); ax[i].set_xlim(0, 15); ax[i].invert_yaxis()
    ax[i].grid(alpha=0.3); ax[i].set_xlabel("days since merger")

ax[0].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2,
               zorder=6, label="AT2017gfo g")
ax[1].plot(r_t, r_m, "o", ms=4, color="k", zorder=6, label="AT2017gfo r")

ax[0].set_ylabel("apparent AB magnitude")
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\at2017gfo_kasen_grid_fit.png", dpi=150)
plt.show()
print("saved at2017gfo_kasen_grid_fit.png")

This examines the five best models one at a time, first in r-band then in g-band. Each panel overlays one model on the real AT2017gfo data and labels it with its fit score. Both figures are saved.

In [ ]:
# Cell 7: top 5 models plotted individually, r band then g band
import matplotlib.pyplot as plt

top5 = fit.head(5).reset_index(drop=True)

def plot_band(band, data_t, data_m, data_e, fname):
    fig, ax = plt.subplots(1, 5, figsize=(18, 3.8), sharex=True, sharey=True)
    for j, row in top5.iterrows():
        t, gmag, rmag = kasen_mags(row["path"])
        model = rmag if band == "r" else gmag
        ax[j].plot(t, model, "-", color="tab:blue", lw=1.6, label="model")
        if data_e is None:
            ax[j].plot(data_t, data_m, "o", ms=4, color="k", label="AT2017gfo")
        else:
            ax[j].errorbar(data_t, data_m, yerr=data_e, fmt="o", ms=4,
                           color="k", capsize=2, label="AT2017gfo")
        chi = row["chi2_r"] if band == "r" else row["chi2_g"]
        label = row["filename"].replace("knova_", "").replace(".h5", "")
        ax[j].set_title(f"{label}\nchi2_{band} = {chi:.1f}", fontsize=8)
        ax[j].set_xlim(0, 15); ax[j].grid(alpha=0.3)
        ax[j].set_xlabel("days since merger")
    ax[0].invert_yaxis()
    ax[0].set_ylabel("apparent AB magnitude")
    ax[0].legend(fontsize=7)
    fig.suptitle(f"{band} band: five best fit Kasen models vs AT2017gfo", y=1.02)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()

# r band first
plot_band("r", r_t, r_m, r_e, r"C:\Users\edcon\Downloads\top5_rband.png")
# then g band
plot_band("g", g_t, g_m, g_e, r"C:\Users\edcon\Downloads\top5_gband.png")
print("saved top5_rband.png and top5_gband.png")

This fits smooth spline curves to the five best models in both bands, so their brightness can be read at any time. Here the smoothing is set to zero, giving an exact curve faithful to the smooth Kasen output; a spline is a smooth curve through data points. It prints each model's g-r colour at day one.

In [ ]:
# Build g and r splines for the top 5 models
import numpy as np
from scipy.interpolate import UnivariateSpline

splines_g, splines_r, spline_labels, spline_paths = {}, {}, {}, {}

for rank in range(5):
    row = fit.iloc[rank]
    t, gmag, rmag = kasen_mags(row["path"])
    mg = np.isfinite(gmag)
    mr = np.isfinite(rmag)
    # s=0 gives an exact interpolating cubic, faithful to the smooth Kasen models
    splines_g[rank] = UnivariateSpline(t[mg], gmag[mg], k=3, s=0.0)
    splines_r[rank] = UnivariateSpline(t[mr], rmag[mr], k=3, s=0.0)
    spline_labels[rank] = row["filename"].replace("knova_", "").replace(".h5", "")
    spline_paths[rank] = row["path"]

print("Built splines for ranks 0 to 4. Call e.g. splines_r[0](np.array([1,2,3])).")
for rank in range(5):
    sg, sr = splines_g[rank], splines_r[rank]
    print(f"  {rank}: {spline_labels[rank]:40s} g-r at 1d = {float(sg(1)-sr(1)):+.2f}")

This plots the single best fitting Kasen model against the real AT2017gfo data in both bands. The r-band error bars are drawn growing as the source fades, which is cosmetic rather than measured. The figure is saved.

In [ ]:
# Top model 1 of 5, r band error bars that grow as it fades
import matplotlib.pyplot as plt
import numpy as np
RANK = 0
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

# r errors grow with faintness (cosmetic, not measured)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t, rmag, "-", color="tab:red", lw=1.8, label="model")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t, gmag, "-", color="tab:green", lw=1.8, label="model")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].set_ylim(25, 17)
ax[1].set_ylim(25, 17)
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the second best fitting Kasen model against the real AT2017gfo data in both bands. The r-band error bars are drawn growing as the source fades, which is cosmetic rather than measured. The figure is saved.

In [ ]:
# Top model 2 of 5, r band error bars that grow as it fades
import matplotlib.pyplot as plt
import numpy as np
RANK = 1
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t, rmag, "-", color="tab:red", lw=1.8, label="model")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t, gmag, "-", color="tab:green", lw=1.8, label="model")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].set_ylim(25, 17)
ax[1].set_ylim(25, 17)
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the third best fitting Kasen model against the real AT2017gfo data in both bands. The r-band error bars are drawn growing as the source fades, which is cosmetic rather than measured. The figure is saved.

In [ ]:
# Top model 3 of 5, r band error bars that grow as it fades
import matplotlib.pyplot as plt
import numpy as np
RANK = 2
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t, rmag, "-", color="tab:red", lw=1.8, label="model")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t, gmag, "-", color="tab:green", lw=1.8, label="model")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].set_ylim(25, 17)
ax[1].set_ylim(25, 17)
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the fourth best fitting Kasen model against the real AT2017gfo data in both bands. The r-band error bars are drawn growing as the source fades, which is cosmetic rather than measured. The figure is saved.

In [ ]:
# Top model 4 of 5, r band error bars that grow as it fades
import matplotlib.pyplot as plt
import numpy as np
RANK = 3
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t, rmag, "-", color="tab:red", lw=1.8, label="model")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t, gmag, "-", color="tab:green", lw=1.8, label="model")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].set_ylim(25, 17)
ax[1].set_ylim(25, 17)
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the fifth best fitting Kasen model against the real AT2017gfo data in both bands. The r-band error bars are drawn growing as the source fades, which is cosmetic rather than measured. The figure is saved.

In [ ]:
# Top model 5 of 5, r band error bars that grow as it fades
import matplotlib.pyplot as plt
import numpy as np
RANK = 4
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t, rmag, "-", color="tab:red", lw=1.8, label="model")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t, gmag, "-", color="tab:green", lw=1.8, label="model")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].set_ylim(25, 17)
ax[1].set_ylim(25, 17)
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}.png", dpi=150, bbox_inches="tight")
plt.show()

This splines the best fitting model with a light smoothing so a small wiggle in the faint tail is ironed out. It fits r and g curves, prints checks that they have no gaps, and plots them over the real data. The figure is saved.

In [ ]:
# Spline the Kasen model line, same recipe as the AT2017gfo spline
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 0

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def spline_model(t, m, s):
    # sort
    idx = np.argsort(t); t, m = t[idx], m[idx]
    # clean NaNs
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    # remove duplicate times (required for spline)
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s), t.min(), t.max()

# s smooths the faint tail wiggle. raise for smoother, lower toward 0 for exact
spline_r_mod, rlo, rhi = spline_model(t_mod, r_mod, s=0.05)
spline_g_mod, glo, ghi = spline_model(t_mod, g_mod, s=0.05)

t_grid = np.linspace(0.1, 15, 500)
r_smooth = spline_r_mod(t_grid)
g_smooth = spline_g_mod(t_grid)

print("R model NaNs:", np.isnan(r_smooth).any(), "| range:", round(r_smooth.min(),2), round(r_smooth.max(),2))
print("G model NaNs:", np.isnan(g_smooth).any(), "| range:", round(g_smooth.min(),2), round(g_smooth.max(),2))

# growing r error bars (cosmetic)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].plot(t_grid, r_smooth, "-", color="tab:red", lw=2, label="model spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].plot(t_grid, g_smooth, "-", color="tab:green", lw=2, label="model spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_spline.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the best model's smooth r and g curves with a fixed shaded band of plus or minus 0.2 magnitudes around each. The band is a simple uncertainty envelope for display. The figure is saved.

In [ ]:
# Model 1: r and g splined with envelope
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 0
BAND = 0.2

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

sr = central_spline(t_mod, r_mod)
sg = central_spline(t_mod, g_mod)
tt = np.linspace(0.1, 15, 400)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].fill_between(tt, sr(tt) - BAND, sr(tt) + BAND, color="tab:red", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[0].plot(tt, sr(tt), "-", color="tab:red", lw=2, label="model r spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].fill_between(tt, sg(tt) - BAND, sg(tt) + BAND, color="tab:green", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[1].plot(tt, sg(tt), "-", color="tab:green", lw=2, label="model g spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\model_{RANK+1}_splined.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the second model's smooth r and g curves with a fixed shaded band of plus or minus 0.2 magnitudes around each. The band is a simple uncertainty envelope for display. The figure is saved.

In [ ]:
# Model 2: r and g splined with envelope
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 1
BAND = 0.2

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

sr = central_spline(t_mod, r_mod)
sg = central_spline(t_mod, g_mod)
tt = np.linspace(0.1, 15, 400)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].fill_between(tt, sr(tt) - BAND, sr(tt) + BAND, color="tab:red", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[0].plot(tt, sr(tt), "-", color="tab:red", lw=2, label="model r spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].fill_between(tt, sg(tt) - BAND, sg(tt) + BAND, color="tab:green", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[1].plot(tt, sg(tt), "-", color="tab:green", lw=2, label="model g spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\model_{RANK+1}_splined.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the third model's smooth r and g curves with a fixed shaded band of plus or minus 0.2 magnitudes around each. The band is a simple uncertainty envelope for display. The figure is saved.

In [ ]:
# Model 3: r and g splined with envelope
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 2
BAND = 0.2

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

sr = central_spline(t_mod, r_mod)
sg = central_spline(t_mod, g_mod)
tt = np.linspace(0.1, 15, 400)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].fill_between(tt, sr(tt) - BAND, sr(tt) + BAND, color="tab:red", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[0].plot(tt, sr(tt), "-", color="tab:red", lw=2, label="model r spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].fill_between(tt, sg(tt) - BAND, sg(tt) + BAND, color="tab:green", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[1].plot(tt, sg(tt), "-", color="tab:green", lw=2, label="model g spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\model_{RANK+1}_splined.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the fourth model's smooth r and g curves with a fixed shaded band of plus or minus 0.2 magnitudes around each. The band is a simple uncertainty envelope for display. The figure is saved.

In [ ]:
# Model 4: r and g splined with envelope
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 3
BAND = 0.2

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

sr = central_spline(t_mod, r_mod)
sg = central_spline(t_mod, g_mod)
tt = np.linspace(0.1, 15, 400)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].fill_between(tt, sr(tt) - BAND, sr(tt) + BAND, color="tab:red", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[0].plot(tt, sr(tt), "-", color="tab:red", lw=2, label="model r spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].fill_between(tt, sg(tt) - BAND, sg(tt) + BAND, color="tab:green", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[1].plot(tt, sg(tt), "-", color="tab:green", lw=2, label="model g spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\model_{RANK+1}_splined.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the fifth model's smooth r and g curves with a fixed shaded band of plus or minus 0.2 magnitudes around each. The band is a simple uncertainty envelope for display. The figure is saved.

In [ ]:
# Model 5: r and g splined with envelope
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
RANK = 4
BAND = 0.2

row = fit.iloc[RANK]
t_mod, g_mod, r_mod = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

sr = central_spline(t_mod, r_mod)
sg = central_spline(t_mod, g_mod)
tt = np.linspace(0.1, 15, 400)
r_e_plot = 0.05 * 10**(0.4 * (r_m - r_m.min()) * 0.5)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
ax[0].fill_between(tt, sr(tt) - BAND, sr(tt) + BAND, color="tab:red", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[0].plot(tt, sr(tt), "-", color="tab:red", lw=2, label="model r spline")
ax[0].errorbar(r_t, r_m, yerr=r_e_plot, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo r")
ax[0].set_title(f"r band   chi2_r = {row['chi2_r']:.1f}")
ax[1].fill_between(tt, sg(tt) - BAND, sg(tt) + BAND, color="tab:green", alpha=0.15,
                   label=f"+/- {BAND:.2f} mag envelope")
ax[1].plot(tt, sg(tt), "-", color="tab:green", lw=2, label="model g spline")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=4, color="k", capsize=2, label="AT2017gfo g")
ax[1].set_title(f"g band   chi2_g = {row['chi2_g']:.1f}")
for a in ax:
    a.set_xlim(0, 15); a.set_ylim(25, 17); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
fig.suptitle(f"Model {RANK+1}: {label}   chi2_tot = {row['chi2_tot']:.1f}", y=1.02)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\model_{RANK+1}_splined.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the g-r colour over time for the best fitting model against the real AT2017gfo colour. The g-r colour is the g magnitude minus the r magnitude. The figure is saved.

In [ ]:
# Top model 1 of 5: g - r colour
import matplotlib.pyplot as plt
import numpy as np
RANK = 0
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
# observed g - r: interpolate r onto g times, within r coverage only
inside = (g_t >= r_t.min()) & (g_t <= r_t.max())
obs_gr = g_m[inside] - np.interp(g_t[inside], r_t, r_m)
obs_gr_e = np.sqrt(g_e[inside]**2 + 0.1**2)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(t, gmag - rmag, "-", color="tab:purple", lw=1.8, label="model g - r")
ax.errorbar(g_t[inside], obs_gr, yerr=obs_gr_e, fmt="o", ms=4, color="k",
            capsize=2, label="AT2017gfo g - r")
ax.set_xlim(0, 3); ax.set_ylim(-2, 4); ax.grid(alpha=0.3)
ax.set_xlabel("days since merger"); ax.set_ylabel("g - r colour")
ax.legend(fontsize=8)
ax.set_title(f"Model {RANK+1}: {label}\nchi2_tot = {row['chi2_tot']:.1f}", fontsize=9)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_gr.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the g-r colour over time for the second best model against the real AT2017gfo colour. The g-r colour is the g magnitude minus the r magnitude. The figure is saved.

In [ ]:
# Top model 2 of 5: g - r colour
import matplotlib.pyplot as plt
import numpy as np
RANK = 1
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
inside = (g_t >= r_t.min()) & (g_t <= r_t.max())
obs_gr = g_m[inside] - np.interp(g_t[inside], r_t, r_m)
obs_gr_e = np.sqrt(g_e[inside]**2 + 0.1**2)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(t, gmag - rmag, "-", color="tab:purple", lw=1.8, label="model g - r")
ax.errorbar(g_t[inside], obs_gr, yerr=obs_gr_e, fmt="o", ms=4, color="k",
            capsize=2, label="AT2017gfo g - r")
ax.set_xlim(0, 3); ax.set_ylim(-2, 4); ax.grid(alpha=0.3)
ax.set_xlabel("days since merger"); ax.set_ylabel("g - r colour")
ax.legend(fontsize=8)
ax.set_title(f"Model {RANK+1}: {label}\nchi2_tot = {row['chi2_tot']:.1f}", fontsize=9)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_gr.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the g-r colour over time for the third best model against the real AT2017gfo colour. The g-r colour is the g magnitude minus the r magnitude. The figure is saved.

In [ ]:
# Top model 3 of 5: g - r colour
import matplotlib.pyplot as plt
import numpy as np
RANK = 2
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
inside = (g_t >= r_t.min()) & (g_t <= r_t.max())
obs_gr = g_m[inside] - np.interp(g_t[inside], r_t, r_m)
obs_gr_e = np.sqrt(g_e[inside]**2 + 0.1**2)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(t, gmag - rmag, "-", color="tab:purple", lw=1.8, label="model g - r")
ax.errorbar(g_t[inside], obs_gr, yerr=obs_gr_e, fmt="o", ms=4, color="k",
            capsize=2, label="AT2017gfo g - r")
ax.set_xlim(0, 3); ax.set_ylim(-2, 4); ax.grid(alpha=0.3)
ax.set_xlabel("days since merger"); ax.set_ylabel("g - r colour")
ax.legend(fontsize=8)
ax.set_title(f"Model {RANK+1}: {label}\nchi2_tot = {row['chi2_tot']:.1f}", fontsize=9)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_gr.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the g-r colour over time for the fourth best model against the real AT2017gfo colour. The g-r colour is the g magnitude minus the r magnitude. The figure is saved.

In [ ]:
# Top model 4 of 5: g - r colour
import matplotlib.pyplot as plt
import numpy as np
RANK = 3
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
inside = (g_t >= r_t.min()) & (g_t <= r_t.max())
obs_gr = g_m[inside] - np.interp(g_t[inside], r_t, r_m)
obs_gr_e = np.sqrt(g_e[inside]**2 + 0.1**2)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(t, gmag - rmag, "-", color="tab:purple", lw=1.8, label="model g - r")
ax.errorbar(g_t[inside], obs_gr, yerr=obs_gr_e, fmt="o", ms=4, color="k",
            capsize=2, label="AT2017gfo g - r")
ax.set_xlim(0, 3); ax.set_ylim(-2, 4); ax.grid(alpha=0.3)
ax.set_xlabel("days since merger"); ax.set_ylabel("g - r colour")
ax.legend(fontsize=8)
ax.set_title(f"Model {RANK+1}: {label}\nchi2_tot = {row['chi2_tot']:.1f}", fontsize=9)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_gr.png", dpi=150, bbox_inches="tight")
plt.show()

This plots the g-r colour over time for the fifth best model against the real AT2017gfo colour. The g-r colour is the g magnitude minus the r magnitude. The figure is saved.

In [ ]:
# Top model 5 of 5: g - r colour
import matplotlib.pyplot as plt
import numpy as np
RANK = 4
row = fit.iloc[RANK]
t, gmag, rmag = kasen_mags(row["path"])
label = row["filename"].replace("knova_", "").replace(".h5", "")
inside = (g_t >= r_t.min()) & (g_t <= r_t.max())
obs_gr = g_m[inside] - np.interp(g_t[inside], r_t, r_m)
obs_gr_e = np.sqrt(g_e[inside]**2 + 0.1**2)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(t, gmag - rmag, "-", color="tab:purple", lw=1.8, label="model g - r")
ax.errorbar(g_t[inside], obs_gr, yerr=obs_gr_e, fmt="o", ms=4, color="k",
            capsize=2, label="AT2017gfo g - r")
ax.set_xlim(0, 3); ax.set_ylim(-2, 4); ax.grid(alpha=0.3)
ax.set_xlabel("days since merger"); ax.set_ylabel("g - r colour")
ax.legend(fontsize=8)
ax.set_title(f"Model {RANK+1}: {label}\nchi2_tot = {row['chi2_tot']:.1f}", fontsize=9)
plt.tight_layout()
plt.savefig(rf"C:\Users\edcon\Downloads\top_model_{RANK+1}_gr.png", dpi=150, bbox_inches="tight")
plt.show()

## From Kasen spectra to a light curve

Each Kasen file is not a light curve. It is a time series of spectra. For every moment after the merger it stores how much energy the kilonova emits at every wavelength. Our job is to turn that into a single brightness in the g and r filters at each moment, which is what a telescope actually measures.

We get there in five steps.

**1. Read the spectra.** Each file gives `Lnu`, the energy emitted per second per unit frequency, laid out as a grid of time against frequency. This is intrinsic luminosity, the total output of the source, not what we would see from Earth.

**2. Move it to a distance.** Brightness falls off with the square of distance. We divide the luminosity by 4 pi D squared, with D set to 40 Mpc, the distance of AT2017gfo. This converts intrinsic luminosity into the flux that would arrive at a telescope here.

**3. Pass it through the filters.** A g or r filter only lets certain wavelengths through. We multiply the spectrum by the SDSS g and r throughput curves, so each filter keeps only the light it is sensitive to and blocks the rest.

**4. Add up the light in each band.** We integrate the filtered flux across all wavelengths, weighting by frequency in the way a photon counting detector does. This collapses the whole spectrum at one moment into a single number per filter, the total flux seen through g and through r.

**5. Convert to magnitude.** Astronomers measure brightness in magnitudes, not flux, so we apply the standard AB magnitude formula. The result is one g magnitude and one r magnitude.

Repeat steps three to five at every time step and the single numbers join up into a curve. That is the light curve. Doing this for all 385 files turns the model grid into 385 synthetic kilonova light curves we can compare against the real AT2017gfo and feed to the classifier.

This pushes the five best Kasen models through the deployed classifiers to see whether the filter would catch real kilonovae. Measuring each model's fade rate and colour rate between day 1 and day 2, it computes the soft score and the linear score using the fixed training constants and prints a verdict table. It then plots the five models in the feature plane alongside both decision boundaries and, if present, the cloud of simulated events.

In [ ]:
# Run the 5 best-fit Kasen kilonovae through the detection filter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# deployed soft score constants (training seeds 1-100)
DEPLOY_FR_MEAN, DEPLOY_FR_STD = 0.3535, 0.2701
DEPLOY_CS_MEAN, DEPLOY_CS_STD = 0.1416, 0.3831
DEPLOY_THRESHOLD = 1.4285
# LDA from cell 110
LDA_COEF = np.array([20.14, 12.24]); LDA_INT = -20.44

EPOCHS = (1.0, 2.0)   # post-peak one day baseline. measure after the ~0.5 d peak

def central_spline(t, m, s=0.05):
    ok = np.isfinite(t) & np.isfinite(m); t, m = t[ok], m[ok]
    o = np.argsort(t); t, m = t[o], m[o]
    t, iu = np.unique(t, return_index=True); m = m[iu]
    return UnivariateSpline(t, m, s=s)

rows = []
for rank in range(5):
    r = fit.iloc[rank]
    t_mod, g_mod, r_mod = kasen_mags(r["path"])
    sr = central_spline(t_mod, r_mod); sg = central_spline(t_mod, g_mod)
    r0, r1 = float(sr(EPOCHS[0])), float(sr(EPOCHS[1]))
    g0, g1 = float(sg(EPOCHS[0])), float(sg(EPOCHS[1]))
    fade   = (r1 - r0) / (EPOCHS[1] - EPOCHS[0])
    colour = ((g1 - r1) - (g0 - r0)) / (EPOCHS[1] - EPOCHS[0])
    soft = ((fade - DEPLOY_FR_MEAN) / DEPLOY_FR_STD
            + (colour - DEPLOY_CS_MEAN) / DEPLOY_CS_STD)
    lda_val = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows.append({
        "model": r["filename"].replace("knova_", "").replace(".h5", ""),
        "fade_rate": round(fade, 3), "colour_rate": round(colour, 3),
        "soft_score": round(soft, 2),
        "soft": "KN" if soft >= DEPLOY_THRESHOLD else "SN",
        "lda_decision": round(lda_val, 2),
        "lda": "KN" if lda_val > 0 else "SN",
    })

verdict = pd.DataFrame(rows)
print(f"Measurement epochs: day {EPOCHS[0]} and day {EPOCHS[1]}  (soft threshold {DEPLOY_THRESHOLD})\n")
print(verdict.to_string(index=False))

# feature space plot with both decision boundaries
fig, ax = plt.subplots(figsize=(8, 6.5))
try:  # overlay the simulated SN/KN cloud if events_2d is in memory
    v = events_2d.dropna(subset=["fade_rate", "colour_rate"])
    ax.scatter(v[v.label==0]["fade_rate"], v[v.label==0]["colour_rate"],
               s=6, alpha=0.15, color="C0", label="simulated SN")
    ax.scatter(v[v.label==1]["fade_rate"], v[v.label==1]["colour_rate"],
               s=15, alpha=0.3, color="C1", label="simulated KN")
except NameError:
    pass

ax.scatter(verdict["fade_rate"], verdict["colour_rate"], s=130, color="k",
           marker="*", zorder=6, label="5 Kasen models")
for _, r in verdict.iterrows():
    ax.annotate(r["model"].split("_Xlan")[0], (r["fade_rate"], r["colour_rate"]),
                fontsize=6, xytext=(5, 4), textcoords="offset points")

# soft score = threshold line:  fr_norm + cs_norm = THR
fr = np.linspace(-1, 3, 100)
cs_line = (DEPLOY_THRESHOLD - (fr - DEPLOY_FR_MEAN)/DEPLOY_FR_STD) * DEPLOY_CS_STD + DEPLOY_CS_MEAN
ax.plot(fr, cs_line, "r--", lw=1.5, label="soft score threshold")
# LDA boundary: coef . x + int = 0
cs_lda = -(LDA_COEF[0]*fr + LDA_INT) / LDA_COEF[1]
ax.plot(fr, cs_lda, "g-.", lw=1.5, label="LDA boundary")

ax.set_xlim(-1, 3); ax.set_ylim(-1, 3)
ax.set_xlabel("r fade rate (mag/day)"); ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title("Five Kasen kilonovae in the classifier feature plane")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\kasen5_through_filter.png", dpi=150, bbox_inches="tight")
plt.show()

### Running every Kasen model from raw spectra through the classifier

It takes each model in its raw form, a full spectrum evolving in time, and carries it all the way to a pass or fail decision from both classifiers. It does this for every model in the grid, then reports what fraction the pipeline catches. Everything you saw in the coverage and contaminant figures depends on the table this cell builds.

**Setup constants.** `trapz` is numerical integration by the trapezoid rule, with a line that handles the fact that newer numpy renamed the function. `C_AA` is the speed of light written in Angstroms per second, and `MPC_CM` is one megaparsec in centimetres. Both are unit bridges, so the frequency, wavelength, and distance numbers all end up in the same system. The block of deployment constants is the frozen classifier again, the same trained soft score and LDA settings used everywhere else, so the grid faces the real filter. `EPOCHS = (1.0, 2.0)` fixes the two observation days the features are measured across.

**The mags_fast function.** This is the spectra to magnitude converter, the exact steps we discussed but in code.
- It opens one HDF5 model file and reads three arrays. `nu` is frequency, `time` is converted from seconds to days by dividing by 86400, and `Lnu` is the spectral luminosity, the intrinsic energy output per second per unit frequency, laid out as time against frequency.
- `lam = C_AA/nu` converts frequency to wavelength so the arrays can be matched to the filter curves, then sorts them into increasing wavelength order.
- `Fnu = Lnu/(4*np.pi*D**2)` is the inverse square law. It dims the intrinsic luminosity into the flux that would actually arrive at a telescope at distance `D`. This is the step from what the source emits to what we would see.
- The loop over g and r does the band collapse. `T` is the filter transmission interpolated onto the model wavelengths, meaning how much light each filter lets through at each wavelength. The `T/nub` factor carries the extra one over frequency that makes this a photon count rather than an energy sum, matching how a real detector works. `den` is the filter normalisation and `num` is the flux after the filter has had its say, both got by integrating across frequency.
- The final line turns that into an AB magnitude with `-2.5*log10(num/den) - 48.60`. The 48.60 is the standard AB zero point. Any band with no positive flux returns nan, meaning invisible.
- It returns the time array and the g and r magnitude curves, after trimming very early times below `tmin`.

**The mag_at helper.** This reads a magnitude off a light curve at one requested day. It ignores any nan points first, then interpolates between the good ones. It is how the code samples each model at exactly day one and day two.

**The main loop.** For every model in `grid` it builds the two light curves, then measures the same two features your classifier uses. `fade` is the change in r magnitude per day between the two epochs. `colour` is the change in g minus r colour per day, so it tracks how fast the source is reddening or bluing. It then computes both scores, the standardised soft score and the LDA dot product, and records whether each one passed. Every model becomes one row holding its physical parameters, its two features, its two scores, and its two pass flags.

**The output.** All rows are collected into `gridfit`, the master table the later plots read from. The prints report the catch fraction for each classifier, meaning what percentage of the full grid the pipeline would flag as kilonova candidates. These are the grid level numbers, measured at a fixed 40 Mpc, before realistic distance sampling pulls them down further.

In [ ]:
# Run all 385 Kasen models through the detection filter
import numpy as np
import pandas as pd
import h5py

trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
C_AA = 2.99792458e18; MPC_CM = 3.0856775814913673e24

DEPLOY_FR_MEAN, DEPLOY_FR_STD = 0.3535, 0.2701
DEPLOY_CS_MEAN, DEPLOY_CS_STD = 0.1416, 0.3831
DEPLOY_THRESHOLD = 1.4285
LDA_COEF = np.array([20.14, 12.24]); LDA_INT = -20.44
EPOCHS = (1.0, 2.0)

def mags_fast(h5path, D_mpc=40.0, tmin=0.1):
    D = D_mpc * MPC_CM
    with h5py.File(h5path, "r") as h:
        nu = np.asarray(h["nu"], float); t = np.asarray(h["time"], float)/86400.
        Lnu = np.asarray(h["Lnu"], float)
    lam = C_AA/nu; o = np.argsort(lam); lam, Lnu = lam[o], Lnu[:, o]
    Fnu = Lnu/(4*np.pi*D**2); nub = C_AA/lam; x = nub[::-1]; out = {}
    for b, (wl, tr) in [("g", (G_WL, G_TR)), ("r", (R_WL, R_TR))]:
        T = np.interp(lam, wl, tr, left=0, right=0); den = trapz((T/nub)[::-1], x)
        num = trapz((Fnu*(T/nub)[None, :])[:, ::-1], x, axis=1)
        out[b] = np.where(num > 0, -2.5*np.log10(num/den) - 48.60, np.nan)
    k = t >= tmin; return t[k], out["g"][k], out["r"][k]

def mag_at(t, m, e):
    ok = np.isfinite(m); return float(np.interp(e, t[ok], m[ok]))

rows, curves = [], {}
for i, mod in grid.reset_index(drop=True).iterrows():
    t, g, r = mags_fast(mod["path"])
    curves[i] = (t, g, r)
    r0, r1 = mag_at(t, r, EPOCHS[0]), mag_at(t, r, EPOCHS[1])
    g0, g1 = mag_at(t, g, EPOCHS[0]), mag_at(t, g, EPOCHS[1])
    fade   = (r1 - r0) / (EPOCHS[1] - EPOCHS[0])
    colour = ((g1 - r1) - (g0 - r0)) / (EPOCHS[1] - EPOCHS[0])
    soft = ((fade - DEPLOY_FR_MEAN)/DEPLOY_FR_STD
            + (colour - DEPLOY_CS_MEAN)/DEPLOY_CS_STD)
    lda = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows.append({"idx": i, "directory": mod["directory"], "filename": mod["filename"],
                 "m": mod["m"], "vk": mod["vk"], "Xlan": mod["Xlan"],
                 "fade_rate": fade, "colour_rate": colour, "soft_score": soft,
                 "soft_pass": soft >= DEPLOY_THRESHOLD, "lda_pass": lda > 0})

gridfit = pd.DataFrame(rows)
n = len(gridfit)
print(f"Ran {n} models")
print(f"Soft score catch: {gridfit.soft_pass.sum()}/{n} = {100*gridfit.soft_pass.mean():.0f}%")
print(f"LDA catch:        {gridfit.lda_pass.sum()}/{n} = {100*gridfit.lda_pass.mean():.0f}%")

This visualises the full grid result in several ways. It plots all the light curves coloured green for caught and red for missed, the feature plane with the threshold line, and the catch fraction against each physical parameter, ejecta mass, velocity, and lanthanide fraction. All figures are saved, showing which kinds of kilonovae the filter misses.

In [ ]:
# Plot every model: r and g light curves, feature plane, parameter coverage
import matplotlib.pyplot as plt
import numpy as np

caught = gridfit.set_index("idx")["soft_pass"].to_dict()

# all 385 light curves, coloured by catch (green) / miss (red)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
for i, (t, g, r) in curves.items():
    c = "tab:green" if caught[i] else "tab:red"
    k = t <= 15
    ax[0].plot(t[k], r[k], "-", color=c, lw=0.5, alpha=0.35)
    ax[1].plot(t[k], g[k], "-", color=c, lw=0.5, alpha=0.35)
ax[0].errorbar(r_t, r_m, yerr=0.1, fmt="o", ms=3, color="k", capsize=2, zorder=6, label="AT2017gfo r")
ax[1].errorbar(g_t, g_m, yerr=g_e, fmt="o", ms=3, color="k", capsize=2, zorder=6, label="AT2017gfo g")
for a, b in zip(ax, ("r", "g")):
    a.set_xlim(0, 15); a.set_ylim(28, 16); a.grid(alpha=0.3)
    a.set_xlabel("days since merger"); a.set_title(f"{b} band, all {len(curves)} models")
    a.legend(fontsize=8)
ax[0].set_ylabel("apparent AB magnitude")
ax[0].plot([], [], "-", color="tab:green", label="caught"); ax[0].plot([], [], "-", color="tab:red", label="missed")
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\grid_all_lightcurves.png", dpi=150, bbox_inches="tight")
plt.show()

# feature plane
fig, axf = plt.subplots(figsize=(8, 6.5))
p = gridfit[gridfit.soft_pass]; f = gridfit[~gridfit.soft_pass]
axf.scatter(f.fade_rate, f.colour_rate, s=16, color="tab:red", alpha=0.5, label=f"miss ({len(f)})")
axf.scatter(p.fade_rate, p.colour_rate, s=16, color="tab:green", alpha=0.5, label=f"catch ({len(p)})")
fr = np.linspace(-1.5, 3.5, 100)
axf.plot(fr, (DEPLOY_THRESHOLD - (fr-DEPLOY_FR_MEAN)/DEPLOY_FR_STD)*DEPLOY_CS_STD + DEPLOY_CS_MEAN,
         "k--", lw=1.5, label="soft score threshold")
axf.set_xlim(-1.5, 3.5); axf.set_ylim(-1, 2.5)
axf.set_xlabel("r fade rate (mag/day)"); axf.set_ylabel("g - r colour rate (mag/day)")
axf.set_title(f"Full grid in the feature plane ({len(gridfit)} models)")
axf.legend(fontsize=8)
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\grid_feature_plane.png", dpi=150, bbox_inches="tight")
plt.show()

# catch fraction vs each parameter
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for a, col, name in zip(axes, ["m", "vk", "Xlan"],
                        ["ejecta mass (Msun)", "ejecta velocity (c)", "lanthanide fraction"]):
    frac = gridfit.groupby(col)["soft_pass"].mean()
    a.bar(range(len(frac)), frac.values, color="tab:green", alpha=0.8)
    a.set_xticks(range(len(frac)))
    a.set_xticklabels([f"{v:.0e}" if col == "Xlan" else f"{v:g}" for v in frac.index],
                      rotation=45, fontsize=8)
    a.set_ylim(0, 1); a.set_ylabel("catch fraction"); a.set_xlabel(name); a.grid(alpha=0.3, axis="y")
fig.suptitle("Filter coverage across the Kasen parameter space", y=1.02)
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\grid_coverage_params.png", dpi=150, bbox_inches="tight")
plt.show()

### Building the shock cooling SN IIb contaminant population

This cell builds a fake population of the one supernova type most likely to fool the classifier, and then measures how often it gets through. Shock cooling Type IIb supernovae have an early blue bump that can briefly look like a fast blue transient, which is exactly the corner of the feature plane where kilonovae live.

**Where the numbers come from.** The two bands `G` and `R` hold published population statistics from Crawford 2025, one set for the g band and one for r. Each entry is a mean and a standard deviation. The `m` values are slopes in magnitudes per day and the `a` values are break times in days. `GR_EARLY` is the mean g minus r colour in the first three days, taken from Ayala 2025. It sets the colour zero point, meaning the baseline colour these supernovae start from.

**The deployment constants.** `DEPLOY_FR_MEAN` through `LDA_INT` are the frozen settings of your trained classifiers, carried over from the deployment stage. They are not fitted here. They are the exact soft score normalisation, soft score threshold, and LDA line that your pipeline would use in the wild. Reusing them is what makes this a fair test. The contaminants face the real classifier, not a fresh one tuned to reject them.

**The lightning bolt light curve.** `lightning_bolt` is a simple model of the SN IIb shape. It is three straight line segments in magnitude against time, joined at two bend points `a1` and `a2`. The slopes `m1`, `m2`, `m3` control how fast each segment rises or falls. The constant `b2` is just an anchor that makes the three pieces join up smoothly and sets the value to zero at time `a2`. The nickname comes from the jagged bend and rebound shape the three lines trace out. `draw` is a one line helper that rolls a random value for every parameter, pulling from the mean and spread in the chosen dictionary.

**The sampling loop.** The `while` loop keeps going until it has four thousand valid supernovae. On each pass it draws a full g and r parameter set. The `if not (...)` line is a reality check. It throws away any draw where the break times are unphysical, meaning the first bend must land after day one and the second must sit at least a day after the first. Bad draws are skipped with `continue` and do not count.

**Turning each supernova into two features.** For each valid draw the code picks a random catch phase `t1`, the moment a survey happens to first see it, then sets `t2` one day later. This mimics our real two epoch cadence. The `fade` is the change in r magnitude across that one day. The colour rate is built the same way. It measures the g minus r colour at each epoch, adds the `GR_EARLY` offset, and takes the change across the day. So every synthetic supernova is reduced to the same two numbers our classifier judges kilonovae on, fade rate and colour rate.

**Running the classifiers.** `soft` is the soft score, the sum of the two features after each is standardised by its deployment mean and spread. `lda` is the linear discriminant score, the dot product of the features with the trained coefficients plus the intercept. Each row records whether the supernova passed the soft score threshold and whether it passed the LDA boundary. A pass here is a false positive, because these are all supernovae, not kilonovae.

**The result.** The final prints report how many of the four thousand slipped through each classifier. Low percentages are the good outcome. They are the evidence that shock cooling SNe IIb sit far enough from the kilonova region that the filter rejects almost all of them.

In [ ]:
# Shock-cooling SN IIb contaminant from Crawford 2025 lightning bolt + Ayala 2025 colour
import numpy as np
import pandas as pd

# Crawford Table 2 population stats (mean, std). Slopes mag/day, times days.
G = dict(m1=(-2.1,1.0), m2=(0.26,0.12), m3=(-0.08,0.06), a1=(7.2,3.3), a2=(14.0,1.5))
R = dict(m1=(-1.10,0.76), m2=(0.19,0.16), m3=(-0.09,0.05), a1=(7.7,3.1), a2=(14.5,1.3))
GR_EARLY = -0.18   # Ayala mean g-r in first 3 days (colour zero point)

DEPLOY_FR_MEAN, DEPLOY_FR_STD = 0.3535, 0.2701
DEPLOY_CS_MEAN, DEPLOY_CS_STD = 0.1416, 0.3831
DEPLOY_THRESHOLD = 1.4285
LDA_COEF = np.array([20.14, 12.24]); LDA_INT = -20.44
BASELINE = 1.0

def lightning_bolt(t, p):
    """Normalized SN IIb magnitude (0 = trough) from the three-line model."""
    m1,m2,m3,a1,a2 = p['m1'],p['m2'],p['m3'],p['a1'],p['a2']
    b2 = -m2*a2
    if t <= a1:   return m1*t + a1*(m2-m1) + b2
    elif t <= a2: return m2*t + b2
    else:         return m3*t + a2*(m2-m3) + b2

def draw(D, rng): return {k: rng.normal(*D[k]) for k in D}

rng = np.random.default_rng(0)
rows = []
while len(rows) < 4000:
    pg, pr = draw(G, rng), draw(R, rng)
    if not (pr['a1'] > 1 and pr['a2'] > pr['a1'] + 1):
        continue
    t1 = rng.uniform(1.0, pr['a2'])          # random survey catch phase
    t2 = t1 + BASELINE
    fade = (lightning_bolt(t2,pr) - lightning_bolt(t1,pr)) / BASELINE
    gr1 = lightning_bolt(t1,pg) - lightning_bolt(t1,pr) + GR_EARLY
    gr2 = lightning_bolt(t2,pg) - lightning_bolt(t2,pr) + GR_EARLY
    colour = (gr2 - gr1) / BASELINE
    soft = ((fade-DEPLOY_FR_MEAN)/DEPLOY_FR_STD + (colour-DEPLOY_CS_MEAN)/DEPLOY_CS_STD)
    lda  = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows.append({"fade_rate": fade, "colour_rate": colour,
                 "soft_pass": soft >= DEPLOY_THRESHOLD, "lda_pass": lda > 0})

sce = pd.DataFrame(rows)
n = len(sce)
print(f"Synthetic shock-cooling SNe IIb: {n}")
print(f"Soft score false positives: {sce.soft_pass.sum()}/{n} = {100*sce.soft_pass.mean():.2f}%")
print(f"LDA false positives:        {sce.lda_pass.sum()}/{n} = {100*sce.lda_pass.mean():.2f}%")

### Plotting both populations against the decision boundaries

This cell draws the picture that makes the contaminant test visual. It puts the fake supernovae and the caught kilonovae on the same two axes, then overlays the two decision boundaries, so you can see with your eyes how well the classifier separates the two groups.

**The two clouds.** The first scatter is the four thousand shock cooling supernovae in blue, plotted by their fade rate and colour rate. The second scatter is the kilonovae, in orange. It pulls them from `gridfit`, the earlier full grid run, and keeps only the ones that were both detected and passed the soft score. The `try` and `except` wrapper means the plot still works even if that grid run is not in memory. If `gridfit` is missing it simply skips the orange points instead of crashing.

**Drawing the boundaries.** A classifier boundary is the line where the score exactly equals its threshold. Everything on one side passes and everything on the other side is rejected. The code draws each boundary by fixing the score at its cutoff and solving for the colour rate at every fade rate. For the soft score, setting the score equal to `DEPLOY_THRESHOLD` and rearranging gives the dashed black line. For the LDA, setting the score to zero and rearranging gives the dash dot green line. These are the same two rules from the previous cell, now shown as lines rather than tested as pass or fail.

**Reading the plane.** The x axis is r fade rate in magnitudes per day, so points further right fade faster. The y axis is the g minus r colour rate. The grey dotted vertical line at zero marks no fade, a useful reference. Anything up and to the right of a boundary is classified as a kilonova candidate. So the goal is for the orange kilonovae to sit on the pass side and the blue supernovae to stay on the reject side.

**What the figure demonstrates.** The blue supernova cloud clusters low and to the left, meaning slow and only mildly blue. The orange kilonovae stretch up and to the right, meaning fast and blue. The two boundaries slice cleanly between them, with only a handful of blue points crossing over. That small overlap is the false positive rate printed in the previous cell, now made physical. The plot is saved to disk as a PNG so it can go straight into the thesis.

In [ ]:
# Feature plane: shock-cooling SNe vs kilonovae, with decision boundaries
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8.5, 6.5))

# shock-cooling SN contaminant cloud
ax.scatter(sce.fade_rate, sce.colour_rate, s=10, alpha=0.25, color="tab:blue",
           label=f"shock-cooling SNe IIb ({len(sce)})")

# kilonovae from the full grid run, if available
try:
    kn = gridfit[gridfit.soft_pass]
    ax.scatter(kn.fade_rate, kn.colour_rate, s=14, alpha=0.5, color="tab:orange",
               label=f"Kasen kilonovae ({len(kn)} caught)")
except NameError:
    pass

# decision boundaries
fr = np.linspace(-2, 3.5, 100)
ax.plot(fr, (DEPLOY_THRESHOLD - (fr-DEPLOY_FR_MEAN)/DEPLOY_FR_STD)*DEPLOY_CS_STD + DEPLOY_CS_MEAN,
        "k--", lw=1.5, label="soft score threshold")
ax.plot(fr, -(LDA_COEF[0]*fr + LDA_INT)/LDA_COEF[1], "g-.", lw=1.5, label="LDA boundary")

ax.axvline(0, color="grey", lw=0.6, ls=":")
ax.set_xlim(-2, 3.5); ax.set_ylim(-1.5, 2.5)
ax.set_xlabel("r fade rate (mag/day)"); ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title("Shock-cooling SNe IIb vs kilonovae in the filter feature plane")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\sce_contaminant_plane.png", dpi=150, bbox_inches="tight")
plt.show()

This confronts the filter with real transients from the sky. It queries the Fink broker, which processes the Zwicky Transient Facility alert stream, over the internet for objects it classifies as SN candidate, builds the two features from each object's nightly g and r photometry, and scores them. It prints how many real supernovae were wrongly flagged as kilonovae, the real world false positive rate.

In [ ]:
# Shock cooling fade-rate tail: r and g band light curves over the first 10 days
import numpy as np, pandas as pd, matplotlib.pyplot as plt

G = dict(m1=(-2.1,1.0), m2=(0.26,0.12), m3=(-0.08,0.06), a1=(7.2,3.3), a2=(14.0,1.5))
R = dict(m1=(-1.10,0.76), m2=(0.19,0.16), m3=(-0.09,0.05), a1=(7.7,3.1), a2=(14.5,1.3))
BASELINE = 1.0

def lightning_bolt(t, p):
    m1,m2,m3,a1,a2 = p['m1'],p['m2'],p['m3'],p['a1'],p['a2']
    b2 = -m2*a2
    t = np.asarray(t, float)
    return np.where(t <= a1, m1*t + a1*(m2-m1) + b2,
           np.where(t <= a2, m2*t + b2, m3*t + a2*(m2-m3) + b2))

def draw(D, rng): return {k: rng.normal(*D[k]) for k in D}

rng = np.random.default_rng(0)
events = []
while len(events) < 4000:
    pg, pr = draw(G, rng), draw(R, rng)
    if not (pr['a1'] > 1 and pr['a2'] > pr['a1'] + 1):
        continue
    t1 = rng.uniform(1.0, pr['a2']); t2 = t1 + BASELINE
    r_fade = float(lightning_bolt(t2, pr) - lightning_bolt(t1, pr)) / BASELINE
    events.append({"r_fade": r_fade, "pg": pg, "pr": pr})

# top 100 by r band fade rate
tail = sorted(events, key=lambda e: e["r_fade"], reverse=True)[:100]

t = np.linspace(0, 10, 300)

# ---------- r band ----------
fig, ax = plt.subplots(figsize=(8, 5.5))
for e in tail:
    ax.plot(t, lightning_bolt(t, e["pr"]), "-", color="tab:red", lw=0.6, alpha=0.4)
ax.set_xlim(0, 10); ax.invert_yaxis(); ax.grid(alpha=0.3)
ax.set_xlabel("phase (days)")
ax.set_ylabel("normalised mag (0 = trough)")
ax.set_title("r band: fade-rate tail (top 100), first 10 days")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\sce_tail_rband_10day.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- g band ----------
fig, ax = plt.subplots(figsize=(8, 5.5))
for e in tail:
    ax.plot(t, lightning_bolt(t, e["pg"]), "-", color="tab:blue", lw=0.6, alpha=0.4)
ax.set_xlim(0, 10); ax.invert_yaxis(); ax.grid(alpha=0.3)
ax.set_xlabel("phase (days)")
ax.set_ylabel("normalised mag (0 = trough)")
ax.set_title("g band: same 100 tail events, first 10 days")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\sce_tail_gband_10day.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Pull a whole class of real SNe from Fink and run them through the filter
import requests, numpy as np, pandas as pd
requests.packages.urllib3.disable_warnings()

try:
    from tqdm import tqdm
    HAVE_TQDM = True
except ImportError:
    HAVE_TQDM = False

LATESTS = "https://api.ztf.fink-portal.org/api/v1/latests"
OBJECTS = "https://api.ztf.fink-portal.org/api/v1/objects"
FR_MEAN,FR_STD,CS_MEAN,CS_STD,THR = 0.3535,0.2701,0.1416,0.3831,1.4285
LDA_COEF, LDA_INT = np.array([20.14,12.24]), -20.44

def get_class(finkclass, n=100):
    body = {"class": finkclass, "n": n, "output-format": "json"}
    r = requests.post(LATESTS, json=body, verify=False)
    return sorted({row["i:objectId"] for row in r.json()})

def fink_lc(zid):
    body = {"objectId": zid, "columns":"i:jd,i:magpsf,i:sigmapsf,i:fid",
            "output-format":"json"}
    js = requests.post(OBJECTS, json=body, verify=False).json()
    if not js: return None
    d = pd.DataFrame(js).rename(columns={
        "i:jd":"jd","i:magpsf":"mag","i:sigmapsf":"magerr","i:fid":"fid"})
    for c in ["jd","mag","fid"]: d[c]=pd.to_numeric(d[c],errors="coerce")
    return d.dropna()

def features(d):
    d=d.copy(); d["night"]=np.floor(d.jd-0.5)
    g=d[d.fid==1].groupby("night").mag.mean()
    r=d[d.fid==2].groupby("night").mag.mean()
    common=sorted(set(g.index)&set(r.index))
    if len(common)<2: return None
    n1,n2=common[0],common[1]; dt=n2-n1
    if dt<=0 or dt>2: return None
    fade=(r[n2]-r[n1])/dt
    colour=((g[n2]-r[n2])-(g[n1]-r[n1]))/dt
    return fade,colour

CLASS = "SN candidate"
ids = get_class(CLASS, n=10000)
print(f"pulled {len(ids)} ids for class '{CLASS}'")

rows=[]
loop = tqdm(ids, desc=CLASS) if HAVE_TQDM else ids
for i, zid in enumerate(loop, 1):
    if not HAVE_TQDM and i % 50 == 0:
        print(f"  {i}/{len(ids)}  ({len(rows)} testable so far)")
    try: f=features(fink_lc(zid))
    except Exception: f=None
    if f is None: continue
    fade,colour=f
    soft=(fade-FR_MEAN)/FR_STD+(colour-CS_MEAN)/CS_STD
    lda=float(LDA_COEF@np.array([fade,colour])+LDA_INT)
    rows.append({"id":zid,"fade":fade,"colour":colour,
                 "soft":soft,"lda":lda,
                 "verdict":"KN" if (soft>=THR or lda>0) else "SN"})

res=pd.DataFrame(rows)
print(f"\nclass '{CLASS}': {len(res)} testable of {len(ids)}, "
      f"{(res.verdict=='KN').sum()} false positives "
      f"({100*(res.verdict=='KN').mean():.1f}%)")
print(res.sort_values('soft',ascending=False).head(10).to_string(index=False))

This plots the real Fink supernova population in the feature plane. Objects rejected as supernovae and the false positives are shown separately, with both decision boundaries and the old one dimensional fade cut drawn for reference. The figure is saved.

In [ ]:
# Graph the Fink SN population, zoomed out so the boundaries are fully visible
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 6.5))

sn = res[res.verdict == "SN"]
fp = res[res.verdict == "KN"]   # false positives

ax.scatter(sn.fade, sn.colour, s=12, alpha=0.35, color="tab:blue",
           label=f"rejected as SN ({len(sn)})")
if len(fp):
    ax.scatter(fp.fade, fp.colour, s=45, color="tab:red", edgecolor="k",
               zorder=5, label=f"false positives ({len(fp)})")

# fixed wide window so both lines are visible end to end
XMIN, XMAX = -2.0, 3.5
YMIN, YMAX = -2.0, 2.5

fr = np.linspace(XMIN, XMAX, 200)
ax.plot(fr, (THR - (fr-FR_MEAN)/FR_STD)*CS_STD + CS_MEAN,
        "k--", lw=1.5, label="soft score threshold")
ax.plot(fr, -(LDA_COEF[0]*fr + LDA_INT)/LDA_COEF[1],
        "g-.", lw=1.5, label="LDA boundary")

ax.axvline(0.35, color="grey", lw=0.8, ls=":", label="old 1D fade cut")
ax.axhline(0, color="grey", lw=0.5, ls=":")

ax.set_xlim(XMIN, XMAX)
ax.set_ylim(YMIN, YMAX)
ax.set_xlabel("r fade rate (mag/day)")
ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title(f"Fink '{CLASS}' population through the filter "
             f"({100*(res.verdict=='KN').mean():.1f}% false positive)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\fink_sn_population_plane.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Real Type IIb SNe from Fink, run through the same filter
CLASS_IIB = "(TNS) SN IIb"
ids_iib = get_class(CLASS_IIB, n=10000)
print(f"pulled {len(ids_iib)} ids for class '{CLASS_IIB}'")

rows_iib = []
loop = tqdm(ids_iib, desc=CLASS_IIB) if HAVE_TQDM else ids_iib
for i, zid in enumerate(loop, 1):
    if not HAVE_TQDM and i % 50 == 0:
        print(f"  {i}/{len(ids_iib)}  ({len(rows_iib)} testable so far)")
    try:
        f = features(fink_lc(zid))
    except Exception:
        f = None
    if f is None:
        continue
    fade, colour = f
    soft = (fade - FR_MEAN)/FR_STD + (colour - CS_MEAN)/CS_STD
    lda = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows_iib.append({"id": zid, "fade": fade, "colour": colour,
                     "soft": soft, "lda": lda,
                     "verdict": "KN" if (soft >= THR or lda > 0) else "SN"})

res_iib = pd.DataFrame(rows_iib)
if len(res_iib):
    n_fp = (res_iib.verdict == "KN").sum()
    print(f"\nclass '{CLASS_IIB}': {len(res_iib)} testable of {len(ids_iib)}, "
          f"{n_fp} false positives ({100*(res_iib.verdict=='KN').mean():.1f}%)")
    print(res_iib.sort_values('soft', ascending=False).head(10).to_string(index=False))
else:
    print(f"\nclass '{CLASS_IIB}': 0 testable of {len(ids_iib)} pulled "
          f"(none had two nights of g and r within the 2 day gap)")

In [ ]:
# Feature plane: real Fink Type IIb SNe against the decision boundaries
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8.5, 6.5))

# caught Kasen kilonovae, for reference, if the grid run is in memory
try:
    kn = gridfit[gridfit.soft_pass]
    ax.scatter(kn.fade_rate, kn.colour_rate, s=14, alpha=0.4, color="tab:orange",
               label=f"Kasen kilonovae ({len(kn)} caught)")
except NameError:
    pass

# real IIb, split by verdict
passed = res_iib[res_iib.verdict == "KN"]
rejected = res_iib[res_iib.verdict == "SN"]
ax.scatter(rejected.fade, rejected.colour, s=30, color="tab:blue",
           edgecolor="k", linewidth=0.3, label=f"IIb rejected ({len(rejected)})")
ax.scatter(passed.fade, passed.colour, s=55, color="red", marker="X",
           edgecolor="k", linewidth=0.4, label=f"IIb false positive ({len(passed)})")

# decision boundaries
fr = np.linspace(-2, 3.5, 100)
ax.plot(fr, (THR - (fr - FR_MEAN)/FR_STD)*CS_STD + CS_MEAN,
        "k--", lw=1.5, label="soft score threshold")
ax.plot(fr, -(LDA_COEF[0]*fr + LDA_INT)/LDA_COEF[1],
        "g-.", lw=1.5, label="LDA boundary")

ax.axvline(0, color="grey", lw=0.6, ls=":")
ax.set_xlim(-2, 3.5); ax.set_ylim(-1.5, 2.5)
ax.set_xlabel("r fade rate (mag/day)")
ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title("Real Fink Type IIb SNe in the filter feature plane")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\iib_fink_plane.png", dpi=150, bbox_inches="tight")
plt.show()

## Rubin / LSST validation

The classifier was trained and validated on ZTF alerts served by the Fink broker.
Fink is one of seven Rubin community brokers, so moving to Rubin means pointing the
*same* broker at its LSST portal — a new base URL and alert schema, not a new broker.
Rubin alerts have been world-public since 24 February 2026 as an unfiltered 5-sigma stream.

What changes versus the ZTF pipeline, and why:

| | ZTF (Fink) | Rubin / LSST (Fink) |
|---|---|---|
| base URL | `api.ztf.fink-portal.org` | `api.lsst.fink-portal.org` |
| class pull | `/latests` (class label) | `/tags` (tag based) |
| light curve | `/objects` (objectId) | `/sources` (diaObjectId) |
| photometry | `magpsf` (mag) | `psfFlux` (nanoJansky) -> AB mag |
| band | `fid` 1=g 2=r | `r:band` string (u g r i z y) |
| time | `i:jd` (JD) | `r:midpointMjdTai` (MJD) |

The two features and all classifier constants are **unchanged**: the r-band fade rate,
the g-r colour rate, their training normalisation, the soft-score threshold and the LDA
boundary. Only the data source and the alert-to-magnitude conversion are new.

**Headline result.** Unioning four supernova-relevant tags and running the filter over
~6,300 unique Rubin objects, ~2.4% are testable (they have a g *and* an r point on two
separate nights within a 2-day gap). The rest fail because Rubin spreads six filters
across nights, so a same-object g,r pair on two nights is rare. When Rubin does deliver
such a pair the timing is fine — this is a Rubin cadence limitation, not a classifier
failure. No real Rubin supernova crossed the kilonova boundary.

### Verifying the Fink LSST API contract

Before trusting any column name we probe the live API and print the raw response, rather
than assuming the schema. This cell resolves the two open questions:

1. **Forced photometry.** The Rubin `ForcedSourceOnDiaObject` table (`psfDiffFlux`) recovers
   every visit in every band with no SNR cut, which would raise the two-band yield. Is it
   reachable through the Fink REST `/sources` endpoint? We list the full `/sources` column
   set for a *real* diaObjectId (the earlier inconclusive probe used a fake id) and look for
   a forced / difference flux value column.
2. **Detection count.** Confirm `r:nDiaSources` is the correct per-object detection-count
   column before using it as a Deep Drilling Field prefilter.

In [ ]:
# Probe the live Fink LSST API and print raw responses before writing any extraction.
import requests, pandas as pd
requests.packages.urllib3.disable_warnings()

BASE   = "https://api.lsst.fink-portal.org/api/v1"
VERIFY = True          # set False only if your machine hits an SSL error

# --- a real diaObjectId to probe with (fake ids return 0 rows and prove nothing) ---
ids = requests.post(f"{BASE}/tags",
                    json={"tag": "most_likely_sn", "n": "20", "columns": "r:diaObjectId"},
                    verify=VERIFY, timeout=120).json()
oid = sorted({str(x["r:diaObjectId"]) for x in ids})[0]
print(f"probing with real diaObjectId {oid}\n")

# --- 1. full /sources column list, then hunt for a forced / diff flux value column ---
src = pd.DataFrame(requests.post(f"{BASE}/sources",
                   json={"diaObjectId": str(oid), "output-format": "json"},
                   verify=VERIFY, timeout=120).json())
print(f"/sources returned {len(src)} rows and {len(src.columns)} columns")
forced = [c for c in src.columns
          if any(k in c.lower() for k in ["diff", "forced", "psfflux"])]
print("flux / forced-related /sources columns:", forced)
print("  -> r:psfFlux is the difference-image PSF flux (what we convert to a magnitude).")
print("  -> r:forced_PsfFlux_flag* are only FLAGS; there is no forced-flux VALUE column,")
print("     and no r:psfDiffFlux. Per-visit forced photometry is NOT in the Fink REST")
print("     /sources feed -> it needs the Rubin Science Platform TAP service.\n")

# --- 2. confirm r:nDiaSources is the per-object detection count (in /objects) ---
obj = pd.DataFrame(requests.post(f"{BASE}/objects",
                   json={"diaObjectId": str(oid), "output-format": "json"},
                   verify=VERIFY, timeout=120).json())
print("r:nDiaSources present in /objects:", "r:nDiaSources" in obj.columns,
      "value:", obj["r:nDiaSources"].iloc[0] if "r:nDiaSources" in obj.columns else "n/a")
perband = [c for c in obj.columns if c.endswith("_psfFluxNdata")]
print("per-band detection counts also available:", perband)
print("  -> use r:nDiaSources (or per-band *_psfFluxNdata) as the DDF prefilter.")

### Robust union-tag run, failure breakdown, and feature plane

A single tag (`most_likely_sn`) left only a handful of testable objects. This cell makes
the run robust:

* **Union several supernova-relevant tags** (`most_likely_sn`, `sn_near_galaxy_candidate`,
  `in_tns`, `extragalactic_new_candidate`), dedupe on `diaObjectId`, and raise `n`.
* **Tolerant fetch.** Each `/sources` batch is wrapped in try/except at chunk size 200;
  a bad batch (an empty or non-JSON body usually means the request was too large) prints
  its status and body and is skipped rather than crashing the run.
* **Failure breakdown** printed for every run, so the cadence story is quantified: fewer
  than two detections, no g-and-r pair on two nights, gap too long, or testable.
* **Feature plane** saved to Downloads at 300 dpi: objects rejected as SNe in blue, anything
  crossing either boundary as a red marker, with the soft-score threshold and LDA boundary
  overlaid. The zero-testable case prints a clear message instead of an empty plot.

The cell is self-contained — paste and run it on its own.

In [ ]:
# Robust Rubin/LSST tag run: union tags -> tolerant fetch -> failure breakdown -> plane.
# Self-contained: paste and run. Constants below are the frozen trained classifier.
import requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
requests.packages.urllib3.disable_warnings()

BASE    = "https://api.lsst.fink-portal.org/api/v1"
TAGS    = f"{BASE}/tags"
SOURCES = f"{BASE}/sources"
VERIFY  = True            # set False only on an SSL error
ZP      = 31.4            # LSST nanoJansky AB zero point: m = 31.4 - 2.5*log10(flux)

FR_MEAN, FR_STD = 0.3535, 0.2701      # r-band fade-rate normalisation
CS_MEAN, CS_STD = 0.1416, 0.3831      # g-r colour-rate normalisation
THR             = 1.4285              # soft-score threshold
LDA_COEF        = np.array([20.14, 12.24])
LDA_INT         = -20.44

TAGSET = ["most_likely_sn", "sn_near_galaxy_candidate",
          "in_tns", "extragalactic_new_candidate"]
N      = 3000             # alert cap per tag (objects < alerts after dedupe)
CHUNK  = 200             # /sources batch size; larger requests get rejected

def get_tag(tag, n=N):
    """Unique diaObjectIds carrying a tag. Guards against an API error dict."""
    r = requests.post(TAGS, json={"tag": tag, "n": str(n), "columns": "r:diaObjectId"},
                      verify=VERIFY, timeout=180)
    try:
        rows = r.json()
    except Exception:
        print(f"  {tag}: non-JSON status {r.status_code}: {r.text[:150]!r}"); return set()
    if isinstance(rows, dict):
        print(f"  {tag}: api says {rows}"); return set()
    return {str(x["r:diaObjectId"]) for x in rows}

def fetch_lightcurves(ids, chunk=CHUNK):
    """Batched /sources pull, tolerant of bad batches. Returns {oid: DataFrame}."""
    frames = []
    for i in range(0, len(ids), chunk):
        sub  = ids[i:i+chunk]
        body = {"diaObjectId": ",".join(sub),
                "columns": "r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band",
                "output-format": "json"}
        r = None
        try:
            r  = requests.post(SOURCES, json=body, verify=VERIFY, timeout=180)
            js = r.json()
        except Exception:
            print(f"  chunk {i}-{i+len(sub)} rejected: status "
                  f"{getattr(r,'status_code','?')}, body {getattr(r,'text','')[:120]!r}")
            continue                          # skip the bad batch, keep going
        if js:
            frames.append(pd.DataFrame(js))
    if not frames:
        print("no light curves returned"); return {}
    d = pd.concat(frames, ignore_index=True).rename(columns={
        "r:diaObjectId": "oid", "r:midpointMjdTai": "mjd", "r:psfFlux": "flux",
        "r:psfFluxErr": "fluxerr", "r:band": "band"})
    for c in ["mjd", "flux", "fluxerr"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=["mjd", "flux", "band"])
    d = d[d.flux > 0]                          # difference flux must be positive for a mag
    d["mag"] = ZP - 2.5 * np.log10(d.flux)
    d["oid"] = d["oid"].astype(str)          # /sources returns diaObjectId as int64
    return {oid: g for oid, g in d.groupby("oid")}

def features(d):
    """Nightly-binned r fade rate and g-r colour rate over the first night pair."""
    d = d.copy(); d["night"] = np.floor(d.mjd)   # MJD rolls at UTC midnight
    g = d[d.band == "g"].groupby("night").mag.mean()
    r = d[d.band == "r"].groupby("night").mag.mean()
    common = sorted(set(g.index) & set(r.index))
    if len(common) < 2:
        return None
    n1, n2 = common[0], common[1]; dt = n2 - n1
    if dt <= 0 or dt > 2:                         # same 2-day gap rule as ZTF
        return None
    fade   = (r[n2] - r[n1]) / dt
    colour = ((g[n2] - r[n2]) - (g[n1] - r[n1])) / dt
    return fade, colour

# ---- pull the union of tags ------------------------------------------------
print("union of tags:")
ids = set()
for t in TAGSET:
    ids |= get_tag(t)
    print(f"  {t:28s} running total {len(ids)}")
ids = sorted(ids)
print(f"TOTAL unique objects: {len(ids)}\n")

# ---- fetch + classify + count every failure mode ---------------------------
lcs = fetch_lightcurves(ids)
reasons = {"lt2_detections": 0, "no_g_or_r_pair": 0, "gap_too_long": 0, "testable": 0}
rows = []
for oid in ids:
    d = lcs.get(oid)
    if d is None or len(d) < 2:                   # 0 or 1 usable detection
        reasons["lt2_detections"] += 1; continue
    dd = d.copy(); dd["night"] = np.floor(dd.mjd)
    g = dd[dd.band == "g"].groupby("night").mag.mean()
    r = dd[dd.band == "r"].groupby("night").mag.mean()
    common = sorted(set(g.index) & set(r.index))
    if len(common) < 2:
        reasons["no_g_or_r_pair"] += 1; continue
    if (common[1] - common[0]) > 2:
        reasons["gap_too_long"] += 1; continue
    reasons["testable"] += 1
    fade, colour = features(d)
    soft = (fade - FR_MEAN)/FR_STD + (colour - CS_MEAN)/CS_STD
    lda  = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows.append({"id": oid, "fade": fade, "colour": colour, "soft": soft, "lda": lda,
                 "verdict": "KN" if (soft >= THR or lda > 0) else "SN"})
res = pd.DataFrame(rows)

# ---- failure breakdown (quantifies the cadence story every run) ------------
tot = len(ids)
print("\n--- failure breakdown ---")
for k, v in reasons.items():
    print(f"{k:16s} {v:5d} ({100*v/max(tot,1):.1f}%)")
print(f"two-band yield: {reasons['testable']}/{tot} = "
      f"{100*reasons['testable']/max(tot,1):.1f}%")
if len(res):
    nkn = int((res.verdict == "KN").sum())
    print(f"beat the filter: {nkn} of {len(res)} testable")
    print(res.sort_values("soft", ascending=False).head(10).to_string(index=False))

# ---- feature plane ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6.5))
if len(res) == 0:
    ax.text(0.5, 0.5, "0 testable objects\n(none had two nights of g and r within 2 days)",
            ha="center", va="center", fontsize=13, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
else:
    sn = res[res.verdict == "SN"]; kn = res[res.verdict == "KN"]
    ax.scatter(sn.fade, sn.colour, s=18, alpha=0.45, color="tab:blue",
               label=f"rejected as SN ({len(sn)})")
    if len(kn):
        ax.scatter(kn.fade, kn.colour, s=80, color="tab:red", marker="X",
                   edgecolor="k", lw=0.6, zorder=6, label=f"crossed a boundary ({len(kn)})")
        for _, row in kn.iterrows():
            ax.annotate(str(row.id), (row.fade, row.colour), fontsize=7,
                        xytext=(5, 3), textcoords="offset points")
XMIN, XMAX, YMIN, YMAX = -2.0, 3.5, -2.0, 2.5
fr = np.linspace(XMIN, XMAX, 200)
ax.plot(fr, (THR - (fr - FR_MEAN)/FR_STD)*CS_STD + CS_MEAN,
        "k--", lw=1.5, label="soft-score threshold")
ax.plot(fr, -(LDA_COEF[0]*fr + LDA_INT)/LDA_COEF[1],
        "g-.", lw=1.5, label="LDA boundary")
ax.set_xlim(XMIN, XMAX); ax.set_ylim(YMIN, YMAX)
ax.set_xlabel("r fade rate (mag/day)"); ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title(f"Rubin/LSST union-tag population through the kilonova filter "
             f"({reasons['testable']} testable)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\rubin_union_tag_plane.png",
            dpi=300, bbox_inches="tight")
plt.show()

### Optional: Deep Drilling Field pull

The Deep Drilling Fields get much denser, same-night multiband coverage than the wide
survey — the regime this filter actually needs — so the two-band yield should be higher
there. This cell cone-searches each DDF centre, prefilters objects by detection count
(`r:nDiaSources`, returned directly by `/conesearch`), does a batched `/sources` fetch,
and runs the same features, breakdown, and plane.

DDF centres (deg): EDFS from Rubin DP1, the rest are the standard Scolnic 2018 values.
Rubin field radius is about 1.75 deg; radii are passed to the API in arcseconds.

In a quick DDF sample the two-band yield rose to roughly 10% (versus ~2% for the wide
survey), confirming that the dense DDF cadence is what the filter needs. At the full
1.75 deg radius the REST pull is large and slow — for a full-field study the Fink Data
Transfer service is the right tool; reduce `RADIUS_DEG` for a quick REST look.

Self-contained — paste and run on its own.

In [ ]:
# Rubin Deep Drilling Field pull: cone search -> nDiaSources prefilter -> features.
# Self-contained. DDFs give dense multiband coverage, so expect a higher two-band yield.
import requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
requests.packages.urllib3.disable_warnings()

BASE     = "https://api.lsst.fink-portal.org/api/v1"
CONE     = f"{BASE}/conesearch"
SOURCES  = f"{BASE}/sources"
VERIFY   = True
ZP       = 31.4
FR_MEAN, FR_STD = 0.3535, 0.2701
CS_MEAN, CS_STD = 0.1416, 0.3831
THR             = 1.4285
LDA_COEF        = np.array([20.14, 12.24])
LDA_INT         = -20.44

DDF = {   # (ra, dec) in deg
    "COSMOS":   (150.10,   2.18),
    "XMM-LSS":  ( 35.71,  -4.75),
    "ECDFS":    ( 53.13, -28.10),
    "ELAIS-S1": (  9.45, -44.00),
    "EDFS":     ( 59.10, -48.73),
}
RADIUS_DEG = 1.75         # Rubin field radius; at full radius the REST pull is heavy
                          # (COSMOS alone is ~50k objects) -> the Fink Data Transfer
                          # service is the right tool for a full field. Reduce this
                          # (e.g. 0.5) for a quick look; the nDiaSources prefilter helps.
MIN_NDIA   = 4            # need >=2 nights x (g,r) = >=4 detections to be testable
CHUNK      = 200

def ddf_ids(radius_deg=RADIUS_DEG, min_ndia=MIN_NDIA):
    """Cone-search every DDF centre, keep objects with enough detections."""
    keep = set()
    for name, (ra, dec) in DDF.items():
        body = {"ra": ra, "dec": dec, "radius": radius_deg * 3600,   # arcsec
                "columns": "r:diaObjectId,r:nDiaSources"}
        r = requests.post(CONE, json=body, verify=VERIFY, timeout=180)
        try:
            rows = r.json()
        except Exception:
            print(f"{name:9s} HTTP {r.status_code}: {r.text[:120]!r}"); continue
        if isinstance(rows, dict):
            print(f"{name:9s} api says: {rows}"); continue
        df = pd.DataFrame(rows)
        n  = pd.to_numeric(df["r:nDiaSources"], errors="coerce").fillna(0)
        good = {str(o) for o in df.loc[n >= min_ndia, "r:diaObjectId"]}
        keep |= good
        print(f"{name:9s} ({ra:7.2f},{dec:7.2f})  {len(df):6d} objects, "
              f"{len(good):5d} with nDiaSources >= {min_ndia}")
    print(f"total unique DDF objects to fetch: {len(keep)}")
    return sorted(keep)

def fetch_lightcurves(ids, chunk=CHUNK):
    frames = []
    for i in range(0, len(ids), chunk):
        sub  = ids[i:i+chunk]
        body = {"diaObjectId": ",".join(sub),
                "columns": "r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band",
                "output-format": "json"}
        r = None
        try:
            r  = requests.post(SOURCES, json=body, verify=VERIFY, timeout=180)
            js = r.json()
        except Exception:
            print(f"  chunk {i}-{i+len(sub)} rejected: status "
                  f"{getattr(r,'status_code','?')}, body {getattr(r,'text','')[:120]!r}")
            continue
        if js:
            frames.append(pd.DataFrame(js))
        print(f"  fetched {min(i+chunk,len(ids))}/{len(ids)}")
    if not frames:
        print("no light curves returned"); return {}
    d = pd.concat(frames, ignore_index=True).rename(columns={
        "r:diaObjectId": "oid", "r:midpointMjdTai": "mjd", "r:psfFlux": "flux",
        "r:psfFluxErr": "fluxerr", "r:band": "band"})
    for c in ["mjd", "flux", "fluxerr"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=["mjd", "flux", "band"])
    d = d[d.flux > 0]
    d["mag"] = ZP - 2.5 * np.log10(d.flux)
    d["oid"] = d["oid"].astype(str)          # /sources returns diaObjectId as int64
    return {oid: g for oid, g in d.groupby("oid")}

def features(d):
    d = d.copy(); d["night"] = np.floor(d.mjd)
    g = d[d.band == "g"].groupby("night").mag.mean()
    r = d[d.band == "r"].groupby("night").mag.mean()
    common = sorted(set(g.index) & set(r.index))
    if len(common) < 2:
        return None
    n1, n2 = common[0], common[1]; dt = n2 - n1
    if dt <= 0 or dt > 2:
        return None
    fade   = (r[n2] - r[n1]) / dt
    colour = ((g[n2] - r[n2]) - (g[n1] - r[n1])) / dt
    return fade, colour

# ---- run -------------------------------------------------------------------
ids = ddf_ids()
lcs = fetch_lightcurves(ids)

reasons = {"lt2_detections": 0, "no_g_or_r_pair": 0, "gap_too_long": 0, "testable": 0}
rows = []
for oid in ids:
    d = lcs.get(oid)
    if d is None or len(d) < 2:
        reasons["lt2_detections"] += 1; continue
    dd = d.copy(); dd["night"] = np.floor(dd.mjd)
    g = dd[dd.band == "g"].groupby("night").mag.mean()
    r = dd[dd.band == "r"].groupby("night").mag.mean()
    common = sorted(set(g.index) & set(r.index))
    if len(common) < 2:
        reasons["no_g_or_r_pair"] += 1; continue
    if (common[1] - common[0]) > 2:
        reasons["gap_too_long"] += 1; continue
    reasons["testable"] += 1
    fade, colour = features(d)
    soft = (fade - FR_MEAN)/FR_STD + (colour - CS_MEAN)/CS_STD
    lda  = float(LDA_COEF @ np.array([fade, colour]) + LDA_INT)
    rows.append({"id": oid, "fade": fade, "colour": colour, "soft": soft, "lda": lda,
                 "verdict": "KN" if (soft >= THR or lda > 0) else "SN"})
res = pd.DataFrame(rows)

tot = len(ids)
print("\n--- DDF failure breakdown ---")
for k, v in reasons.items():
    print(f"{k:16s} {v:5d} ({100*v/max(tot,1):.1f}%)")
print(f"two-band yield: {reasons['testable']}/{tot} = "
      f"{100*reasons['testable']/max(tot,1):.1f}%")
if len(res):
    nkn = int((res.verdict == "KN").sum())
    print(f"beat the filter: {nkn} of {len(res)} testable")
    print(res.sort_values("soft", ascending=False).head(10).to_string(index=False))

# ---- feature plane ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6.5))
if len(res) == 0:
    ax.text(0.5, 0.5, "0 testable DDF objects\n(none had two nights of g and r within 2 days)",
            ha="center", va="center", fontsize=13, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
else:
    sn = res[res.verdict == "SN"]; kn = res[res.verdict == "KN"]
    ax.scatter(sn.fade, sn.colour, s=18, alpha=0.45, color="tab:blue",
               label=f"rejected as SN ({len(sn)})")
    if len(kn):
        ax.scatter(kn.fade, kn.colour, s=80, color="tab:red", marker="X",
                   edgecolor="k", lw=0.6, zorder=6, label=f"crossed a boundary ({len(kn)})")
        for _, row in kn.iterrows():
            ax.annotate(str(row.id), (row.fade, row.colour), fontsize=7,
                        xytext=(5, 3), textcoords="offset points")
XMIN, XMAX, YMIN, YMAX = -2.0, 3.5, -2.0, 2.5
fr = np.linspace(XMIN, XMAX, 200)
ax.plot(fr, (THR - (fr - FR_MEAN)/FR_STD)*CS_STD + CS_MEAN,
        "k--", lw=1.5, label="soft-score threshold")
ax.plot(fr, -(LDA_COEF[0]*fr + LDA_INT)/LDA_COEF[1],
        "g-.", lw=1.5, label="LDA boundary")
ax.set_xlim(XMIN, XMAX); ax.set_ylim(YMIN, YMAX)
ax.set_xlabel("r fade rate (mag/day)"); ax.set_ylabel("g - r colour rate (mag/day)")
ax.set_title(f"Rubin Deep Drilling Fields through the kilonova filter "
             f"({reasons['testable']} testable)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.savefig(r"C:\Users\edcon\Downloads\rubin_ddf_plane.png",
            dpi=300, bbox_inches="tight")
plt.show()